In [1]:
# Locate the repository when running from any notebook subfolder.
from pathlib import Path
import os
_notebook_start = Path.cwd().resolve()
_notebook_root = next(
    (p for p in [_notebook_start, *_notebook_start.parents]
     if (p / "docs" / "predictor_list.csv").is_file()
     and (p / "notebooks").is_dir()), None
)
if _notebook_root is None:
    raise FileNotFoundError("Cannot locate the project repository.")
os.chdir(_notebook_root)


In [2]:
# STAGE 3 - INFERENTIAL STATISTICS
# Cell 1: Libraries and project configuration

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

# 1. Locate project root

current_dir = Path.cwd().resolve()

candidates = [
    current_dir,
    current_dir.parent
]

PROJECT_ROOT = next(
    (
        path for path in candidates
        if (path / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate project root. "
        "Open the notebook inside Group5_STAT3013.R11."
    )

# 2. Official Stage 3 dataset
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "research_data_weather_price_holiday_assume_utc.csv"
)

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Official dataset not found: {DATA_PATH}"
    )


# 3. Output directories
TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "inferential_statistics"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
    / "inferential_statistics"
)

# Do not create or export files during initial validation.

# 4. Statistical configuration
ALPHA = 0.05
CONFIDENCE_LEVEL = 0.95
RANDOM_SEED = 42

# 5. Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

print("STAGE 3 - ENVIRONMENT CHECK")
print("-" * 55)
print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH.name)
print("Significance level:", ALPHA)
print("Confidence level:", CONFIDENCE_LEVEL)
print("Random seed:", RANDOM_SEED)
print("Status: PASS")


STAGE 3 - ENVIRONMENT CHECK
-------------------------------------------------------
Project root: C:\Users\LENOVO\OneDrive\Desktop\PTTK
Dataset: research_data_weather_price_holiday_assume_utc.csv
Significance level: 0.05
Confidence level: 0.95
Random seed: 42
Status: PASS


In [3]:

# ============================================================
# Cell 2: Load and validate the official research dataset
# ============================================================

df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

print("DATASET VALIDATION")
print("-" * 55)

# ------------------------------------------------------------
# 1. Dataset dimensions
# ------------------------------------------------------------

total_orders = len(df)
total_columns = df.shape[1]

print("Total rows:", total_orders)
print("Total columns:", total_columns)

# ------------------------------------------------------------
# 2. Order-level uniqueness
# ------------------------------------------------------------

required_keys = ["order_id", "is_late"]

missing_keys = [
    col for col in required_keys
    if col not in df.columns
]

if missing_keys:
    raise ValueError(
        f"Required columns missing: {missing_keys}"
    )

missing_order_ids = df["order_id"].isna().sum()
duplicate_order_ids = df["order_id"].duplicated().sum()

print("Missing order_id:", missing_order_ids)
print("Duplicate order_id:", duplicate_order_ids)

if missing_order_ids > 0:
    raise ValueError("order_id contains missing values.")

if duplicate_order_ids > 0:
    raise ValueError("Dataset is not order-level.")

# ------------------------------------------------------------
# 3. Target validation
# ------------------------------------------------------------

invalid_target = (
    df["is_late"].notna()
    & ~df["is_late"].isin([0, 1])
)

invalid_target_count = int(invalid_target.sum())

if invalid_target_count > 0:
    raise ValueError(
        f"Invalid is_late values: {invalid_target_count}"
    )

target_counts = (
    df["is_late"]
    .value_counts(dropna=False)
)

n_not_late = int((df["is_late"] == 0).sum())
n_late = int((df["is_late"] == 1).sum())
n_missing_target = int(df["is_late"].isna().sum())

n_valid = n_not_late + n_late

late_rate = (
    n_late / n_valid
    if n_valid > 0
    else np.nan
)

print("\nTARGET DISTRIBUTION")
print(target_counts)

print("\nValid target orders:", n_valid)
print("Missing target orders:", n_missing_target)
print("Late delivery rate:", f"{late_rate:.4%}")

# ------------------------------------------------------------
# 4. Check timezone scenario
# ------------------------------------------------------------

if "timezone_scenario" in df.columns:
    print("\nTimezone scenario values:")
    print(
        df["timezone_scenario"]
        .value_counts(dropna=False)
    )

# ------------------------------------------------------------
# 5. Main late-delivery sample
# ------------------------------------------------------------

df_labeled = df.loc[
    df["is_late"].isin([0, 1])
].copy()

df_labeled["is_late"] = (
    df_labeled["is_late"].astype(int)
)

# Do not remove missing values in other features.

assert df_labeled["order_id"].is_unique
assert len(df_labeled) == n_valid

print("\nVALIDATION STATUS: PASS")


DATASET VALIDATION
-------------------------------------------------------
Total rows: 99441
Total columns: 74
Missing order_id: 0
Duplicate order_id: 0

TARGET DISTRIBUTION
is_late
0.0000    89936
1.0000     6534
NaN        2971
Name: count, dtype: int64

Valid target orders: 96470
Missing target orders: 2971
Late delivery rate: 6.7731%

Timezone scenario values:
timezone_scenario
assume_utc    99441
Name: count, dtype: int64

VALIDATION STATUS: PASS


In [4]:

# ============================================================
# Cell 3: Research question variable availability
# ============================================================

RQ_VARIABLES = {
    "RQ1": ["customer_state"],
    "RQ2": ["purchase_weekday"],
    "RQ3": ["purchase_month"],
    "RQ4": ["has_interstate_seller"],

    "RQ5": ["distance_km_mean"],

    "RQ6": [
        "order_total_value",
        "freight_value"
    ],

    "RQ7": [
        "freight_to_items_ratio",
        "mean_item_value",
        "freight_per_item"
    ],

    "RQ8": [
        "total_product_weight_g",
        "total_product_volume_cm3"
    ],

    "RQ9": ["seller_count"],
    "RQ10": ["product_category_count"],
    "RQ11": ["estimated_window_days"],

    "RQ12": ["purchase_is_national_holiday"],
    "RQ13": ["national_holiday_count_in_estimated_window"],
    "RQ14": ["days_to_next_national_holiday"],

    "RQ15a": ["origin_rain_state_mean_mm"],
    "RQ15b": ["destination_rain_state_mean_mm"],

    "RQ16a": ["origin_humidity_state_mean_pct"],
    "RQ16b": ["destination_humidity_state_mean_pct"],

    "RQ17a": ["origin_air_temp_state_mean_c"],
    "RQ17b": ["destination_air_temp_state_mean_c"],

    "RQ19": ["low_review"],
    "RQ20": ["is_canceled"]
}

availability_records = []

for rq, variables in RQ_VARIABLES.items():
    for variable in variables:

        exists = variable in df.columns

        availability_records.append({
            "rq": rq,
            "variable": variable,
            "exists": exists,
            "status": (
                "AVAILABLE"
                if exists
                else "MISSING"
            )
        })

availability_df = pd.DataFrame(
    availability_records
)

display(availability_df)

missing_variables = availability_df.loc[
    ~availability_df["exists"],
    ["rq", "variable"]
]

print("\nVARIABLE AVAILABILITY SUMMARY")
print("-" * 55)

print(
    "Total required variable entries:",
    len(availability_df)
)

print(
    "Available:",
    int(availability_df["exists"].sum())
)

print(
    "Missing:",
    int((~availability_df["exists"]).sum())
)

if not missing_variables.empty:
    print("\nVariables requiring additional preparation:")
    display(missing_variables)

print("\nCell 3 completed.")


,rq,variable,exists,status
0,RQ1,customer_state,True,AVAILABLE
1,RQ2,purchase_weekday,True,AVAILABLE
2,RQ3,purchase_month,True,AVAILABLE
3,RQ4,has_interstate_seller,True,AVAILABLE
4,RQ5,distance_km_mean,True,AVAILABLE
5,RQ6,order_total_value,True,AVAILABLE
6,RQ6,freight_value,True,AVAILABLE
7,RQ7,freight_to_items_ratio,True,AVAILABLE
8,RQ7,mean_item_value,True,AVAILABLE
9,RQ7,freight_per_item,True,AVAILABLE



VARIABLE AVAILABILITY SUMMARY
-------------------------------------------------------
Total required variable entries: 26
Available: 24
Missing: 2

Variables requiring additional preparation:


,rq,variable
24,RQ19,low_review
25,RQ20,is_canceled



Cell 3 completed.


In [5]:

# ============================================================
# Cell 4: Missing values and preliminary sample availability
# ============================================================

sample_audit_records = []

for rq, variables in RQ_VARIABLES.items():

    # RQ19 and RQ20 require separate sample construction.
    if rq in ["RQ19", "RQ20"]:
        continue

    for variable in variables:

        if variable not in df_labeled.columns:
            continue

        # Start with all orders with a valid late-delivery label.
        initial_n = len(df_labeled)

        missing_mask = df_labeled[variable].isna()

        missing_n = int(missing_mask.sum())
        retained_n = initial_n - missing_n

        missing_pct = (
            missing_n / initial_n * 100
            if initial_n > 0
            else np.nan
        )

        sample_audit_records.append({
            "rq": rq,
            "variable": variable,
            "initial_n": initial_n,
            "missing_n": missing_n,
            "retained_n": retained_n,
            "missing_pct": missing_pct
        })

sample_audit_df = pd.DataFrame(
    sample_audit_records
)

print("PRELIMINARY SAMPLE AVAILABILITY")
print("-" * 55)

display(sample_audit_df)

print("\nSUMMARY")
print("Variables audited:", len(sample_audit_df))

print(
    "Variables with missing values:",
    int((sample_audit_df["missing_n"] > 0).sum())
)

print(
    "Variables with no usable observations:",
    int((sample_audit_df["retained_n"] == 0).sum())
)

print("\nNOTE:")
print(
    "This is a preliminary feature-level missingness audit."
)
print(
    "It is not the final sample_summary.csv."
)
print(
    "Each statistical test must later report its actual "
    "Late/Not Late group sizes and any additional exclusions."
)


PRELIMINARY SAMPLE AVAILABILITY
-------------------------------------------------------


,rq,variable,initial_n,missing_n,retained_n,missing_pct
0,RQ1,customer_state,96470,0,96470,0.0000
1,RQ2,purchase_weekday,96470,0,96470,0.0000
2,RQ3,purchase_month,96470,0,96470,0.0000
3,RQ4,has_interstate_seller,96470,0,96470,0.0000
4,RQ5,distance_km_mean,96470,480,95990,0.4976
5,RQ6,order_total_value,96470,0,96470,0.0000
6,RQ6,freight_value,96470,0,96470,0.0000
7,RQ7,freight_to_items_ratio,96470,0,96470,0.0000
8,RQ7,mean_item_value,96470,0,96470,0.0000
9,RQ7,freight_per_item,96470,0,96470,0.0000



SUMMARY
Variables audited: 24
Variables with missing values: 10
Variables with no usable observations: 0

NOTE:
This is a preliminary feature-level missingness audit.
It is not the final sample_summary.csv.
Each statistical test must later report its actual Late/Not Late group sizes and any additional exclusions.


In [6]:

# ============================================================
# Cell 5: Categorical data quality checks
# ============================================================

CATEGORICAL_VARIABLES = [
    "customer_state",
    "purchase_weekday",
    "purchase_month",
    "has_interstate_seller"
]

print("CATEGORICAL DATA QUALITY CHECKS")
print("=" * 65)

for variable in CATEGORICAL_VARIABLES:

    print(f"\nVARIABLE: {variable}")
    print("-" * 65)

    series = df_labeled[variable]

    print("Data type:", series.dtype)
    print("Missing values:", int(series.isna().sum()))
    print("Unique values:", series.nunique(dropna=True))

    value_summary = (
        series.value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="n_orders")
    )

    display(value_summary)

    # Cross-tabulation with the late-delivery target.
    contingency = pd.crosstab(
        df_labeled[variable],
        df_labeled["is_late"],
        dropna=False
    )

    contingency = contingency.reindex(
        columns=[0, 1],
        fill_value=0
    )

    contingency.columns = ["Not_Late", "Late"]

    contingency["total_orders"] = (
        contingency["Not_Late"]
        + contingency["Late"]
    )

    contingency["late_rate_pct"] = np.where(
        contingency["total_orders"] > 0,
        contingency["Late"]
        / contingency["total_orders"] * 100,
        np.nan
    )

    print("\nLate delivery distribution by group:")
    display(contingency)

    small_groups = contingency.loc[
        contingency["total_orders"] < 100
    ]

    if not small_groups.empty:
        print(
            "WARNING: Some groups have fewer than "
            "100 valid orders."
        )
        display(small_groups)

    else:
        print("All groups contain at least 100 valid orders.")

print("\nCATEGORICAL QUALITY CHECK COMPLETED.")


CATEGORICAL DATA QUALITY CHECKS

VARIABLE: customer_state
-----------------------------------------------------------------
Data type: object
Missing values: 0
Unique values: 27


,value,n_orders
0,SP,40494
1,RJ,12350
2,MG,11354
3,RS,5344
4,PR,4923
5,SC,3546
6,BA,3256
7,DF,2080
8,ES,1995
9,GO,1957



Late delivery distribution by group:


,Not_Late,Late,total_orders,late_rate_pct
customer_state,,,,
AC,77,3,80,3.7500
AL,312,85,397,21.4106
AM,141,4,145,2.7586
AP,65,2,67,2.9851
BA,2860,396,3256,12.1622
CE,1103,176,1279,13.7608
DF,1962,118,2080,5.6731
ES,1781,214,1995,10.7268
GO,1829,128,1957,6.5406


,Not_Late,Late,total_orders,late_rate_pct
customer_state,,,,
AC,77,3,80,3.7500
AP,65,2,67,2.9851
RR,36,5,41,12.1951



VARIABLE: purchase_weekday
-----------------------------------------------------------------
Data type: object
Missing values: 0
Unique values: 7


... (9 outputs left out)

,value,n_orders
0,1.0000,61895
1,0.0000,34575



Late delivery distribution by group:


,Not_Late,Late,total_orders,late_rate_pct
has_interstate_seller,,,,
0.0000,33012,1563,34575,4.5206
1.0000,56924,4971,61895,8.0313


All groups contain at least 100 valid orders.

CATEGORICAL QUALITY CHECK COMPLETED.


In [7]:

# ============================================================
# Cell 6: RQ1 - Customer State vs Late Delivery
# ============================================================

# Research question:
# Does the late delivery rate differ across customer states?
#
# H0: Late delivery rates are equal across states.
# H1: At least one state has a different late delivery rate.
#
# Two-sided / non-directional association test.
# Alpha = 0.05.
# Holm correction will be handled across the planned test family.

RQ_ID = "RQ1"
RQ_VARIABLE = "customer_state"

print("RQ1 - CUSTOMER STATE AND LATE DELIVERY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare the RQ-specific sample
# ------------------------------------------------------------

initial_n = len(df_labeled)

rq1 = df_labeled[
    ["order_id", RQ_VARIABLE, "is_late"]
].copy()

missing_n = int(rq1[RQ_VARIABLE].isna().sum())

rq1 = rq1.dropna(
    subset=[RQ_VARIABLE]
)

retained_n = len(rq1)

print("Initial labeled orders:", initial_n)
print("Excluded for missing state:", missing_n)
print("Retained orders:", retained_n)

assert rq1["order_id"].is_unique
assert retained_n + missing_n == initial_n

# ------------------------------------------------------------
# 2. Create contingency table
# ------------------------------------------------------------

contingency = pd.crosstab(
    rq1[RQ_VARIABLE],
    rq1["is_late"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

contingency.columns = ["Not_Late", "Late"]

# Remove empty groups, if any.
contingency = contingency.loc[
    contingency.sum(axis=1) > 0
]

if len(contingency) < 2:
    raise ValueError(
        "RQ1 requires at least two customer states."
    )

if (contingency.sum(axis=0) == 0).any():
    raise ValueError(
        "RQ1 requires observations in both target classes."
    )

group_summary = contingency.copy()

group_summary["total_orders"] = (
    group_summary["Not_Late"]
    + group_summary["Late"]
)

group_summary["late_rate"] = (
    group_summary["Late"]
    / group_summary["total_orders"]
)

# ------------------------------------------------------------
# 3. Wilson 95% confidence intervals
# ------------------------------------------------------------

ci_lower, ci_upper = proportion_confint(
    count=group_summary["Late"].to_numpy(),
    nobs=group_summary["total_orders"].to_numpy(),
    alpha=ALPHA,
    method="wilson"
)

group_summary["ci_lower"] = ci_lower
group_summary["ci_upper"] = ci_upper

group_summary["late_rate_pct"] = (
    group_summary["late_rate"] * 100
)

group_summary["ci_lower_pct"] = (
    group_summary["ci_lower"] * 100
)

group_summary["ci_upper_pct"] = (
    group_summary["ci_upper"] * 100
)

group_summary = group_summary.sort_values(
    "late_rate",
    ascending=False
)

print("\nGROUP SUMMARY WITH WILSON 95% CI")
display(group_summary)

# ------------------------------------------------------------
# 4. Pearson Chi-square test
# ------------------------------------------------------------

observed = contingency.to_numpy()

chi2_stat, asymptotic_p, dof, expected = (
    stats.chi2_contingency(
        observed,
        correction=False
    )
)

expected_df = pd.DataFrame(
    expected,
    index=contingency.index,
    columns=contingency.columns
)

small_expected = expected < 5
n_small_expected = int(small_expected.sum())
n_expected_cells = int(expected.size)

small_expected_pct = (
    n_small_expected / n_expected_cells * 100
)

print("\nCHI-SQUARE DIAGNOSTICS")
print("-" * 70)

print(f"Chi-square statistic: {chi2_stat:.6f}")
print(f"Degrees of freedom: {dof}")
print(f"Asymptotic p-value: {asymptotic_p:.10g}")
print(f"Minimum expected count: {expected.min():.4f}")
print(f"Cells with expected count < 5: {n_small_expected}")
print(f"Percentage of cells < 5: {small_expected_pct:.2f}%")
print(f"Cells with expected count < 1: {(expected < 1).sum()}")

print("\nEXPECTED COUNTS")
display(expected_df)

# ------------------------------------------------------------
# 5. Assess expected-count diagnostics
# ------------------------------------------------------------

# A commonly used screening rule:
# no expected count < 1, and at most 20% below 5.
# This is a rule of thumb, not an absolute guarantee.

asymptotic_check = (
    expected.min() >= 1
    and small_expected_pct <= 20
)

if asymptotic_check:
    print("\nExpected-count screening: PASS")
else:
    print("\nExpected-count screening: WARNING")
    print(
        "Use an alternative or simulation-based "
        "p-value before final inference."
    )

# ------------------------------------------------------------
# 6. Monte Carlo conditional test
# ------------------------------------------------------------

# Fixed-margin random contingency tables provide
# a simulation-based alternative for sparse cells.
# All observed states remain in the analysis.

N_RESAMPLES = 20000
rng = np.random.default_rng(RANDOM_SEED)

row_totals = observed.sum(axis=1)
col_totals = observed.sum(axis=0)

simulated_tables = stats.random_table.rvs(
    row_totals,
    col_totals,
    size=N_RESAMPLES,
    random_state=rng
)

simulated_chi2 = (
    ((simulated_tables - expected) ** 2 / expected)
    .sum(axis=(1, 2))
)

n_extreme = int(
    np.count_nonzero(
        simulated_chi2 >= chi2_stat - 1e-12
    )
)

monte_carlo_p = (
    (n_extreme + 1)
    / (N_RESAMPLES + 1)
)

# Use the Monte Carlo result for the final RQ1 raw
# p-value in this workflow.
raw_p_value = float(monte_carlo_p)

print("\nMONTE CARLO CONDITIONAL TEST")
print("-" * 70)
print("Resamples:", N_RESAMPLES)
print("Extreme resamples:", n_extreme)
print(f"Monte Carlo p-value: {monte_carlo_p:.8f}")
print(
    "The p-value is a simulation estimate "
    "conditional on the observed margins."
)

# ------------------------------------------------------------
# 7. Cramer's V
# ------------------------------------------------------------

n = observed.sum()
r, c = observed.shape

cramers_v = np.sqrt(
    chi2_stat / (n * min(r - 1, c - 1))
)

print("\nEFFECT SIZE")
print("-" * 70)
print(f"Cramer's V: {cramers_v:.6f}")

# ------------------------------------------------------------
# 8. Save results in notebook memory
# ------------------------------------------------------------

rq1_result = {
    "rq": RQ_ID,
    "variable": RQ_VARIABLE,
    "method": "Chi-square statistic with conditional Monte Carlo p",
    "initial_n": initial_n,
    "missing_n": missing_n,
    "n": retained_n,
    "n_groups": len(contingency),
    "test_statistic": float(chi2_stat),
    "degrees_of_freedom": int(dof),
    "p_value_raw": raw_p_value,
    "p_value_holm": np.nan,
    "effect_size_name": "Cramer's V",
    "effect_size": float(cramers_v),
    "expected_min": float(expected.min()),
    "expected_lt5": n_small_expected,
    "monte_carlo_resamples": N_RESAMPLES
}

# ------------------------------------------------------------
# 9. Preliminary conclusion
# ------------------------------------------------------------

print("\nPRELIMINARY RESULT")
print("-" * 70)

if raw_p_value < ALPHA:
    print(
        "Unadjusted result: evidence of an association "
        "between customer state and late delivery."
    )
else:
    print(
        "Unadjusted result: insufficient evidence "
        "to reject independence."
    )

print(
    "Final interpretation is pending the "
    "pre-specified Holm adjustment."
)
print(
    "Association does not establish causation."
)

print("\nRQ1 ANALYSIS COMPLETED.")


RQ1 - CUSTOMER STATE AND LATE DELIVERY


Initial labeled orders: 96470
Excluded for missing state: 0
Retained orders: 96470



GROUP SUMMARY WITH WILSON 95% CI


,Not_Late,Late,total_orders,late_rate,ci_lower,ci_upper,late_rate_pct,ci_lower_pct,ci_upper_pct
customer_state,,,,,,,,,
AL,312,85,397,0.2141,0.1766,0.2571,21.4106,17.6596,25.7096
MA,592,125,717,0.1743,0.1483,0.2038,17.4338,14.8322,20.3824
SE,284,51,335,0.1522,0.1177,0.1946,15.2239,11.7727,19.4636
PI,410,66,476,0.1387,0.1105,0.1726,13.8655,11.0492,17.2604
CE,1103,176,1279,0.1376,0.1198,0.1576,13.7608,11.9810,15.7575
RR,36,5,41,0.1220,0.0532,0.2554,12.1951,5.3233,25.5442
BA,2860,396,3256,0.1216,0.1108,0.1333,12.1622,11.0839,13.3296
RJ,10855,1495,12350,0.1211,0.1154,0.1269,12.1053,11.5417,12.6924
PA,840,106,946,0.1121,0.0935,0.1337,11.2051,9.3499,13.3741



CHI-SQUARE DIAGNOSTICS
----------------------------------------------------------------------
Chi-square statistic: 1785.792751
Degrees of freedom: 26
Asymptotic p-value: 0
Minimum expected count: 2.7770
Cells with expected count < 5: 2
Percentage of cells < 5: 3.70%
Cells with expected count < 1: 0

EXPECTED COUNTS


,Not_Late,Late
customer_state,,
AC,74.5815,5.4185
AL,370.1108,26.8892
AM,135.1790,9.8210
AP,62.4620,4.5380
BA,3035.4682,220.5318
CE,1192.3722,86.6278
DF,1939.1197,140.8803
ES,1859.8769,135.1231
GO,1824.4506,132.5494



Expected-count screening: PASS



MONTE CARLO CONDITIONAL TEST
----------------------------------------------------------------------
Resamples: 20000
Extreme resamples: 0
Monte Carlo p-value: 0.00005000
The p-value is a simulation estimate conditional on the observed margins.

EFFECT SIZE
----------------------------------------------------------------------
Cramer's V: 0.136057

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted result: evidence of an association between customer state and late delivery.
Final interpretation is pending the pre-specified Holm adjustment.
Association does not establish causation.

RQ1 ANALYSIS COMPLETED.


In [8]:

# ============================================================
# Cell 7: Pre-specified statistical testing plan
# ============================================================

# Proposed single Holm family for Stage 3.
# This plan must be confirmed by the research team
# before final inferential conclusions are reported.

TEST_PLAN = {
    "RQ1": ["customer_state"],
    "RQ2": ["purchase_weekday"],
    "RQ3": ["purchase_month"],
    "RQ4": ["has_interstate_seller"],
    "RQ5": ["distance_km_mean"],
    "RQ6": ["order_total_value", "freight_value"],
    "RQ7": [
        "freight_to_items_ratio",
        "mean_item_value",
        "freight_per_item"
    ],
    "RQ8": [
        "total_product_weight_g",
        "total_product_volume_cm3"
    ],
    "RQ9": ["seller_count_group"],
    "RQ10": ["product_category_count_group"],
    "RQ11": ["estimated_window_days"],
    "RQ12": ["purchase_is_national_holiday"],
    "RQ13": ["national_holiday_count_group"],
    "RQ14": ["days_to_next_holiday_group"],
    "RQ15a": ["origin_rain_state_mean_mm"],
    "RQ15b": ["destination_rain_state_mean_mm"],
    "RQ16a": ["origin_humidity_state_mean_pct"],
    "RQ16b": ["destination_humidity_state_mean_pct"],
    "RQ17a": ["origin_air_temp_state_mean_c"],
    "RQ17b": ["destination_air_temp_state_mean_c"],
    "RQ19": ["low_review"],
    "RQ20": ["is_canceled"]
}

planned_tests = [
    {
        "rq": rq,
        "variable": variable,
        "holm_family": "stage3_all",
        "alpha": ALPHA,
        "alternative": "two-sided"
    }
    for rq, variables in TEST_PLAN.items()
    for variable in variables
]

test_plan_df = pd.DataFrame(planned_tests)

# Prevent accidental duplicate test definitions.
assert not test_plan_df[
    ["rq", "variable"]
].duplicated().any()

assert len(test_plan_df) == 26

print("STAGE 3 - PROPOSED TESTING PLAN")
print("=" * 65)
print("Number of planned tests:", len(test_plan_df))
print("Significance level:", ALPHA)
print("Multiple-testing method: Holm")
print("Correction family: stage3_all")
print("Status: PROPOSED - TEAM CONFIRMATION REQUIRED")

display(test_plan_df)


STAGE 3 - PROPOSED TESTING PLAN
Number of planned tests: 26
Significance level: 0.05
Multiple-testing method: Holm
Correction family: stage3_all
Status: PROPOSED - TEAM CONFIRMATION REQUIRED


,rq,variable,holm_family,alpha,alternative
0,RQ1,customer_state,stage3_all,0.0500,two-sided
1,RQ2,purchase_weekday,stage3_all,0.0500,two-sided
2,RQ3,purchase_month,stage3_all,0.0500,two-sided
3,RQ4,has_interstate_seller,stage3_all,0.0500,two-sided
4,RQ5,distance_km_mean,stage3_all,0.0500,two-sided
5,RQ6,order_total_value,stage3_all,0.0500,two-sided
6,RQ6,freight_value,stage3_all,0.0500,two-sided
7,RQ7,freight_to_items_ratio,stage3_all,0.0500,two-sided
8,RQ7,mean_item_value,stage3_all,0.0500,two-sided
9,RQ7,freight_per_item,stage3_all,0.0500,two-sided


In [9]:

# ============================================================
# Cell 8: RQ2 - Purchase Weekday vs Late Delivery
# ============================================================

RQ_ID = "RQ2"
RQ_VARIABLE = "purchase_weekday"

WEEKDAY_ORDER = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

print("RQ2 - PURCHASE WEEKDAY AND LATE DELIVERY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare sample
# ------------------------------------------------------------

initial_n = len(df_labeled)

rq2 = df_labeled[
    ["order_id", RQ_VARIABLE, "is_late"]
].copy()

missing_n = int(rq2[RQ_VARIABLE].isna().sum())

rq2 = rq2.dropna(
    subset=[RQ_VARIABLE]
)

retained_n = len(rq2)

unexpected_weekdays = sorted(
    set(rq2[RQ_VARIABLE].unique())
    - set(WEEKDAY_ORDER),
    key=str
)

if unexpected_weekdays:
    raise ValueError(
        f"Unexpected weekday values: {unexpected_weekdays}"
    )

assert rq2["order_id"].is_unique
assert initial_n == missing_n + retained_n

print("Initial labeled orders:", initial_n)
print("Excluded for missing weekday:", missing_n)
print("Retained orders:", retained_n)

# ------------------------------------------------------------
# 2. Contingency table
# ------------------------------------------------------------

contingency_rq2 = pd.crosstab(
    rq2[RQ_VARIABLE],
    rq2["is_late"]
).reindex(
    index=WEEKDAY_ORDER,
    columns=[0, 1],
    fill_value=0
)

contingency_rq2.columns = [
    "Not_Late",
    "Late"
]

# Remove only genuinely unobserved weekday categories.
contingency_rq2 = contingency_rq2.loc[
    contingency_rq2.sum(axis=1) > 0
]

if len(contingency_rq2) < 2:
    raise ValueError(
        "RQ2 requires at least two weekday groups."
    )

if (contingency_rq2.sum(axis=0) == 0).any():
    raise ValueError(
        "RQ2 requires both Late and Not Late orders."
    )

# ------------------------------------------------------------
# 3. Descriptive statistics
# ------------------------------------------------------------

rq2_summary = contingency_rq2.copy()

rq2_summary["total_orders"] = (
    rq2_summary["Not_Late"]
    + rq2_summary["Late"]
)

rq2_summary["late_rate"] = (
    rq2_summary["Late"]
    / rq2_summary["total_orders"]
)

lower, upper = proportion_confint(
    count=rq2_summary["Late"].to_numpy(),
    nobs=rq2_summary["total_orders"].to_numpy(),
    alpha=ALPHA,
    method="wilson"
)

rq2_summary["ci_lower"] = lower
rq2_summary["ci_upper"] = upper

rq2_summary["late_rate_pct"] = (
    rq2_summary["late_rate"] * 100
)

rq2_summary["ci_lower_pct"] = lower * 100
rq2_summary["ci_upper_pct"] = upper * 100

print("\nGROUP SUMMARY WITH WILSON 95% CI")
display(rq2_summary)

# ------------------------------------------------------------
# 4. Pearson Chi-square
# ------------------------------------------------------------

observed = contingency_rq2.to_numpy()

chi2_stat, p_value, dof, expected = (
    stats.chi2_contingency(
        observed,
        correction=False
    )
)

expected_min = float(expected.min())

n_expected_lt5 = int(
    (expected < 5).sum()
)

n_expected_lt1 = int(
    (expected < 1).sum()
)

expected_lt5_pct = (
    n_expected_lt5 / expected.size * 100
)

print("\nCHI-SQUARE DIAGNOSTICS")
print("-" * 70)

print(f"Chi-square statistic: {chi2_stat:.6f}")
print("Degrees of freedom:", dof)
print(f"Raw p-value: {p_value:.10g}")
print(f"Minimum expected count: {expected_min:.4f}")
print("Cells with expected count < 5:", n_expected_lt5)
print("Cells with expected count < 1:", n_expected_lt1)
print(f"Percentage below 5: {expected_lt5_pct:.2f}%")

asymptotic_check = (
    expected_min >= 1
    and expected_lt5_pct <= 20
)

if asymptotic_check:
    print("Expected-count screening: PASS")
else:
    print("Expected-count screening: WARNING")
    print(
        "Further assessment is required before "
        "using the asymptotic p-value."
    )

# ------------------------------------------------------------
# 5. Cramer's V
# ------------------------------------------------------------

n = observed.sum()
r, c = observed.shape

cramers_v = np.sqrt(
    chi2_stat / (n * min(r - 1, c - 1))
)

print("\nEFFECT SIZE")
print("-" * 70)
print(f"Cramer's V: {cramers_v:.6f}")

# ------------------------------------------------------------
# 6. Store results
# ------------------------------------------------------------

rq2_result = {
    "rq": RQ_ID,
    "variable": RQ_VARIABLE,
    "method": "Pearson Chi-square",
    "initial_n": initial_n,
    "missing_n": missing_n,
    "n": retained_n,
    "n_groups": len(contingency_rq2),
    "test_statistic": float(chi2_stat),
    "degrees_of_freedom": int(dof),
    "p_value_raw": (
        float(p_value)
        if asymptotic_check
        else np.nan
    ),
    "p_value_holm": np.nan,
    "effect_size_name": "Cramer's V",
    "effect_size": float(cramers_v),
    "expected_min": expected_min,
    "expected_lt5": n_expected_lt5,
    "assumption_screening_pass": bool(asymptotic_check)
}

# ------------------------------------------------------------
# 7. Preliminary interpretation
# ------------------------------------------------------------

print("\nPRELIMINARY RESULT")
print("-" * 70)

if not asymptotic_check:
    print(
        "The asymptotic test requires further review."
    )
elif p_value < ALPHA:
    print(
        "Unadjusted evidence of association "
        "between purchase weekday and late delivery."
    )
else:
    print(
        "Insufficient unadjusted evidence "
        "to reject independence."
    )

print(
    "Final inference is pending the Holm correction."
)
print(
    "This exploratory association does not "
    "establish causation."
)

print("\nRQ2 ANALYSIS COMPLETED.")


RQ2 - PURCHASE WEEKDAY AND LATE DELIVERY
Initial labeled orders: 96470
Excluded for missing weekday: 0
Retained orders: 96470

GROUP SUMMARY WITH WILSON 95% CI


,Not_Late,Late,total_orders,late_rate,ci_lower,ci_upper,late_rate_pct,ci_lower_pct,ci_upper_pct
purchase_weekday,,,,,,,,,
Monday,14533,1168,15701,0.0744,0.0704,0.0786,7.4390,7.0389,7.8600
Tuesday,14409,1093,15502,0.0705,0.0666,0.0746,7.0507,6.6583,7.4644
Wednesday,14084,990,15074,0.0657,0.0618,0.0697,6.5676,6.1831,6.9742
Thursday,13419,903,14322,0.0630,0.0592,0.0671,6.3050,5.9185,6.7149
Friday,12724,960,13684,0.0702,0.0660,0.0746,7.0155,6.5995,7.4556
Saturday,9859,696,10555,0.0659,0.0614,0.0708,6.5940,6.1362,7.0835
Sunday,10908,724,11632,0.0622,0.0580,0.0668,6.2242,5.7995,6.6779



CHI-SQUARE DIAGNOSTICS
----------------------------------------------------------------------
Chi-square statistic: 26.256247
Degrees of freedom: 6
Raw p-value: 0.0001994516719
Minimum expected count: 714.8997
Cells with expected count < 5: 0
Cells with expected count < 1: 0
Percentage below 5: 0.00%
Expected-count screening: PASS

EFFECT SIZE
----------------------------------------------------------------------
Cramer's V: 0.016498

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted evidence of association between purchase weekday and late delivery.
Final inference is pending the Holm correction.
This exploratory association does not establish causation.

RQ2 ANALYSIS COMPLETED.


In [10]:

# ============================================================
# Cell 9: RQ3 - Purchase Month vs Late Delivery
# ============================================================

# Research question:
# Does the late delivery rate differ across purchase months?
#
# H0: All monthly late delivery rates are equal.
# H1: At least one monthly late delivery rate differs.
#
# Alpha = 0.05
# Final inference requires Holm correction.
# This is exploratory analysis, not evidence of seasonality.

RQ_ID = "RQ3"
RQ_VARIABLE = "purchase_month"

print("RQ3 - PURCHASE MONTH AND LATE DELIVERY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare sample
# ------------------------------------------------------------

initial_n = len(df_labeled)

rq3 = df_labeled[
    ["order_id", RQ_VARIABLE, "is_late"]
].copy()

missing_n = int(rq3[RQ_VARIABLE].isna().sum())

rq3 = rq3.dropna(
    subset=[RQ_VARIABLE]
)

retained_n = len(rq3)

assert rq3["order_id"].is_unique
assert retained_n + missing_n == initial_n

print("Initial labeled orders:", initial_n)
print("Excluded for missing month:", missing_n)
print("Retained orders:", retained_n)

# ------------------------------------------------------------
# 2. Validate YYYY-MM month format
# ------------------------------------------------------------

parsed_months = pd.to_datetime(
    rq3[RQ_VARIABLE].astype(str),
    format="%Y-%m",
    errors="coerce"
)

invalid_month_count = int(
    parsed_months.isna().sum()
)

if invalid_month_count > 0:
    raise ValueError(
        f"Invalid purchase_month: {invalid_month_count}"
    )

rq3["month_parsed"] = parsed_months

# ------------------------------------------------------------
# 3. Create monthly contingency table
# ------------------------------------------------------------

contingency_rq3 = pd.crosstab(
    rq3[RQ_VARIABLE],
    rq3["is_late"]
).reindex(
    columns=[0, 1],
    fill_value=0
).sort_index()

contingency_rq3.columns = [
    "Not_Late",
    "Late"
]

if len(contingency_rq3) < 2:
    raise ValueError(
        "RQ3 requires at least two purchase months."
    )

if (contingency_rq3.sum(axis=0) == 0).any():
    raise ValueError(
        "RQ3 requires both target classes."
    )

# ------------------------------------------------------------
# 4. Descriptive statistics and Wilson CI
# ------------------------------------------------------------

rq3_summary = contingency_rq3.copy()

rq3_summary["total_orders"] = (
    rq3_summary["Not_Late"]
    + rq3_summary["Late"]
)

rq3_summary["late_rate"] = (
    rq3_summary["Late"]
    / rq3_summary["total_orders"]
)

ci_lower, ci_upper = proportion_confint(
    count=rq3_summary["Late"].to_numpy(),
    nobs=rq3_summary["total_orders"].to_numpy(),
    alpha=ALPHA,
    method="wilson"
)

rq3_summary["ci_lower"] = ci_lower
rq3_summary["ci_upper"] = ci_upper

rq3_summary["late_rate_pct"] = (
    rq3_summary["late_rate"] * 100
)

rq3_summary["ci_lower_pct"] = ci_lower * 100
rq3_summary["ci_upper_pct"] = ci_upper * 100

# Stage 2 EDA convention:
# fewer than 100 valid orders -> small group warning.
rq3_summary["small_sample_flag"] = (
    rq3_summary["total_orders"] < 100
)

print("\nMONTHLY SUMMARY WITH WILSON 95% CI")
display(rq3_summary)

print("\nMONTHS WITH FEWER THAN 100 ORDERS")

small_months = rq3_summary.loc[
    rq3_summary["small_sample_flag"]
]

display(small_months)

# ------------------------------------------------------------
# 5. Check calendar coverage
# ------------------------------------------------------------

month_index = pd.PeriodIndex(
    rq3_summary.index,
    freq="M"
)

full_month_range = pd.period_range(
    start=month_index.min(),
    end=month_index.max(),
    freq="M"
)

missing_months = full_month_range.difference(
    month_index
)

print("\nCALENDAR COVERAGE CHECK")
print("-" * 70)

print("First observed month:", month_index.min())
print("Last observed month:", month_index.max())
print("Observed months:", len(month_index))
print("Calendar months in range:", len(full_month_range))
print(
    "Months with no labeled orders:",
    len(missing_months)
)

if len(missing_months) > 0:
    print("Missing months:", list(map(str, missing_months)))

print(
    "NOTE: Coverage cannot be established "
    "from labeled orders alone."
)
print(
    "Refer to Stage 2 monthly coverage and "
    "missing-target analysis."
)

# ------------------------------------------------------------
# 6. Pearson Chi-square statistic
# ------------------------------------------------------------

observed = contingency_rq3.to_numpy()

chi2_stat, asymptotic_p, dof, expected = (
    stats.chi2_contingency(
        observed,
        correction=False
    )
)

expected_min = float(expected.min())

expected_lt5 = int(
    (expected < 5).sum()
)

expected_lt1 = int(
    (expected < 1).sum()
)

expected_lt5_pct = (
    expected_lt5 / expected.size * 100
)

print("\nCHI-SQUARE DIAGNOSTICS")
print("-" * 70)

print(f"Chi-square statistic: {chi2_stat:.6f}")
print("Degrees of freedom:", dof)
print(f"Asymptotic p-value: {asymptotic_p:.10g}")
print(f"Minimum expected count: {expected_min:.6f}")
print("Expected cells < 5:", expected_lt5)
print("Expected cells < 1:", expected_lt1)
print(f"Percentage of cells < 5: {expected_lt5_pct:.2f}%")

asymptotic_check = (
    expected_min >= 1
    and expected_lt5_pct <= 20
)

print(
    "Expected-count screening:",
    "PASS" if asymptotic_check else "WARNING"
)

# ------------------------------------------------------------
# 7. Conditional Monte Carlo test
# ------------------------------------------------------------

# Use a conditional random-table simulation because
# some early months contain very few observations.
#
# Fixed row totals and column totals.
# The test statistic is Pearson's Chi-square.
#
# The simulation is not a time-series dependence model.

N_RESAMPLES_RQ3 = 20000

rng_rq3 = np.random.default_rng(
    RANDOM_SEED + 3
)

row_totals = observed.sum(axis=1)
col_totals = observed.sum(axis=0)

simulated_tables = stats.random_table.rvs(
    row_totals,
    col_totals,
    size=N_RESAMPLES_RQ3,
    random_state=rng_rq3
)

simulated_chi2 = (
    (
        (simulated_tables - expected) ** 2
        / expected
    ).sum(axis=(1, 2))
)

n_extreme = int(
    np.count_nonzero(
        simulated_chi2 >= chi2_stat - 1e-12
    )
)

monte_carlo_p = (
    (n_extreme + 1)
    / (N_RESAMPLES_RQ3 + 1)
)

raw_p_value = float(monte_carlo_p)

print("\nCONDITIONAL MONTE CARLO TEST")
print("-" * 70)

print("Resamples:", N_RESAMPLES_RQ3)
print("Extreme resamples:", n_extreme)
print(f"Monte Carlo p-value: {monte_carlo_p:.8f}")

if n_extreme == 0:
    print(
        "WARNING: No extreme simulated table observed."
    )
    print(
        "The displayed p-value is limited by "
        "simulation resolution."
    )

# ------------------------------------------------------------
# 8. Cramer's V
# ------------------------------------------------------------

n = observed.sum()
r, c = observed.shape

cramers_v = np.sqrt(
    chi2_stat
    / (n * min(r - 1, c - 1))
)

print("\nEFFECT SIZE")
print("-" * 70)
print(f"Cramer's V: {cramers_v:.6f}")

# ------------------------------------------------------------
# 9. Store results for final summary
# ------------------------------------------------------------

rq3_result = {
    "rq": RQ_ID,
    "variable": RQ_VARIABLE,
    "method": (
        "Chi-square statistic with "
        "conditional Monte Carlo p"
    ),
    "initial_n": initial_n,
    "missing_n": missing_n,
    "n": retained_n,
    "n_groups": len(contingency_rq3),
    "test_statistic": float(chi2_stat),
    "degrees_of_freedom": int(dof),
    "p_value_raw": raw_p_value,
    "p_value_holm": np.nan,
    "effect_size_name": "Cramer's V",
    "effect_size": float(cramers_v),
    "expected_min": expected_min,
    "expected_lt5": expected_lt5,
    "expected_lt1": expected_lt1,
    "monte_carlo_resamples": N_RESAMPLES_RQ3,
    "small_sample_groups": int(
        rq3_summary["small_sample_flag"].sum()
    ),
    "missing_calendar_months": len(missing_months)
}

# ------------------------------------------------------------
# 10. Preliminary conclusion
# ------------------------------------------------------------

print("\nPRELIMINARY RESULT")
print("-" * 70)

if raw_p_value < ALPHA:
    print(
        "Unadjusted evidence of an association "
        "between purchase month and late delivery."
    )
else:
    print(
        "Insufficient unadjusted evidence "
        "to reject independence."
    )

print(
    "Final conclusion requires the agreed "
    "Holm correction."
)
print(
    "Monthly differences do not by themselves "
    "establish seasonality or causation."
)
print(
    "Calendar-time dependence and incomplete "
    "coverage remain limitations."
)

print("\nRQ3 ANALYSIS COMPLETED.")


RQ3 - PURCHASE MONTH AND LATE DELIVERY


Initial labeled orders: 96470
Excluded for missing month: 0
Retained orders: 96470



MONTHLY SUMMARY WITH WILSON 95% CI

,Not_Late,Late,total_orders,late_rate,ci_lower,ci_upper,late_rate_pct,ci_lower_pct,ci_upper_pct,small_sample_flag
purchase_month,,,,,,,,,,
2016-09,0,1,1,1.0000,0.2065,1.0000,100.0000,20.6549,100.0000,True
2016-10,263,2,265,0.0075,0.0021,0.0271,0.7547,0.2072,2.7095,False
2016-12,1,0,1,0.0000,0.0000,0.7935,0.0000,0.0000,79.3451,True
2017-01,728,22,750,0.0293,0.0194,0.0440,2.9333,1.9450,4.4014,False
2017-02,1604,49,1653,0.0296,0.0225,0.0390,2.9643,2.2495,3.8973,False
2017-03,2430,116,2546,0.0456,0.0381,0.0544,4.5562,3.8123,5.4369,False
2017-04,2152,151,2303,0.0656,0.0562,0.0764,6.5567,5.6163,7.6417,False
2017-05,3439,106,3545,0.0299,0.0248,0.0360,2.9901,2.4784,3.6037,False
2017-06,3040,95,3135,0.0303,0.0249,0.0369,3.0303,2.4854,3.6902,False



MONTHS WITH FEWER THAN 100 ORDERS


,Not_Late,Late,total_orders,late_rate,ci_lower,ci_upper,late_rate_pct,ci_lower_pct,ci_upper_pct,small_sample_flag
purchase_month,,,,,,,,,,
2016-09,0,1,1,1.0000,0.2065,1.0000,100.0000,20.6549,100.0000,True
2016-12,1,0,1,0.0000,0.0000,0.7935,0.0000,0.0000,79.3451,True



CALENDAR COVERAGE CHECK
----------------------------------------------------------------------
First observed month: 2016-09
Last observed month: 2018-08
Observed months: 23
Calendar months in range: 24
Months with no labeled orders: 1
Missing months: ['2016-11']
NOTE: Coverage cannot be established from labeled orders alone.
Refer to Stage 2 monthly coverage and missing-target analysis.

CHI-SQUARE DIAGNOSTICS
----------------------------------------------------------------------
Chi-square statistic: 3603.927650
Degrees of freedom: 22
Asymptotic p-value: 0
Minimum expected count: 0.067731
Expected cells < 5: 4
Expected cells < 1: 4
Percentage of cells < 5: 8.70%
Expected-count screening: WARNING



CONDITIONAL MONTE CARLO TEST
----------------------------------------------------------------------
Resamples: 20000
Extreme resamples: 0
Monte Carlo p-value: 0.00005000
The displayed p-value is limited by simulation resolution.

EFFECT SIZE
----------------------------------------------------------------------
Cramer's V: 0.193282

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted evidence of an association between purchase month and late delivery.
Final conclusion requires the agreed Holm correction.
Monthly differences do not by themselves establish seasonality or causation.
Calendar-time dependence and incomplete coverage remain limitations.

RQ3 ANALYSIS COMPLETED.


In [11]:

# ============================================================
# Cell 10: RQ4 - Interstate Seller vs Late Delivery
# ============================================================

from statsmodels.stats.proportion import (
    confint_proportions_2indep,
)

RQ_ID = "RQ4"
RQ_VARIABLE = "has_interstate_seller"

print("RQ4 - INTERSTATE SELLER AND LATE DELIVERY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare sample
# ------------------------------------------------------------

initial_n = len(df_labeled)

rq4 = df_labeled[
    ["order_id", RQ_VARIABLE, "is_late"]
].copy()

missing_n = int(rq4[RQ_VARIABLE].isna().sum())

rq4 = rq4.dropna(subset=[RQ_VARIABLE])

retained_n = len(rq4)

if not rq4[RQ_VARIABLE].isin([0, 1]).all():
    raise ValueError(
        "Unexpected values in has_interstate_seller."
    )

assert rq4["order_id"].is_unique
assert initial_n == missing_n + retained_n

print("Initial labeled orders:", initial_n)
print("Excluded for missing interstate feature:", missing_n)
print("Retained orders:", retained_n)

# ------------------------------------------------------------
# 2. Contingency table
# ------------------------------------------------------------

contingency_rq4 = pd.crosstab(
    rq4[RQ_VARIABLE].astype(int),
    rq4["is_late"]
).reindex(
    index=[0, 1],
    columns=[0, 1],
    fill_value=0
)

contingency_rq4.index = [
    "Same-state seller only",
    "Has interstate seller"
]

contingency_rq4.columns = [
    "Not_Late",
    "Late"
]

if (contingency_rq4.sum(axis=1) == 0).any():
    raise ValueError("One interstate group is empty.")

if (contingency_rq4.sum(axis=0) == 0).any():
    raise ValueError("One target class is empty.")

print("\nCONTINGENCY TABLE")
display(contingency_rq4)

# ------------------------------------------------------------
# 3. Late delivery rates and Wilson 95% CI
# ------------------------------------------------------------

rq4_summary = contingency_rq4.copy()

rq4_summary["total_orders"] = (
    rq4_summary["Not_Late"]
    + rq4_summary["Late"]
)

rq4_summary["late_rate"] = (
    rq4_summary["Late"]
    / rq4_summary["total_orders"]
)

lower, upper = proportion_confint(
    count=rq4_summary["Late"].to_numpy(),
    nobs=rq4_summary["total_orders"].to_numpy(),
    alpha=ALPHA,
    method="wilson"
)

rq4_summary["ci_lower"] = lower
rq4_summary["ci_upper"] = upper

print("\nGROUP SUMMARY WITH WILSON 95% CI")
display(rq4_summary)

# ------------------------------------------------------------
# 4. Difference in proportions with Newcombe CI
# ------------------------------------------------------------

n1 = int(rq4_summary.loc[
    "Has interstate seller", "total_orders"
])

x1 = int(rq4_summary.loc[
    "Has interstate seller", "Late"
])

n0 = int(rq4_summary.loc[
    "Same-state seller only", "total_orders"
])

x0 = int(rq4_summary.loc[
    "Same-state seller only", "Late"
])

p1 = x1 / n1
p0 = x0 / n0

risk_difference = p1 - p0

rd_ci_lower, rd_ci_upper = (
    confint_proportions_2indep(
        count1=x1,
        nobs1=n1,
        count2=x0,
        nobs2=n0,
        compare="diff",
        method="newcomb",
        alpha=ALPHA
    )
)

print("\nRISK DIFFERENCE")
print("-" * 70)
print(f"Interstate late rate: {p1:.4%}")
print(f"Same-state late rate: {p0:.4%}")
print(
    f"Difference: {risk_difference * 100:.4f} "
    "percentage points"
)
print(
    f"Newcombe 95% CI: "
    f"[{rd_ci_lower * 100:.4f}, "
    f"{rd_ci_upper * 100:.4f}] percentage points"
)

# ------------------------------------------------------------
# 5. Chi-square and expected-count diagnostics
# ------------------------------------------------------------

observed = contingency_rq4.to_numpy()

chi2_stat, chi2_p, dof, expected = (
    stats.chi2_contingency(
        observed,
        correction=False
    )
)

expected_min = float(expected.min())
expected_lt5 = int((expected < 5).sum())
expected_lt1 = int((expected < 1).sum())
expected_lt5_pct = (
    expected_lt5 / expected.size * 100
)

asymptotic_check = (
    expected_min >= 1
    and expected_lt5_pct <= 20
)

print("\nCHI-SQUARE DIAGNOSTICS")
print("-" * 70)
print(f"Chi-square: {chi2_stat:.6f}")
print("Degrees of freedom:", dof)
print(f"Pearson p-value: {chi2_p:.10g}")
print(f"Minimum expected count: {expected_min:.4f}")
print("Cells with expected count < 5:", expected_lt5)
print("Cells with expected count < 1:", expected_lt1)
print(
    "Expected-count screening:",
    "PASS" if asymptotic_check else "WARNING"
)

# ------------------------------------------------------------
# 6. Select appropriate test
# ------------------------------------------------------------

if asymptotic_check:
    test_method = "Pearson Chi-square"
    test_statistic = float(chi2_stat)
    raw_p_value = float(chi2_p)
else:
    odds_ratio, fisher_p = stats.fisher_exact(
        observed,
        alternative="two-sided"
    )
    test_method = "Fisher exact"
    test_statistic = float(odds_ratio)
    raw_p_value = float(fisher_p)

print("\nSELECTED TEST")
print("-" * 70)
print("Method:", test_method)
print("Test statistic:", test_statistic)
print(f"Raw p-value: {raw_p_value:.10g}")

# ------------------------------------------------------------
# 7. Effect size: Phi coefficient
# ------------------------------------------------------------

phi = np.sqrt(
    chi2_stat / observed.sum()
)

print("\nEFFECT SIZE")
print("-" * 70)
print(f"Phi coefficient (magnitude): {phi:.6f}")

# ------------------------------------------------------------
# 8. Store result for future Holm adjustment
# ------------------------------------------------------------

rq4_result = {
    "rq": RQ_ID,
    "variable": RQ_VARIABLE,
    "method": test_method,
    "initial_n": initial_n,
    "missing_n": missing_n,
    "n": retained_n,
    "n_groups": 2,
    "test_statistic": test_statistic,
    "degrees_of_freedom": (
        int(dof) if asymptotic_check else np.nan
    ),
    "p_value_raw": raw_p_value,
    "p_value_holm": np.nan,
    "effect_size_name": "Phi magnitude",
    "effect_size": float(phi),
    "estimate_name": "Risk difference",
    "estimate": float(risk_difference),
    "ci_lower": float(rd_ci_lower),
    "ci_upper": float(rd_ci_upper),
    "expected_min": expected_min,
    "expected_lt5": expected_lt5,
    "assumption_screening_pass": bool(asymptotic_check)
}

# ------------------------------------------------------------
# 9. Preliminary interpretation
# ------------------------------------------------------------

print("\nPRELIMINARY RESULT")
print("-" * 70)

if raw_p_value < ALPHA:
    print(
        "Unadjusted evidence that late delivery "
        "rates differ between interstate groups."
    )
else:
    print(
        "Insufficient unadjusted evidence "
        "to reject equal late delivery rates."
    )

print(
    "Final conclusion requires Holm correction."
)
print(
    "This observational association "
    "does not establish causation."
)

print("\nRQ4 ANALYSIS COMPLETED.")


RQ4 - INTERSTATE SELLER AND LATE DELIVERY
Initial labeled orders: 96470
Excluded for missing interstate feature: 0
Retained orders: 96470

CONTINGENCY TABLE


,Not_Late,Late
Same-state seller only,33012,1563
Has interstate seller,56924,4971



GROUP SUMMARY WITH WILSON 95% CI


,Not_Late,Late,total_orders,late_rate,ci_lower,ci_upper
Same-state seller only,33012,1563,34575,0.0452,0.0431,0.0474
Has interstate seller,56924,4971,61895,0.0803,0.0782,0.0825



RISK DIFFERENCE
----------------------------------------------------------------------
Interstate late rate: 8.0313%
Same-state late rate: 4.5206%
Difference: 3.5107 percentage points
Newcombe 95% CI: [3.2026, 3.8153] percentage points

CHI-SQUARE DIAGNOSTICS
----------------------------------------------------------------------
Chi-square: 433.005775
Degrees of freedom: 1
Pearson p-value: 3.603184736e-96
Minimum expected count: 2341.7959
Cells with expected count < 5: 0
Cells with expected count < 1: 0
Expected-count screening: PASS

SELECTED TEST
----------------------------------------------------------------------
Method: Pearson Chi-square
Test statistic: 433.0057753697845
Raw p-value: 3.603184736e-96

EFFECT SIZE
----------------------------------------------------------------------
Phi coefficient (magnitude): 0.066996

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted evidence that late delivery rates differ between interstate

In [12]:

# ============================================================
# Cell 11: RQ5 - Delivery Distance vs Late Delivery
# ============================================================

# Research question:
# Does the distribution of delivery distance differ
# between Late and Not Late orders?
#
# H0: Both distance distributions are identical.
# H1: The distributions differ.
#
# Method: Two-sided Mann-Whitney U test.
# Effect size: Rank-biserial correlation.
# CI: Stratified bootstrap percentile CI (exploratory).
#
# Holm adjustment will be applied after all RQs.

RQ5_ID = "RQ5"
RQ5_VARIABLE = "distance_km_mean"

print("RQ5 - DELIVERY DISTANCE VS LATE DELIVERY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare analysis sample
# ------------------------------------------------------------

rq5_initial_n = len(df_labeled)

rq5 = df_labeled[
    ["order_id", "is_late", RQ5_VARIABLE]
].copy()

rq5_missing_n = int(
    rq5[RQ5_VARIABLE].isna().sum()
)

rq5 = rq5.dropna(
    subset=[RQ5_VARIABLE]
).copy()

# Validate numeric data without silently replacing values.

rq5_distance = pd.to_numeric(
    rq5[RQ5_VARIABLE],
    errors="raise"
)

rq5_nonfinite_n = int(
    (~np.isfinite(rq5_distance.to_numpy())).sum()
)

if rq5_nonfinite_n > 0:
    raise ValueError(
        f"Found {rq5_nonfinite_n} non-finite distances. "
        "Review the source data before testing."
    )

rq5_negative_n = int(
    (rq5_distance < 0).sum()
)

if rq5_negative_n > 0:
    raise ValueError(
        f"Found {rq5_negative_n} negative distances. "
        "Review data quality before testing."
    )

rq5[RQ5_VARIABLE] = rq5_distance

rq5_retained_n = len(rq5)

assert rq5["order_id"].is_unique
assert (
    rq5_initial_n
    == rq5_missing_n + rq5_retained_n
)

print("INITIAL SAMPLE")
print("-" * 70)

print("Initial labeled orders:", rq5_initial_n)
print("Missing distance:", rq5_missing_n)
print("Non-finite distance:", rq5_nonfinite_n)
print("Negative distance:", rq5_negative_n)
print("Retained orders:", rq5_retained_n)

# ------------------------------------------------------------
# 2. Split Late and Not Late groups
# ------------------------------------------------------------

late_distance = rq5.loc[
    rq5["is_late"] == 1,
    RQ5_VARIABLE
].to_numpy(dtype=float)

not_late_distance = rq5.loc[
    rq5["is_late"] == 0,
    RQ5_VARIABLE
].to_numpy(dtype=float)

n_late = len(late_distance)
n_not_late = len(not_late_distance)

if min(n_late, n_not_late) == 0:
    raise ValueError(
        "Both Late and Not Late groups are required."
    )

print("\nGROUP SAMPLE SIZES")
print("-" * 70)

print("Late:", n_late)
print("Not Late:", n_not_late)
print("Total:", n_late + n_not_late)

assert n_late + n_not_late == rq5_retained_n

# ------------------------------------------------------------
# 3. Descriptive statistics
# ------------------------------------------------------------

def rq5_describe(values, group_name):
    return {
        "group": group_name,
        "n": len(values),
        "mean_km": float(np.mean(values)),
        "sd_km": float(np.std(values, ddof=1)),
        "median_km": float(np.median(values)),
        "q1_km": float(np.percentile(values, 25)),
        "q3_km": float(np.percentile(values, 75)),
        "min_km": float(np.min(values)),
        "max_km": float(np.max(values))
    }

rq5_descriptive = pd.DataFrame([
    rq5_describe(late_distance, "Late"),
    rq5_describe(not_late_distance, "Not Late")
])

print("\nDESCRIPTIVE STATISTICS (KM)")
print("-" * 70)

display(rq5_descriptive)

# ------------------------------------------------------------
# 4. Check distribution and ties
# ------------------------------------------------------------

combined_distance = np.concatenate([
    late_distance,
    not_late_distance
])

unique_n = np.unique(combined_distance).size

tie_count = len(combined_distance) - unique_n

print("\nDISTRIBUTION DIAGNOSTICS")
print("-" * 70)

print("Unique distance values:", unique_n)
print("Repeated-value count beyond unique values:", tie_count)
print("Zero distances:", int((combined_distance == 0).sum()))

print(
    "NOTE: The distance is based on geographic "
    "coordinates, not actual road distance."
)

# ------------------------------------------------------------
# 5. Mann-Whitney U test
# ------------------------------------------------------------

mw_result = stats.mannwhitneyu(
    late_distance,
    not_late_distance,
    alternative="two-sided",
    method="asymptotic",
    use_continuity=True
)

u_stat = float(mw_result.statistic)
raw_p = float(mw_result.pvalue)

print("\nMANN-WHITNEY U TEST")
print("-" * 70)

print(f"U statistic (Late group): {u_stat:.4f}")
print(f"Raw p-value: {raw_p:.10g}")
print("Method: Asymptotic, tie-aware")
print("Alternative: Two-sided")

# ------------------------------------------------------------
# 6. Rank-biserial effect size
# ------------------------------------------------------------

# Positive value:
# Late distances tend to rank higher than Not Late.
#
# Negative value:
# Late distances tend to rank lower.
#
# Ties contribute half a win.

probability_superiority = (
    u_stat / (n_late * n_not_late)
)

rank_biserial = (
    2 * probability_superiority - 1
)

print("\nEFFECT SIZE")
print("-" * 70)

print(
    f"Probability superiority (Late > Not Late, "
    f"ties half): {probability_superiority:.6f}"
)

print(
    f"Rank-biserial correlation: {rank_biserial:.6f}"
)

# ------------------------------------------------------------
# 7. Bootstrap CI for rank-biserial correlation
# ------------------------------------------------------------

# Preliminary stratified bootstrap:
# resample orders within each target class.
#
# IMPORTANT:
# This assumes independently resampled orders.
# Seller/time clustering is not accounted for.
# The resulting CI is exploratory, not cluster-robust.

RQ5_BOOTSTRAP_REPS = 400

rq5_rng = np.random.default_rng(
    RANDOM_SEED + 5
)

bootstrap_effects = np.empty(
    RQ5_BOOTSTRAP_REPS,
    dtype=float
)

for b in range(RQ5_BOOTSTRAP_REPS):

    boot_late = rq5_rng.choice(
        late_distance,
        size=n_late,
        replace=True
    )

    boot_not_late = rq5_rng.choice(
        not_late_distance,
        size=n_not_late,
        replace=True
    )

    boot_combined = np.concatenate([
        boot_late,
        boot_not_late
    ])

    ranks = stats.rankdata(
        boot_combined,
        method="average"
    )

    rank_sum_late = ranks[:n_late].sum()

    boot_u = (
        rank_sum_late
        - n_late * (n_late + 1) / 2
    )

    bootstrap_effects[b] = (
        2 * boot_u / (n_late * n_not_late) - 1
    )

rq5_ci_lower, rq5_ci_upper = np.percentile(
    bootstrap_effects,
    [2.5, 97.5]
)

print("\nEXPLORATORY BOOTSTRAP 95% CI")
print("-" * 70)

print("Bootstrap replications:", RQ5_BOOTSTRAP_REPS)
print(f"Rank-biserial estimate: {rank_biserial:.6f}")

print(
    f"95% CI: [{rq5_ci_lower:.6f}, "
    f"{rq5_ci_upper:.6f}]"
)

print(
    "WARNING: CI uses order-level resampling "
    "and is not robust to seller/time clustering."
)

# ------------------------------------------------------------
# 8. Store results for final reporting
# ------------------------------------------------------------

rq5_result = {
    "rq": RQ5_ID,
    "variable": RQ5_VARIABLE,
    "method": "Mann-Whitney U (asymptotic, two-sided)",
    "initial_n": int(rq5_initial_n),
    "missing_n": int(rq5_missing_n),
    "n": int(rq5_retained_n),
    "n_late": int(n_late),
    "n_not_late": int(n_not_late),
    "test_statistic": u_stat,
    "degrees_of_freedom": np.nan,
    "p_value_raw": raw_p,
    "p_value_holm": np.nan,
    "effect_size_name": "Rank-biserial correlation",
    "effect_size": float(rank_biserial),
    "estimate_name": "Probability superiority",
    "estimate": float(probability_superiority),
    "ci_lower": float(rq5_ci_lower),
    "ci_upper": float(rq5_ci_upper),
    "ci_method": "Order-level stratified bootstrap percentile",
    "bootstrap_reps": RQ5_BOOTSTRAP_REPS,
    "cluster_robust_ci": False
}

# ------------------------------------------------------------
# 9. Preliminary interpretation
# ------------------------------------------------------------

print("\nPRELIMINARY RESULT")
print("-" * 70)

if raw_p < ALPHA:
    print(
        "Unadjusted evidence that distance "
        "distributions differ between the two groups."
    )
else:
    print(
        "Insufficient unadjusted evidence "
        "to reject identical distance distributions."
    )

if rank_biserial > 0:
    print(
        "Late orders tend to have higher "
        "distance ranks."
    )
elif rank_biserial < 0:
    print(
        "Late orders tend to have lower "
        "distance ranks."
    )
else:
    print(
        "No rank-based direction observed."
    )

print(
    "Do not interpret this test automatically "
    "as a median-difference test."
)

print(
    "Final inference requires Holm adjustment "
    "and review of dependence assumptions."
)

print(
    "The observed association does not "
    "establish causation."
)

print("\nRQ5 ANALYSIS COMPLETED.")


RQ5 - DELIVERY DISTANCE VS LATE DELIVERY
INITIAL SAMPLE
----------------------------------------------------------------------
Initial labeled orders: 96470
Missing distance: 480
Non-finite distance: 0
Negative distance: 0
Retained orders: 95990

GROUP SAMPLE SIZES
----------------------------------------------------------------------
Late: 6489
Not Late: 89501
Total: 95990

DESCRIPTIVE STATISTICS (KM)
----------------------------------------------------------------------


,group,n,mean_km,sd_km,median_km,q1_km,q3_km,min_km,max_km
0,Late,6489,766.9787,686.7176,528.7212,316.0224,969.7181,0.0000,3710.4202
1,Not Late,89501,588.4024,582.9303,426.3774,174.0882,783.4783,0.0000,3398.5482



DISTRIBUTION DIAGNOSTICS
----------------------------------------------------------------------
Unique distance values: 90360
Repeated-value count beyond unique values: 5630
Zero distances: 23
NOTE: The distance is based on geographic coordinates, not actual road distance.

MANN-WHITNEY U TEST
----------------------------------------------------------------------
U statistic (Late group): 336702963.0000
Raw p-value: 1.9806304e-102
Method: Asymptotic, tie-aware
Alternative: Two-sided

EFFECT SIZE
----------------------------------------------------------------------
Probability superiority (Late > Not Late, ties half): 0.579751
Rank-biserial correlation: 0.159501



EXPLORATORY BOOTSTRAP 95% CI
----------------------------------------------------------------------
Bootstrap replications: 400
Rank-biserial estimate: 0.159501
95% CI: [0.146281, 0.172708]

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted evidence that distance distributions differ between the two groups.
Late orders tend to have higher distance ranks.
Do not interpret this test automatically as a median-difference test.
Final inference requires Holm adjustment and review of dependence assumptions.
The observed association does not establish causation.

RQ5 ANALYSIS COMPLETED.


In [13]:

# ============================================================
# Cell 12: RQ6 - Order Value & Freight Value
# ============================================================

# Research question:
# Do order value and freight cost means differ
# between Late and Not Late orders?
#
# RQ6a: order_total_value
# RQ6b: freight_value
#
# H0: Mean(Late) = Mean(Not Late)
# H1: Mean(Late) != Mean(Not Late)
#
# Each variable is tested separately.
# Method: Welch's two-sample t-test, two-sided.
# Alpha = 0.05.
# Final inference requires Holm adjustment.

RQ6_VARIABLES = {
    "order_total_value": "BRL",
    "freight_value": "BRL"
}

rq6_results = []
rq6_sample_records = []
rq6_descriptive_records = []

print("RQ6 - ORDER VALUE AND FREIGHT COST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Analyze each variable separately
# ------------------------------------------------------------

for variable, unit in RQ6_VARIABLES.items():

    print(f"\nVARIABLE: {variable}")
    print("=" * 70)

    initial_n = len(df_labeled)

    sample = df_labeled[
        ["order_id", "is_late", variable]
    ].copy()

    # Count missing values only for the current variable.
    missing_n = int(sample[variable].isna().sum())

    sample = sample.dropna(
        subset=[variable]
    ).copy()

    # Validate numeric values.
    values = pd.to_numeric(
        sample[variable],
        errors="raise"
    )

    nonfinite_n = int(
        (~np.isfinite(values.to_numpy(dtype=float))).sum()
    )

    if nonfinite_n > 0:
        raise ValueError(
            f"{variable} contains {nonfinite_n} "
            "non-finite values. Review source data."
        )

    sample[variable] = values.astype(float)

    negative_n = int(
        (sample[variable] < 0).sum()
    )

    if negative_n > 0:
        print(
            f"WARNING: {negative_n} negative values "
            "found. Verify their meaning."
        )

    retained_n = len(sample)

    assert sample["order_id"].is_unique
    assert initial_n == missing_n + retained_n

    # --------------------------------------------------------
    # 2. Split Late and Not Late
    # --------------------------------------------------------

    late = sample.loc[
        sample["is_late"] == 1,
        variable
    ].to_numpy(dtype=float)

    not_late = sample.loc[
        sample["is_late"] == 0,
        variable
    ].to_numpy(dtype=float)

    n1 = len(late)
    n0 = len(not_late)

    if min(n1, n0) < 2:
        raise ValueError(
            f"{variable}: Both groups require "
            "at least two observations."
        )

    print("\nSAMPLE SUMMARY")
    print("-" * 70)

    print("Initial labeled orders:", initial_n)
    print("Missing values:", missing_n)
    print("Retained orders:", retained_n)
    print("Late:", n1)
    print("Not Late:", n0)

    assert n1 + n0 == retained_n

    # --------------------------------------------------------
    # 3. Descriptive statistics
    # --------------------------------------------------------

    descriptive_rows = []

    for label, group_values in [
        ("Late", late),
        ("Not Late", not_late)
    ]:

        descriptive_rows.append({
            "variable": variable,
            "group": label,
            "unit": unit,
            "n": len(group_values),
            "mean": float(np.mean(group_values)),
            "sd": float(np.std(group_values, ddof=1)),
            "median": float(np.median(group_values)),
            "q1": float(np.percentile(group_values, 25)),
            "q3": float(np.percentile(group_values, 75)),
            "min": float(np.min(group_values)),
            "max": float(np.max(group_values))
        })

    descriptive_df = pd.DataFrame(descriptive_rows)

    display(descriptive_df)

    rq6_descriptive_records.extend(
        descriptive_rows
    )

    # --------------------------------------------------------
    # 4. Distribution diagnostics
    # --------------------------------------------------------

    print("\nDISTRIBUTION DIAGNOSTICS")
    print("-" * 70)

    for label, group_values in [
        ("Late", late),
        ("Not Late", not_late)
    ]:

        skewness = stats.skew(
            group_values,
            bias=False
        )

        q1, q3 = np.percentile(
            group_values,
            [25, 75]
        )

        iqr = q3 - q1

        upper_fence = q3 + 1.5 * iqr
        lower_fence = q1 - 1.5 * iqr

        outlier_count = int(
            (
                (group_values < lower_fence)
                | (group_values > upper_fence)
            ).sum()
        )

        print(f"{label}:")
        print(f"  Skewness: {skewness:.4f}")
        print(f"  IQR outliers: {outlier_count}")

    print(
        "IQR outliers are diagnostic only; "
        "no observations are automatically removed."
    )

    # --------------------------------------------------------
    # 5. Welch's t-test
    # --------------------------------------------------------

    result = stats.ttest_ind(
        late,
        not_late,
        equal_var=False,
        alternative="two-sided"
    )

    t_stat = float(result.statistic)
    p_value = float(result.pvalue)

    # --------------------------------------------------------
    # 6. Mean difference and Welch 95% CI
    # --------------------------------------------------------

    mean1 = float(np.mean(late))
    mean0 = float(np.mean(not_late))

    var1 = float(np.var(late, ddof=1))
    var0 = float(np.var(not_late, ddof=1))

    mean_difference = mean1 - mean0

    a = var1 / n1
    b = var0 / n0

    standard_error = np.sqrt(a + b)

    if standard_error <= 0:
        raise ValueError(
            f"{variable}: Standard error is zero."
        )

    welch_df = (
        (a + b) ** 2
        / (
            a ** 2 / (n1 - 1)
            + b ** 2 / (n0 - 1)
        )
    )

    critical_t = stats.t.ppf(
        1 - ALPHA / 2,
        df=welch_df
    )

    ci_lower = (
        mean_difference - critical_t * standard_error
    )

    ci_upper = (
        mean_difference + critical_t * standard_error
    )

    print("\nWELCH'S T-TEST")
    print("-" * 70)

    print(f"Mean Late: {mean1:.4f} {unit}")
    print(f"Mean Not Late: {mean0:.4f} {unit}")
    print(
        f"Mean difference (Late - Not Late): "
        f"{mean_difference:.4f} {unit}"
    )

    print(f"t-statistic: {t_stat:.6f}")
    print(f"Welch degrees of freedom: {welch_df:.4f}")
    print(f"Raw p-value: {p_value:.10g}")

    print(
        f"95% CI for mean difference: "
        f"[{ci_lower:.4f}, {ci_upper:.4f}] {unit}"
    )

    # --------------------------------------------------------
    # 7. Store sample information
    # --------------------------------------------------------

    rq6_sample_records.append({
        "rq": "RQ6",
        "variable": variable,
        "initial_n": int(initial_n),
        "missing_n": int(missing_n),
        "retained_n": int(retained_n),
        "n_late": int(n1),
        "n_not_late": int(n0),
        "negative_values": negative_n,
        "nonfinite_values": nonfinite_n
    })

    # --------------------------------------------------------
    # 8. Store statistical test result
    # --------------------------------------------------------

    rq6_results.append({
        "rq": "RQ6",
        "variable": variable,
        "method": "Welch two-sample t-test",
        "initial_n": int(initial_n),
        "missing_n": int(missing_n),
        "n": int(retained_n),
        "n_late": int(n1),
        "n_not_late": int(n0),
        "test_statistic": t_stat,
        "degrees_of_freedom": float(welch_df),
        "p_value_raw": p_value,
        "p_value_holm": np.nan,
        "effect_size_name": "Mean difference",
        "effect_size": float(mean_difference),
        "estimate_name": "Mean difference",
        "estimate": float(mean_difference),
        "ci_lower": float(ci_lower),
        "ci_upper": float(ci_upper),
        "ci_method": "Welch t interval",
        "unit": unit
    })

    # --------------------------------------------------------
    # 9. Preliminary interpretation
    # --------------------------------------------------------

    print("\nPRELIMINARY RESULT")
    print("-" * 70)

    if p_value < ALPHA:
        print(
            "Unadjusted evidence of a difference "
            "in group means."
        )
    else:
        print(
            "Insufficient unadjusted evidence "
            "to reject equal group means."
        )

    print(
        "Final inference requires Holm adjustment."
    )

    print(
        "Distribution skewness, outliers and "
        "potential order dependence must be considered."
    )

    print("-" * 70)

# ------------------------------------------------------------
# 10. Combined RQ6 results
# ------------------------------------------------------------

rq6_results_df = pd.DataFrame(
    rq6_results
)

rq6_sample_df = pd.DataFrame(
    rq6_sample_records
)

rq6_descriptive_df = pd.DataFrame(
    rq6_descriptive_records
)

assert len(rq6_results_df) == 2

print("\nRQ6 - COMBINED STATISTICAL RESULTS")
print("=" * 70)

display(rq6_results_df)

print("\nRQ6 ANALYSIS COMPLETED.")


RQ6 - ORDER VALUE AND FREIGHT COST

VARIABLE: order_total_value

SAMPLE SUMMARY
----------------------------------------------------------------------
Initial labeled orders: 96470
Missing values: 0
Retained orders: 96470
Late: 6534
Not Late: 89936


,variable,group,unit,n,mean,sd,median,q1,q3,min,max
0,order_total_value,Late,BRL,6534,176.1390,240.8889,114.0950,67.5075,190.5700,14.2900,6922.2100
1,order_total_value,Not Late,BRL,89936,158.6406,217.0635,104.5900,61.5900,175.3425,9.5900,13664.0800



DISTRIBUTION DIAGNOSTICS
----------------------------------------------------------------------
Late:
  Skewness: 7.6081
  IQR outliers: 536
Not Late:
  Skewness: 9.5333
  IQR outliers: 7032
IQR outliers are diagnostic only; no observations are automatically removed.

WELCH'S T-TEST
----------------------------------------------------------------------
Mean Late: 176.1390 BRL
Mean Not Late: 158.6406 BRL
Mean difference (Late - Not Late): 17.4984 BRL
t-statistic: 5.705899
Welch degrees of freedom: 7324.6590
Raw p-value: 1.202441418e-08
95% CI for mean difference: [11.4867, 23.5100] BRL

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted evidence of a difference in group means.
Final inference requires Holm adjustment.
Distribution skewness, outliers and potential order dependence must be considered.
----------------------------------------------------------------------

VARIABLE: freight_value

SAMPLE SUMMARY
---------------------------

,variable,group,unit,n,mean,sd,median,q1,q3,min,max
0,freight_value,Late,BRL,6534,25.2476,23.7156,18.2300,15.1000,26.9000,0.0000,711.3300
1,freight_value,Not Late,BRL,89936,22.6069,21.3840,17.0600,13.7100,23.8500,0.0000,1794.9600



DISTRIBUTION DIAGNOSTICS
----------------------------------------------------------------------
Late:
  Skewness: 7.6579
  IQR outliers: 662
Not Late:
  Skewness: 12.7322
  IQR outliers: 8971
IQR outliers are diagnostic only; no observations are automatically removed.

WELCH'S T-TEST
----------------------------------------------------------------------
Mean Late: 25.2476 BRL
Mean Not Late: 22.6069 BRL
Mean difference (Late - Not Late): 2.6407 BRL
t-statistic: 8.745906
Welch degrees of freedom: 7325.7256
Raw p-value: 2.711482804e-18
95% CI for mean difference: [2.0488, 3.2325] BRL

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted evidence of a difference in group means.
Final inference requires Holm adjustment.
Distribution skewness, outliers and potential order dependence must be considered.
----------------------------------------------------------------------

RQ6 - COMBINED STATISTICAL RESULTS


,rq,variable,method,initial_n,missing_n,n,n_late,n_not_late,test_statistic,degrees_of_freedom,p_value_raw,p_value_holm,effect_size_name,effect_size,estimate_name,estimate,ci_lower,ci_upper,ci_method,unit
0,RQ6,order_total_value,Welch two-sample t-test,96470,0,96470,6534,89936,5.7059,7324.6590,0.0000,NaN,Mean difference,17.4984,Mean difference,17.4984,11.4867,23.5100,Welch t interval,BRL
1,RQ6,freight_value,Welch two-sample t-test,96470,0,96470,6534,89936,8.7459,7325.7256,0.0000,NaN,Mean difference,2.6407,Mean difference,2.6407,2.0488,3.2325,Welch t interval,BRL



RQ6 ANALYSIS COMPLETED.


In [14]:

# ============================================================
# Cell 13: RQ6 - Sensitivity Analysis
# ============================================================

# Purpose:
# Assess sensitivity of RQ6 findings to skewness
# and extreme values.
#
# Main analysis:
#   Welch's t-test for ordinary means (Cell 12)
#
# Sensitivity analysis:
#   10% trimmed Welch / Yuen's test
#
# This is exploratory and does NOT replace the
# pre-specified RQ6 primary tests or Holm family.

RQ6_TRIM = 0.10

rq6_sensitivity_results = []

print("RQ6 - SENSITIVITY ANALYSIS")
print("=" * 70)

for variable, unit in RQ6_VARIABLES.items():

    print(f"\nVARIABLE: {variable}")
    print("-" * 70)

    # --------------------------------------------------------
    # 1. Prepare data
    # --------------------------------------------------------

    sample = df_labeled[
        ["is_late", variable]
    ].dropna(subset=[variable]).copy()

    sample[variable] = pd.to_numeric(
        sample[variable],
        errors="raise"
    )

    values = sample[variable].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError(
            f"{variable} contains non-finite values."
        )

    late = sample.loc[
        sample["is_late"] == 1,
        variable
    ].to_numpy(dtype=float)

    not_late = sample.loc[
        sample["is_late"] == 0,
        variable
    ].to_numpy(dtype=float)

    # --------------------------------------------------------
    # 2. Calculate ordinary and trimmed means
    # --------------------------------------------------------

    mean_late = float(np.mean(late))
    mean_not_late = float(np.mean(not_late))

    ordinary_diff = mean_late - mean_not_late

    trimmed_mean_late = float(
        stats.trim_mean(late, proportiontocut=RQ6_TRIM)
    )

    trimmed_mean_not_late = float(
        stats.trim_mean(not_late, proportiontocut=RQ6_TRIM)
    )

    trimmed_diff = (
        trimmed_mean_late - trimmed_mean_not_late
    )

    # --------------------------------------------------------
    # 3. Yuen's two-sample trimmed t-test
    # --------------------------------------------------------

    sensitivity_test = stats.ttest_ind(
        late,
        not_late,
        equal_var=False,
        alternative="two-sided",
        trim=RQ6_TRIM
    )

    sensitivity_t = float(
        sensitivity_test.statistic
    )

    sensitivity_p = float(
        sensitivity_test.pvalue
    )

    sensitivity_df = float(
        sensitivity_test.df
    )

    # --------------------------------------------------------
    # 4. Diagnostics
    # --------------------------------------------------------

    skew_late = float(
        stats.skew(late, bias=False)
    )

    skew_not_late = float(
        stats.skew(not_late, bias=False)
    )

    # --------------------------------------------------------
    # 5. Print results
    # --------------------------------------------------------

    print("Sample Late:", len(late))
    print("Sample Not Late:", len(not_late))

    print("\nORDINARY MEAN DIFFERENCE")
    print(f"Late mean: {mean_late:.4f} {unit}")
    print(f"Not Late mean: {mean_not_late:.4f} {unit}")
    print(f"Difference: {ordinary_diff:.4f} {unit}")

    print("\n10% TRIMMED MEAN DIFFERENCE")

    print(
        f"Trimmed mean Late: "
        f"{trimmed_mean_late:.4f} {unit}"
    )

    print(
        f"Trimmed mean Not Late: "
        f"{trimmed_mean_not_late:.4f} {unit}"
    )

    print(
        f"Trimmed difference: "
        f"{trimmed_diff:.4f} {unit}"
    )

    print("\nYUEN'S TEST")
    print(f"t-statistic: {sensitivity_t:.6f}")
    print(f"Degrees of freedom: {sensitivity_df:.4f}")
    print(f"Raw p-value: {sensitivity_p:.10g}")

    print("\nSKEWNESS")
    print(f"Late: {skew_late:.4f}")
    print(f"Not Late: {skew_not_late:.4f}")

    # --------------------------------------------------------
    # 6. Store sensitivity result
    # --------------------------------------------------------

    rq6_sensitivity_results.append({
        "rq": "RQ6",
        "variable": variable,
        "unit": unit,
        "trim_proportion_each_tail": RQ6_TRIM,
        "n_late": len(late),
        "n_not_late": len(not_late),
        "ordinary_mean_difference": ordinary_diff,
        "trimmed_mean_difference": trimmed_diff,
        "yuen_t_statistic": sensitivity_t,
        "yuen_df": sensitivity_df,
        "sensitivity_p_raw": sensitivity_p,
        "skewness_late": skew_late,
        "skewness_not_late": skew_not_late
    })

# ------------------------------------------------------------
# 7. Combined sensitivity results
# ------------------------------------------------------------

rq6_sensitivity_df = pd.DataFrame(
    rq6_sensitivity_results
)

print("\nRQ6 - SENSITIVITY SUMMARY")
print("=" * 70)

display(rq6_sensitivity_df)

print(
    "\nNOTE: Trimmed means and ordinary means "
    "are different estimands."
)

print(
    "Sensitivity p-values are exploratory; "
    "they do not replace the primary "
    "Holm-adjusted tests."
)

print(
    "Neither analysis establishes causation "
    "or resolves dependence between orders."
)

print("\nRQ6 SENSITIVITY ANALYSIS COMPLETED.")


RQ6 - SENSITIVITY ANALYSIS

VARIABLE: order_total_value
----------------------------------------------------------------------
Sample Late: 6534
Sample Not Late: 89936

ORDINARY MEAN DIFFERENCE
Late mean: 176.1390 BRL
Not Late mean: 158.6406 BRL
Difference: 17.4984 BRL

10% TRIMMED MEAN DIFFERENCE
Trimmed mean Late: 130.8297 BRL
Trimmed mean Not Late: 119.0753 BRL
Trimmed difference: 11.7543 BRL

YUEN'S TEST
t-statistic: 7.936167
Degrees of freedom: 5856.0128
Raw p-value: 2.480204407e-15

SKEWNESS
Late: 7.6081
Not Late: 9.5333

VARIABLE: freight_value
----------------------------------------------------------------------


Sample Late: 6534
Sample Not Late: 89936

ORDINARY MEAN DIFFERENCE
Late mean: 25.2476 BRL
Not Late mean: 22.6069 BRL
Difference: 2.6407 BRL

10% TRIMMED MEAN DIFFERENCE
Trimmed mean Late: 20.9189 BRL
Trimmed mean Not Late: 18.8996 BRL
Trimmed difference: 2.0194 BRL

YUEN'S TEST
t-statistic: 12.195496
Degrees of freedom: 5821.6784
Raw p-value: 8.465361044e-34

SKEWNESS
Late: 7.6579
Not Late: 12.7322

RQ6 - SENSITIVITY SUMMARY


,rq,variable,unit,trim_proportion_each_tail,n_late,n_not_late,ordinary_mean_difference,trimmed_mean_difference,yuen_t_statistic,yuen_df,sensitivity_p_raw,skewness_late,skewness_not_late
0,RQ6,order_total_value,BRL,0.1000,6534,89936,17.4984,11.7543,7.9362,5856.0128,0.0000,7.6081,9.5333
1,RQ6,freight_value,BRL,0.1000,6534,89936,2.6407,2.0194,12.1955,5821.6784,0.0000,7.6579,12.7322



NOTE: Trimmed means and ordinary means are different estimands.
Sensitivity p-values are exploratory; they do not replace the primary Holm-adjusted tests.
Neither analysis establishes causation or resolves dependence between orders.

RQ6 SENSITIVITY ANALYSIS COMPLETED.


In [15]:

# ============================================================
# Cell 14: RQ7 - Relative Freight Cost & Mean Item Value
# ============================================================

# Research question:
# Do relative freight costs and mean item values differ
# between Late and Not Late orders?
#
# Three separate tests:
# RQ7a: freight_to_items_ratio
# RQ7b: mean_item_value
# RQ7c: freight_per_item
#
# H0: Mean(Late) = Mean(Not Late)
# H1: Mean(Late) != Mean(Not Late)
#
# Method: Welch's two-sample t-test (two-sided).
# Alpha = 0.05.
# Final conclusions require Holm correction.

RQ7_VARIABLES = {
    "freight_to_items_ratio": "ratio",
    "mean_item_value": "BRL/item",
    "freight_per_item": "BRL/item"
}

rq7_results = []
rq7_sample_records = []
rq7_descriptive_records = []
rq7_diagnostic_records = []

print("RQ7 - RELATIVE FREIGHT COST & MEAN ITEM VALUE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Analyze each variable independently
# ------------------------------------------------------------

for variable, unit in RQ7_VARIABLES.items():

    print(f"\nVARIABLE: {variable}")
    print("=" * 70)

    # --------------------------------------------------------
    # 2. Prepare the sample
    # --------------------------------------------------------

    initial_n = len(df_labeled)

    sample = df_labeled[
        ["order_id", "is_late", variable]
    ].copy()

    missing_n = int(
        sample[variable].isna().sum()
    )

    sample = sample.dropna(
        subset=[variable]
    ).copy()

    sample[variable] = pd.to_numeric(
        sample[variable],
        errors="raise"
    ).astype(float)

    # Check infinite and non-finite values.
    numeric_values = sample[variable].to_numpy()

    nonfinite_n = int(
        (~np.isfinite(numeric_values)).sum()
    )

    if nonfinite_n > 0:
        raise ValueError(
            f"{variable}: found {nonfinite_n} "
            "non-finite values. Review source data."
        )

    negative_n = int(
        (numeric_values < 0).sum()
    )

    zero_n = int(
        (numeric_values == 0).sum()
    )

    if negative_n > 0:
        raise ValueError(
            f"{variable}: found {negative_n} "
            "negative values. Verify data definitions."
        )

    retained_n = len(sample)

    assert sample["order_id"].is_unique
    assert initial_n == missing_n + retained_n

    print("\nSAMPLE SUMMARY")
    print("-" * 70)

    print("Initial labeled orders:", initial_n)
    print("Missing values:", missing_n)
    print("Non-finite values:", nonfinite_n)
    print("Negative values:", negative_n)
    print("Zero values:", zero_n)
    print("Retained orders:", retained_n)

    # --------------------------------------------------------
    # 3. Split Late / Not Late
    # --------------------------------------------------------

    late = sample.loc[
        sample["is_late"] == 1,
        variable
    ].to_numpy(dtype=float)

    not_late = sample.loc[
        sample["is_late"] == 0,
        variable
    ].to_numpy(dtype=float)

    n1 = len(late)
    n0 = len(not_late)

    if min(n1, n0) < 2:
        raise ValueError(
            f"{variable}: insufficient observations "
            "for Welch's t-test."
        )

    print("Late orders:", n1)
    print("Not Late orders:", n0)

    assert n1 + n0 == retained_n

    # --------------------------------------------------------
    # 4. Descriptive statistics
    # --------------------------------------------------------

    descriptive_rows = []

    for group_label, group_values in [
        ("Late", late),
        ("Not Late", not_late)
    ]:

        descriptive_rows.append({
            "rq": "RQ7",
            "variable": variable,
            "group": group_label,
            "unit": unit,
            "n": len(group_values),
            "mean": float(np.mean(group_values)),
            "sd": float(np.std(group_values, ddof=1)),
            "median": float(np.median(group_values)),
            "q1": float(np.percentile(group_values, 25)),
            "q3": float(np.percentile(group_values, 75)),
            "min": float(np.min(group_values)),
            "max": float(np.max(group_values))
        })

    descriptive_df = pd.DataFrame(
        descriptive_rows
    )

    print("\nDESCRIPTIVE STATISTICS")
    print("-" * 70)

    display(descriptive_df)

    rq7_descriptive_records.extend(
        descriptive_rows
    )

    # --------------------------------------------------------
    # 5. Distribution diagnostics
    # --------------------------------------------------------

    print("\nDISTRIBUTION DIAGNOSTICS")
    print("-" * 70)

    for group_label, group_values in [
        ("Late", late),
        ("Not Late", not_late)
    ]:

        skewness = float(
            stats.skew(group_values, bias=False)
        )

        q1, q3 = np.percentile(
            group_values,
            [25, 75]
        )

        iqr = q3 - q1

        lower_fence = q1 - 1.5 * iqr
        upper_fence = q3 + 1.5 * iqr

        outlier_count = int(
            (
                (group_values < lower_fence)
                | (group_values > upper_fence)
            ).sum()
        )

        print(f"{group_label}:")
        print(f"  Skewness: {skewness:.4f}")
        print(f"  IQR outliers: {outlier_count}")
        print(f"  Maximum value: {np.max(group_values):.4f}")

        rq7_diagnostic_records.append({
            "rq": "RQ7",
            "variable": variable,
            "group": group_label,
            "skewness": skewness,
            "iqr_outliers": outlier_count,
            "max_value": float(np.max(group_values))
        })

    print(
        "NOTE: No outliers are automatically removed."
    )

    # --------------------------------------------------------
    # 6. Welch's two-sample t-test
    # --------------------------------------------------------

    welch_result = stats.ttest_ind(
        late,
        not_late,
        equal_var=False,
        alternative="two-sided"
    )

    t_stat = float(
        welch_result.statistic
    )

    p_value = float(
        welch_result.pvalue
    )

    # --------------------------------------------------------
    # 7. Mean difference and Welch 95% CI
    # --------------------------------------------------------

    mean1 = float(np.mean(late))
    mean0 = float(np.mean(not_late))

    var1 = float(np.var(late, ddof=1))
    var0 = float(np.var(not_late, ddof=1))

    mean_difference = mean1 - mean0

    a = var1 / n1
    b = var0 / n0

    standard_error = np.sqrt(a + b)

    if standard_error <= 0:
        raise ValueError(
            f"{variable}: invalid standard error."
        )

    welch_df = (
        (a + b) ** 2
        / (
            a ** 2 / (n1 - 1)
            + b ** 2 / (n0 - 1)
        )
    )

    critical_t = stats.t.ppf(
        1 - ALPHA / 2,
        df=welch_df
    )

    ci_lower = (
        mean_difference
        - critical_t * standard_error
    )

    ci_upper = (
        mean_difference
        + critical_t * standard_error
    )

    print("\nWELCH'S T-TEST")
    print("-" * 70)

    print(f"Mean Late: {mean1:.6f} {unit}")
    print(f"Mean Not Late: {mean0:.6f} {unit}")

    print(
        f"Mean difference (Late - Not Late): "
        f"{mean_difference:.6f} {unit}"
    )

    print(f"t-statistic: {t_stat:.6f}")
    print(f"Degrees of freedom: {welch_df:.4f}")
    print(f"Raw p-value: {p_value:.10g}")

    print(
        f"Welch 95% CI: "
        f"[{ci_lower:.6f}, {ci_upper:.6f}] {unit}"
    )

    # --------------------------------------------------------
    # 8. Store sample summary
    # --------------------------------------------------------

    rq7_sample_records.append({
        "rq": "RQ7",
        "variable": variable,
        "initial_n": int(initial_n),
        "missing_n": int(missing_n),
        "retained_n": int(retained_n),
        "n_late": int(n1),
        "n_not_late": int(n0),
        "negative_values": negative_n,
        "zero_values": zero_n,
        "nonfinite_values": nonfinite_n
    })

    # --------------------------------------------------------
    # 9. Store statistical test result
    # --------------------------------------------------------

    rq7_results.append({
        "rq": "RQ7",
        "variable": variable,
        "method": "Welch two-sample t-test",
        "initial_n": int(initial_n),
        "missing_n": int(missing_n),
        "n": int(retained_n),
        "n_late": int(n1),
        "n_not_late": int(n0),
        "test_statistic": t_stat,
        "degrees_of_freedom": float(welch_df),
        "p_value_raw": p_value,
        "p_value_holm": np.nan,
        "effect_size_name": "Mean difference",
        "effect_size": float(mean_difference),
        "estimate_name": "Mean difference",
        "estimate": float(mean_difference),
        "ci_lower": float(ci_lower),
        "ci_upper": float(ci_upper),
        "ci_method": "Welch t interval",
        "unit": unit
    })

    # --------------------------------------------------------
    # 10. Preliminary interpretation
    # --------------------------------------------------------

    print("\nPRELIMINARY RESULT")
    print("-" * 70)

    if p_value < ALPHA:
        print(
            "Unadjusted evidence of a difference "
            "in group means."
        )
    else:
        print(
            "Insufficient unadjusted evidence "
            "to reject equal group means."
        )

    if mean_difference > 0:
        print("The Late group has the higher mean.")
    elif mean_difference < 0:
        print("The Not Late group has the higher mean.")
    else:
        print("The observed group means are equal.")

    print(
        "Final conclusion requires Holm correction."
    )

    print(
        "Skewness, outliers, and dependence "
        "must be considered."
    )

# ------------------------------------------------------------
# 11. Combined RQ7 results
# ------------------------------------------------------------

rq7_results_df = pd.DataFrame(
    rq7_results
)

rq7_sample_df = pd.DataFrame(
    rq7_sample_records
)

rq7_descriptive_df = pd.DataFrame(
    rq7_descriptive_records
)

rq7_diagnostics_df = pd.DataFrame(
    rq7_diagnostic_records
)

assert len(rq7_results_df) == 3
assert len(rq7_sample_df) == 3
assert len(rq7_descriptive_df) == 6

print("\nRQ7 - COMBINED STATISTICAL RESULTS")
print("=" * 70)

display(rq7_results_df)

print("\nRQ7 ANALYSIS COMPLETED.")


RQ7 - RELATIVE FREIGHT COST & MEAN ITEM VALUE

VARIABLE: freight_to_items_ratio



SAMPLE SUMMARY
----------------------------------------------------------------------
Initial labeled orders: 96470
Missing values: 0
Non-finite values: 0
Negative values: 0
Zero values: 336
Retained orders: 96470
Late orders: 6534
Not Late orders: 89936

DESCRIPTIVE STATISTICS
----------------------------------------------------------------------


,rq,variable,group,unit,n,mean,sd,median,q1,q3,min,max
0,RQ7,freight_to_items_ratio,Late,ratio,6534,0.3185,0.3210,0.2276,0.1316,0.3872,0.0000,5.2914
1,RQ7,freight_to_items_ratio,Not Late,ratio,89936,0.3077,0.3109,0.2244,0.1320,0.3801,0.0000,21.4471



DISTRIBUTION DIAGNOSTICS
----------------------------------------------------------------------
Late:
  Skewness: 4.0364
  IQR outliers: 440
  Maximum value: 5.2914
Not Late:
  Skewness: 10.1211
  IQR outliers: 5915
  Maximum value: 21.4471
NOTE: No outliers are automatically removed.

WELCH'S T-TEST
----------------------------------------------------------------------
Mean Late: 0.318467 ratio
Mean Not Late: 0.307653 ratio
Mean difference (Late - Not Late): 0.010814 ratio
t-statistic: 2.634689
Degrees of freedom: 7451.3566
Raw p-value: 0.008438862137
Welch 95% CI: [0.002768, 0.018859] ratio

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted evidence of a difference in group means.
The Late group has the higher mean.
Final conclusion requires Holm correction.
Skewness, outliers, and dependence must be considered.

VARIABLE: mean_item_value



SAMPLE SUMMARY
----------------------------------------------------------------------
Initial labeled orders: 96470
Missing values: 0
Non-finite values: 0
Negative values: 0
Zero values: 0
Retained orders: 96470
Late orders: 6534
Not Late orders: 89936

DESCRIPTIVE STATISTICS
----------------------------------------------------------------------


,rq,variable,group,unit,n,mean,sd,median,q1,q3,min,max
0,RQ7,mean_item_value,Late,BRL/item,6534,142.0190,222.7879,86.9000,48.0000,150.0000,1.5143,6729.0000
1,RQ7,mean_item_value,Not Late,BRL/item,89936,124.0119,187.0158,78.9900,40.9000,139.0000,0.8500,6735.0000


... (2 outputs left out)

,rq,variable,group,unit,n,mean,sd,median,q1,q3,min,max
0,RQ7,freight_per_item,Late,BRL/item,6534,22.9676,18.5360,17.7400,15.1000,23.4675,0.0000,237.1100
1,RQ7,freight_per_item,Not Late,BRL/item,89936,19.9437,15.4396,16.2500,13.1600,21.0325,0.0000,409.6800



DISTRIBUTION DIAGNOSTICS
----------------------------------------------------------------------
Late:
  Skewness: 4.3875
  IQR outliers: 806
  Maximum value: 237.1100
Not Late:
  Skewness: 5.7621
  IQR outliers: 9543
  Maximum value: 409.6800
NOTE: No outliers are automatically removed.

WELCH'S T-TEST
----------------------------------------------------------------------
Mean Late: 22.967615 BRL/item
Mean Not Late: 19.943696 BRL/item
Mean difference (Late - Not Late): 3.023919 BRL/item
t-statistic: 12.866653
Degrees of freedom: 7206.8843
Raw p-value: 1.788928709e-37
Welch 95% CI: [2.563211, 3.484627] BRL/item

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted evidence of a difference in group means.
The Late group has the higher mean.
Final conclusion requires Holm correction.
Skewness, outliers, and dependence must be considered.

RQ7 - COMBINED STATISTICAL RESULTS


,rq,variable,method,initial_n,missing_n,n,n_late,n_not_late,test_statistic,degrees_of_freedom,p_value_raw,p_value_holm,effect_size_name,effect_size,estimate_name,estimate,ci_lower,ci_upper,ci_method,unit
0,RQ7,freight_to_items_ratio,Welch two-sample t-test,96470,0,96470,6534,89936,2.6347,7451.3566,0.0084,NaN,Mean difference,0.0108,Mean difference,0.0108,0.0028,0.0189,Welch t interval,ratio
1,RQ7,mean_item_value,Welch two-sample t-test,96470,0,96470,6534,89936,6.3724,7217.6483,0.0000,NaN,Mean difference,18.0071,Mean difference,18.0071,12.4677,23.5465,Welch t interval,BRL/item
2,RQ7,freight_per_item,Welch two-sample t-test,96470,0,96470,6534,89936,12.8667,7206.8843,0.0000,NaN,Mean difference,3.0239,Mean difference,3.0239,2.5632,3.4846,Welch t interval,BRL/item



RQ7 ANALYSIS COMPLETED.


In [16]:

# ============================================================
# Cell 15: RQ7 - Sensitivity Analysis
# ============================================================

# Compare primary Welch results against
# 10% trimmed Welch / Yuen's tests.
#
# These are supplementary analyses.
# They do NOT replace the three primary RQ7 tests
# and are not added to the planned primary Holm family.
#
# Ordinary and trimmed means are different estimands.

RQ7_TRIM = 0.10

rq7_sensitivity_records = []

print("RQ7 - SENSITIVITY ANALYSIS")
print("=" * 70)

for variable, unit in RQ7_VARIABLES.items():

    print(f"\nVARIABLE: {variable}")
    print("-" * 70)

    # --------------------------------------------------------
    # 1. Prepare the same sample as the primary test
    # --------------------------------------------------------

    sample = df_labeled[
        ["is_late", variable]
    ].dropna(subset=[variable]).copy()

    sample[variable] = pd.to_numeric(
        sample[variable],
        errors="raise"
    ).astype(float)

    if not np.isfinite(sample[variable].to_numpy()).all():
        raise ValueError(
            f"{variable} contains non-finite values."
        )

    late = sample.loc[
        sample["is_late"] == 1,
        variable
    ].to_numpy(dtype=float)

    not_late = sample.loc[
        sample["is_late"] == 0,
        variable
    ].to_numpy(dtype=float)

    assert len(late) >= 2
    assert len(not_late) >= 2

    # --------------------------------------------------------
    # 2. Original mean difference
    # --------------------------------------------------------

    ordinary_diff = float(
        np.mean(late) - np.mean(not_late)
    )

    # --------------------------------------------------------
    # 3. Trimmed mean difference
    # --------------------------------------------------------

    trimmed_mean_late = float(
        stats.trim_mean(
            late,
            proportiontocut=RQ7_TRIM
        )
    )

    trimmed_mean_not_late = float(
        stats.trim_mean(
            not_late,
            proportiontocut=RQ7_TRIM
        )
    )

    trimmed_diff = (
        trimmed_mean_late - trimmed_mean_not_late
    )

    # --------------------------------------------------------
    # 4. Yuen's test
    # --------------------------------------------------------

    yuen_result = stats.ttest_ind(
        late,
        not_late,
        equal_var=False,
        alternative="two-sided",
        trim=RQ7_TRIM
    )

    yuen_t = float(yuen_result.statistic)
    yuen_df = float(yuen_result.df)
    yuen_p = float(yuen_result.pvalue)

    # --------------------------------------------------------
    # 5. Descriptive distribution diagnostics
    # --------------------------------------------------------

    median_late = float(np.median(late))
    median_not_late = float(np.median(not_late))

    skew_late = float(
        stats.skew(late, bias=False)
    )

    skew_not_late = float(
        stats.skew(not_late, bias=False)
    )

    # --------------------------------------------------------
    # 6. Display results
    # --------------------------------------------------------

    print("Sample Late:", len(late))
    print("Sample Not Late:", len(not_late))

    print("\nORDINARY MEAN DIFFERENCE")
    print(f"{ordinary_diff:.6f} {unit}")

    print("\n10% TRIMMED MEANS")
    print(
        f"Trimmed Late: "
        f"{trimmed_mean_late:.6f} {unit}"
    )
    print(
        f"Trimmed Not Late: "
        f"{trimmed_mean_not_late:.6f} {unit}"
    )
    print(
        f"Trimmed difference: "
        f"{trimmed_diff:.6f} {unit}"
    )

    print("\nYUEN'S TEST")
    print(f"t-statistic: {yuen_t:.6f}")
    print(f"Degrees of freedom: {yuen_df:.4f}")
    print(f"Raw p-value: {yuen_p:.10g}")

    print("\nMEDIAN COMPARISON (DESCRIPTIVE)")
    print(f"Median Late: {median_late:.6f}")
    print(f"Median Not Late: {median_not_late:.6f}")

    print("\nSKEWNESS")
    print(f"Late: {skew_late:.4f}")
    print(f"Not Late: {skew_not_late:.4f}")

    # --------------------------------------------------------
    # 7. Store sensitivity result
    # --------------------------------------------------------

    rq7_sensitivity_records.append({
        "rq": "RQ7",
        "variable": variable,
        "unit": unit,
        "trim_proportion_each_tail": RQ7_TRIM,
        "n_late": len(late),
        "n_not_late": len(not_late),
        "ordinary_mean_difference": ordinary_diff,
        "trimmed_mean_difference": float(trimmed_diff),
        "yuen_t_statistic": yuen_t,
        "yuen_df": yuen_df,
        "sensitivity_p_raw": yuen_p,
        "median_late": median_late,
        "median_not_late": median_not_late,
        "skewness_late": skew_late,
        "skewness_not_late": skew_not_late
    })

# ------------------------------------------------------------
# 8. Combined table
# ------------------------------------------------------------

rq7_sensitivity_df = pd.DataFrame(
    rq7_sensitivity_records
)

assert len(rq7_sensitivity_df) == 3

print("\nRQ7 - SENSITIVITY SUMMARY")
print("=" * 70)

display(
    rq7_sensitivity_df[
        [
            "variable",
            "n_late",
            "n_not_late",
            "ordinary_mean_difference",
            "trimmed_mean_difference",
            "yuen_t_statistic",
            "yuen_df",
            "sensitivity_p_raw",
            "median_late",
            "median_not_late"
        ]
    ]
)

print(
    "\nNOTE: Ordinary and trimmed means "
    "measure different quantities."
)
print(
    "These sensitivity tests do not replace "
    "the primary Welch tests."
)
print(
    "Order-level dependence remains a limitation."
)

print("\nRQ7 SENSITIVITY ANALYSIS COMPLETED.")


RQ7 - SENSITIVITY ANALYSIS

VARIABLE: freight_to_items_ratio
----------------------------------------------------------------------


Sample Late: 6534
Sample Not Late: 89936

ORDINARY MEAN DIFFERENCE
0.010814 ratio

10% TRIMMED MEANS
Trimmed Late: 0.260877 ratio
Trimmed Not Late: 0.255856 ratio
Trimmed difference: 0.005021 ratio

YUEN'S TEST
t-statistic: 1.723612
Degrees of freedom: 5938.5336
Raw p-value: 0.0848300357

MEDIAN COMPARISON (DESCRIPTIVE)
Median Late: 0.227568
Median Not Late: 0.224374

SKEWNESS
Late: 4.0364
Not Late: 10.1211

VARIABLE: mean_item_value
----------------------------------------------------------------------


Sample Late: 6534
Sample Not Late: 89936

ORDINARY MEAN DIFFERENCE
18.007127 BRL/item

10% TRIMMED MEANS
Trimmed Late: 100.635616 BRL/item
Trimmed Not Late: 89.720153 BRL/item
Trimmed difference: 10.915463 BRL/item

YUEN'S TEST
t-statistic: 8.575581
Degrees of freedom: 5773.3651
Raw p-value: 1.251432932e-17

MEDIAN COMPARISON (DESCRIPTIVE)
Median Late: 86.900000
Median Not Late: 78.990000

SKEWNESS
Late: 8.4845
Not Late: 7.7601

VARIABLE: freight_per_item
----------------------------------------------------------------------
Sample Late: 6534
Sample Not Late: 89936

ORDINARY MEAN DIFFERENCE
3.023919 BRL/item

10% TRIMMED MEANS
Trimmed Late: 19.422459 BRL/item
Trimmed Not Late: 17.189024 BRL/item
Trimmed difference: 2.233434 BRL/item

YUEN'S TEST
t-statistic: 16.902486
Degrees of freedom: 5795.5878
Raw p-value: 1.336146362e-62

MEDIAN COMPARISON (DESCRIPTIVE)
Median Late: 17.740000
Median Not Late: 16.250000

SKEWNESS
Late: 4.3875
Not Late: 5.7621

RQ7 - SENSITIVITY SUMMARY


,variable,n_late,n_not_late,ordinary_mean_difference,trimmed_mean_difference,yuen_t_statistic,yuen_df,sensitivity_p_raw,median_late,median_not_late
0,freight_to_items_ratio,6534,89936,0.0108,0.0050,1.7236,5938.5336,0.0848,0.2276,0.2244
1,mean_item_value,6534,89936,18.0071,10.9155,8.5756,5773.3651,0.0000,86.9000,78.9900
2,freight_per_item,6534,89936,3.0239,2.2334,16.9025,5795.5878,0.0000,17.7400,16.2500



NOTE: Ordinary and trimmed means measure different quantities.
These sensitivity tests do not replace the primary Welch tests.
Order-level dependence remains a limitation.

RQ7 SENSITIVITY ANALYSIS COMPLETED.


In [17]:

# ============================================================
# Cell 16: RQ8 - Product Weight & Volume
# ============================================================

# RQ8a: total_product_weight_g
# RQ8b: total_product_volume_cm3
#
# Primary method:
# Two-sided Mann-Whitney U test
#
# Effect size:
# Rank-biserial correlation
#
# Confidence interval:
# Stratified order-level bootstrap percentile CI
#
# Final inference requires Holm correction.

RQ8_VARIABLES = {
    "total_product_weight_g": "g",
    "total_product_volume_cm3": "cm3"
}

RQ8_BOOTSTRAP_REPS = 300

rq8_results = []
rq8_sample_records = []
rq8_descriptive_records = []
rq8_diagnostic_records = []

print("RQ8 - PRODUCT WEIGHT & VOLUME")
print("=" * 70)

# ------------------------------------------------------------
# 1. Process each variable independently
# ------------------------------------------------------------

for variable_index, (variable, unit) in enumerate(
    RQ8_VARIABLES.items()
):

    print(f"\nVARIABLE: {variable}")
    print("=" * 70)

    # --------------------------------------------------------
    # 2. Prepare sample
    # --------------------------------------------------------

    initial_n = len(df_labeled)

    sample = df_labeled[
        ["order_id", "is_late", variable]
    ].copy()

    missing_n = int(
        sample[variable].isna().sum()
    )

    sample = sample.dropna(
        subset=[variable]
    ).copy()

    sample[variable] = pd.to_numeric(
        sample[variable],
        errors="raise"
    ).astype(float)

    values = sample[variable].to_numpy()

    nonfinite_n = int(
        (~np.isfinite(values)).sum()
    )

    if nonfinite_n > 0:
        raise ValueError(
            f"{variable}: {nonfinite_n} "
            "non-finite values found."
        )

    negative_n = int(
        (values < 0).sum()
    )

    if negative_n > 0:
        raise ValueError(
            f"{variable}: {negative_n} "
            "negative values found."
        )

    zero_n = int(
        (values == 0).sum()
    )

    retained_n = len(sample)

    assert sample["order_id"].is_unique
    assert initial_n == missing_n + retained_n

    print("\nSAMPLE SUMMARY")
    print("-" * 70)

    print("Initial labeled orders:", initial_n)
    print("Missing values:", missing_n)
    print("Retained orders:", retained_n)
    print("Zero values:", zero_n)
    print("Negative values:", negative_n)
    print("Non-finite values:", nonfinite_n)

    # --------------------------------------------------------
    # 3. Split target groups
    # --------------------------------------------------------

    late = sample.loc[
        sample["is_late"] == 1,
        variable
    ].to_numpy(dtype=float)

    not_late = sample.loc[
        sample["is_late"] == 0,
        variable
    ].to_numpy(dtype=float)

    n1 = len(late)
    n0 = len(not_late)

    if min(n1, n0) < 2:
        raise ValueError(
            f"{variable}: insufficient observations."
        )

    assert n1 + n0 == retained_n

    print("Late orders:", n1)
    print("Not Late orders:", n0)

    # --------------------------------------------------------
    # 4. Descriptive statistics
    # --------------------------------------------------------

    descriptive_rows = []

    for group_name, group_values in [
        ("Late", late),
        ("Not Late", not_late)
    ]:

        descriptive_rows.append({
            "rq": "RQ8",
            "variable": variable,
            "group": group_name,
            "unit": unit,
            "n": len(group_values),
            "mean": float(np.mean(group_values)),
            "sd": float(
                np.std(group_values, ddof=1)
            ),
            "median": float(np.median(group_values)),
            "q1": float(
                np.percentile(group_values, 25)
            ),
            "q3": float(
                np.percentile(group_values, 75)
            ),
            "min": float(np.min(group_values)),
            "max": float(np.max(group_values))
        })

    descriptive_df = pd.DataFrame(
        descriptive_rows
    )

    rq8_descriptive_records.extend(
        descriptive_rows
    )

    print("\nDESCRIPTIVE STATISTICS")
    print("-" * 70)

    display(descriptive_df)

    # --------------------------------------------------------
    # 5. Distribution diagnostics
    # --------------------------------------------------------

    combined = np.concatenate([
        late,
        not_late
    ])

    unique_count = int(
        np.unique(combined).size
    )

    repeated_count = (
        len(combined) - unique_count
    )

    skew_late = float(
        stats.skew(late, bias=False)
    )

    skew_not_late = float(
        stats.skew(not_late, bias=False)
    )

    print("\nDISTRIBUTION DIAGNOSTICS")
    print("-" * 70)

    print("Unique values:", unique_count)
    print("Repeated values beyond unique:", repeated_count)
    print("Skewness Late:", round(skew_late, 4))
    print("Skewness Not Late:", round(skew_not_late, 4))

    rq8_diagnostic_records.append({
        "rq": "RQ8",
        "variable": variable,
        "unique_values": unique_count,
        "repeated_values": repeated_count,
        "skewness_late": skew_late,
        "skewness_not_late": skew_not_late,
        "zero_values": zero_n
    })

    # --------------------------------------------------------
    # 6. Mann-Whitney U test
    # --------------------------------------------------------

    mw_result = stats.mannwhitneyu(
        late,
        not_late,
        alternative="two-sided",
        method="asymptotic",
        use_continuity=True
    )

    u_stat = float(
        mw_result.statistic
    )

    p_value = float(
        mw_result.pvalue
    )

    print("\nMANN-WHITNEY U TEST")
    print("-" * 70)

    print(f"U statistic (Late group): {u_stat:.4f}")
    print(f"Raw p-value: {p_value:.10g}")
    print("Method: asymptotic with tie correction")

    # --------------------------------------------------------
    # 7. Rank-biserial correlation
    # --------------------------------------------------------

    probability_superiority = (
        u_stat / (n1 * n0)
    )

    rank_biserial = (
        2 * probability_superiority - 1
    )

    print("\nEFFECT SIZE")
    print("-" * 70)

    print(
        "Probability superiority:",
        f"{probability_superiority:.6f}"
    )

    print(
        "Rank-biserial correlation:",
        f"{rank_biserial:.6f}"
    )

    # --------------------------------------------------------
    # 8. Bootstrap effect-size CI
    # --------------------------------------------------------

    # Resample observations within each target group.
    # Ties receive average ranks.
    #
    # This is an exploratory order-level CI,
    # not a seller/time cluster-robust interval.

    rng = np.random.default_rng(
        RANDOM_SEED + 8 + variable_index
    )

    boot_effects = np.empty(
        RQ8_BOOTSTRAP_REPS,
        dtype=float
    )

    for bootstrap_index in range(
        RQ8_BOOTSTRAP_REPS
    ):

        boot_late = rng.choice(
            late,
            size=n1,
            replace=True
        )

        boot_not_late = rng.choice(
            not_late,
            size=n0,
            replace=True
        )

        boot_values = np.concatenate([
            boot_late,
            boot_not_late
        ])

        ranks = stats.rankdata(
            boot_values,
            method="average"
        )

        rank_sum_late = float(
            ranks[:n1].sum()
        )

        boot_u = (
            rank_sum_late
            - n1 * (n1 + 1) / 2
        )

        boot_effects[bootstrap_index] = (
            2 * boot_u / (n1 * n0) - 1
        )

    ci_lower, ci_upper = np.percentile(
        boot_effects,
        [2.5, 97.5]
    )

    print("\nEXPLORATORY BOOTSTRAP 95% CI")
    print("-" * 70)

    print(
        "Bootstrap replications:",
        RQ8_BOOTSTRAP_REPS
    )

    print(
        f"95% CI for rank-biserial: "
        f"[{ci_lower:.6f}, {ci_upper:.6f}]"
    )

    print(
        "WARNING: The CI does not account "
        "for seller/time clustering."
    )

    # --------------------------------------------------------
    # 9. Save sample audit in memory
    # --------------------------------------------------------

    rq8_sample_records.append({
        "rq": "RQ8",
        "variable": variable,
        "initial_n": int(initial_n),
        "missing_n": int(missing_n),
        "retained_n": int(retained_n),
        "n_late": int(n1),
        "n_not_late": int(n0),
        "zero_values": int(zero_n)
    })

    # --------------------------------------------------------
    # 10. Store one result per test
    # --------------------------------------------------------

    rq8_results.append({
        "rq": "RQ8",
        "variable": variable,
        "method": "Mann-Whitney U (two-sided)",
        "initial_n": int(initial_n),
        "missing_n": int(missing_n),
        "n": int(retained_n),
        "n_late": int(n1),
        "n_not_late": int(n0),
        "test_statistic": u_stat,
        "degrees_of_freedom": np.nan,
        "p_value_raw": p_value,
        "p_value_holm": np.nan,
        "effect_size_name": "Rank-biserial correlation",
        "effect_size": float(rank_biserial),
        "estimate_name": "Probability superiority",
        "estimate": float(probability_superiority),
        "ci_lower": float(ci_lower),
        "ci_upper": float(ci_upper),
        "ci_method": (
            "Stratified order-level bootstrap percentile"
        ),
        "bootstrap_reps": RQ8_BOOTSTRAP_REPS,
        "unit": unit,
        "cluster_robust_ci": False
    })

    # --------------------------------------------------------
    # 11. Preliminary result
    # --------------------------------------------------------

    print("\nPRELIMINARY RESULT")
    print("-" * 70)

    if p_value < ALPHA:
        print(
            "Unadjusted evidence that the "
            "distributions differ."
        )
    else:
        print(
            "Insufficient unadjusted evidence "
            "to reject identical distributions."
        )

    if rank_biserial > 0:
        print(
            "Late orders tend to have "
            "higher-ranked values."
        )
    elif rank_biserial < 0:
        print(
            "Late orders tend to have "
            "lower-ranked values."
        )
    else:
        print(
            "No rank-based direction observed."
        )

    print(
        "Final inference requires Holm adjustment."
    )

    print(
        "Mann-Whitney U does not automatically "
        "test a difference in medians."
    )

    print("-" * 70)

# ------------------------------------------------------------
# 12. Combined RQ8 tables
# ------------------------------------------------------------

rq8_results_df = pd.DataFrame(
    rq8_results
)

rq8_sample_df = pd.DataFrame(
    rq8_sample_records
)

rq8_descriptive_df = pd.DataFrame(
    rq8_descriptive_records
)

rq8_diagnostics_df = pd.DataFrame(
    rq8_diagnostic_records
)

assert len(rq8_results_df) == 2
assert len(rq8_sample_df) == 2
assert len(rq8_descriptive_df) == 4

print("\nRQ8 - COMBINED STATISTICAL RESULTS")
print("=" * 70)

display(
    rq8_results_df[
        [
            "rq",
            "variable",
            "n",
            "missing_n",
            "n_late",
            "n_not_late",
            "test_statistic",
            "p_value_raw",
            "effect_size",
            "ci_lower",
            "ci_upper",
            "unit"
        ]
    ]
)

print("\nRQ8 ANALYSIS COMPLETED.")


RQ8 - PRODUCT WEIGHT & VOLUME

VARIABLE: total_product_weight_g

SAMPLE SUMMARY
----------------------------------------------------------------------
Initial labeled orders: 96470
Missing values: 22
Retained orders: 96448
Zero values: 0
Negative values: 0
Non-finite values: 0
Late orders: 6534
Not Late orders: 89914

DESCRIPTIVE STATISTICS
----------------------------------------------------------------------


,rq,variable,group,unit,n,mean,sd,median,q1,q3,min,max
0,RQ8,total_product_weight_g,Late,g,6534,2735.6374,5593.7164,817.0000,340.0000,2392.0000,50.0000,144300.0000
1,RQ8,total_product_weight_g,Not Late,g,89914,2361.6827,4703.7929,750.0000,300.0000,2037.2500,2.0000,184400.0000



DISTRIBUTION DIAGNOSTICS
----------------------------------------------------------------------
Unique values: 2858
Repeated values beyond unique: 93590
Skewness Late: 7.1691
Skewness Not Late: 6.4706

MANN-WHITNEY U TEST
----------------------------------------------------------------------
U statistic (Late group): 304873689.5000
Raw p-value: 3.043394643e-07
Method: asymptotic with tie correction

EFFECT SIZE
----------------------------------------------------------------------
Probability superiority: 0.518936
Rank-biserial correlation: 0.037871



EXPLORATORY BOOTSTRAP 95% CI
----------------------------------------------------------------------
Bootstrap replications: 300
95% CI for rank-biserial: [0.026322, 0.051129]

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted evidence that the distributions differ.
Late orders tend to have higher-ranked values.
Final inference requires Holm adjustment.
Mann-Whitney U does not automatically test a difference in medians.
----------------------------------------------------------------------

VARIABLE: total_product_volume_cm3

SAMPLE SUMMARY
----------------------------------------------------------------------
Initial labeled orders: 96470
Missing values: 16
Retained orders: 96454
Zero values: 0
Negative values: 0
Non-finite values: 0
Late orders: 6534
Not Late orders: 89920

DESCRIPTIVE STATISTICS
----------------------------------------------------------------------


,rq,variable,group,unit,n,mean,sd,median,q1,q3,min,max
0,RQ8,total_product_volume_cm3,Late,cm3,6534,19029.3251,33883.6940,7629.0000,2964.0000,20944.0000,352.0000,864000.0000
1,RQ8,total_product_volume_cm3,Not Late,cm3,89920,17208.5968,29850.3144,7200.0000,2964.0000,19800.0000,168.0000,1476000.0000



DISTRIBUTION DIAGNOSTICS
----------------------------------------------------------------------
Unique values: 6262
Repeated values beyond unique: 90192
Skewness Late: 7.0134
Skewness Not Late: 7.3794

MANN-WHITNEY U TEST
----------------------------------------------------------------------
U statistic (Late group): 300302511.0000
Raw p-value: 0.002641235837
Method: asymptotic with tie correction

EFFECT SIZE
----------------------------------------------------------------------
Probability superiority: 0.511121
Rank-biserial correlation: 0.022242



EXPLORATORY BOOTSTRAP 95% CI
----------------------------------------------------------------------
Bootstrap replications: 300
95% CI for rank-biserial: [0.008577, 0.034281]

PRELIMINARY RESULT
----------------------------------------------------------------------
Unadjusted evidence that the distributions differ.
Late orders tend to have higher-ranked values.
Final inference requires Holm adjustment.
Mann-Whitney U does not automatically test a difference in medians.
----------------------------------------------------------------------

RQ8 - COMBINED STATISTICAL RESULTS


,rq,variable,n,missing_n,n_late,n_not_late,test_statistic,p_value_raw,effect_size,ci_lower,ci_upper,unit
0,RQ8,total_product_weight_g,96448,22,6534,89914,304873689.5000,0.0000,0.0379,0.0263,0.0511,g
1,RQ8,total_product_volume_cm3,96454,16,6534,89920,300302511.0000,0.0026,0.0222,0.0086,0.0343,cm3



RQ8 ANALYSIS COMPLETED.


In [18]:

# ============================================================
# Cell 17: RQ9 - Seller Count vs Late Delivery
# ============================================================

from statsmodels.stats.proportion import (
    confint_proportions_2indep
)

# Research question:
# Does the late delivery rate differ between
# single-seller and multiple-seller orders?
#
# Single seller: seller_count == 1
# Multiple sellers: seller_count >= 2
#
# H0: p_multiple = p_single
# H1: p_multiple != p_single
#
# Method: Pearson Chi-square or Fisher exact.
# Effect: Risk difference, Phi coefficient.
# Final inference requires Holm adjustment.

print("RQ9 - SELLER COUNT VS LATE DELIVERY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Prepare and validate sample
# ------------------------------------------------------------

initial_n = len(df_labeled)

rq9 = df_labeled[
    ["order_id", "is_late", "seller_count"]
].copy()

missing_n = int(
    rq9["seller_count"].isna().sum()
)

rq9 = rq9.dropna(
    subset=["seller_count"]
).copy()

rq9["seller_count"] = pd.to_numeric(
    rq9["seller_count"],
    errors="raise"
).astype(float)

values = rq9["seller_count"].to_numpy()

if not np.isfinite(values).all():
    raise ValueError(
        "seller_count contains non-finite values."
    )

invalid_n = int(
    ((values < 1) | (values != np.floor(values))).sum()
)

if invalid_n > 0:
    raise ValueError(
        f"Found {invalid_n} invalid seller counts. "
        "Review the data before grouping."
    )

rq9["seller_count"] = (
    rq9["seller_count"].astype(int)
)

assert rq9["order_id"].is_unique

# ------------------------------------------------------------
# 2. Inspect original seller counts
# ------------------------------------------------------------

print("\nORIGINAL SELLER COUNT DISTRIBUTION")
print("-" * 70)

display(
    rq9["seller_count"]
    .value_counts()
    .sort_index()
    .rename_axis("seller_count")
    .to_frame("orders")
)

# ------------------------------------------------------------
# 3. Define groups
# ------------------------------------------------------------

rq9["seller_count_group"] = np.where(
    rq9["seller_count"] == 1,
    "Single seller",
    "Multiple sellers"
)

retained_n = len(rq9)

assert initial_n == missing_n + retained_n

print("\nSAMPLE SUMMARY")
print("-" * 70)

print("Initial labeled orders:", initial_n)
print("Missing seller_count:", missing_n)
print("Retained orders:", retained_n)

# ------------------------------------------------------------
# 4. Build contingency table
# ------------------------------------------------------------

group_order = [
    "Single seller",
    "Multiple sellers"
]

contingency_rq9 = pd.crosstab(
    rq9["seller_count_group"],
    rq9["is_late"]
).reindex(
    index=group_order,
    columns=[0, 1],
    fill_value=0
)

contingency_rq9.columns = [
    "Not_Late",
    "Late"
]

if (contingency_rq9.sum(axis=1) == 0).any():
    raise ValueError(
        "One seller-count group is empty. "
        "RQ9 cannot be tested as planned."
    )

if (contingency_rq9.sum(axis=0) == 0).any():
    raise ValueError(
        "One late-delivery class is empty."
    )

print("\nCONTINGENCY TABLE")
print("-" * 70)

display(contingency_rq9)

# ------------------------------------------------------------
# 5. Group rates and Wilson confidence intervals
# ------------------------------------------------------------

rq9_summary = contingency_rq9.copy()

rq9_summary["total_orders"] = (
    rq9_summary["Not_Late"]
    + rq9_summary["Late"]
)

rq9_summary["late_rate"] = (
    rq9_summary["Late"]
    / rq9_summary["total_orders"]
)

ci_lower, ci_upper = proportion_confint(
    count=rq9_summary["Late"].to_numpy(),
    nobs=rq9_summary["total_orders"].to_numpy(),
    alpha=ALPHA,
    method="wilson"
)

rq9_summary["ci_lower"] = ci_lower
rq9_summary["ci_upper"] = ci_upper

rq9_summary["late_rate_pct"] = (
    rq9_summary["late_rate"] * 100
)

print("\nGROUP SUMMARY WITH WILSON 95% CI")
print("-" * 70)

display(rq9_summary)

# ------------------------------------------------------------
# 6. Risk difference with Newcombe CI
# ------------------------------------------------------------

# Group 1: Multiple sellers
# Group 0: Single seller

x1 = int(
    rq9_summary.loc["Multiple sellers", "Late"]
)

n1 = int(
    rq9_summary.loc[
        "Multiple sellers", "total_orders"
    ]
)

x0 = int(
    rq9_summary.loc["Single seller", "Late"]
)

n0 = int(
    rq9_summary.loc[
        "Single seller", "total_orders"
    ]
)

p1 = x1 / n1
p0 = x0 / n0

risk_difference = p1 - p0

rd_ci_lower, rd_ci_upper = (
    confint_proportions_2indep(
        count1=x1,
        nobs1=n1,
        count2=x0,
        nobs2=n0,
        compare="diff",
        method="newcomb",
        alpha=ALPHA
    )
)

print("\nRISK DIFFERENCE")
print("-" * 70)

print(f"Multiple sellers late rate: {p1:.4%}")
print(f"Single seller late rate: {p0:.4%}")

print(
    f"Difference: {risk_difference * 100:.4f} "
    "percentage points"
)

print(
    f"Newcombe 95% CI: "
    f"[{rd_ci_lower * 100:.4f}, "
    f"{rd_ci_upper * 100:.4f}] percentage points"
)

# ------------------------------------------------------------
# 7. Pearson Chi-square and diagnostics
# ------------------------------------------------------------

observed = contingency_rq9.to_numpy()

chi2_stat, chi2_p, dof, expected = (
    stats.chi2_contingency(
        observed,
        correction=False
    )
)

expected_min = float(expected.min())

expected_lt5 = int(
    (expected < 5).sum()
)

expected_lt1 = int(
    (expected < 1).sum()
)

expected_lt5_pct = (
    expected_lt5 / expected.size * 100
)

# For a 2x2 table use a conservative screening rule:
# all expected counts should be at least 5.

asymptotic_check = expected_min >= 5

print("\nCHI-SQUARE DIAGNOSTICS")
print("-" * 70)

print(f"Chi-square statistic: {chi2_stat:.6f}")
print("Degrees of freedom:", dof)
print(f"Pearson p-value: {chi2_p:.10g}")
print(f"Minimum expected count: {expected_min:.4f}")
print("Cells with expected count < 5:", expected_lt5)
print("Cells with expected count < 1:", expected_lt1)
print(f"Percentage of cells < 5: {expected_lt5_pct:.2f}%")

print(
    "Expected-count screening:",
    "PASS" if asymptotic_check else "WARNING"
)

# ------------------------------------------------------------
# 8. Select primary test
# ------------------------------------------------------------

if asymptotic_check:

    selected_method = "Pearson Chi-square"

    selected_statistic = float(
        chi2_stat
    )

    raw_p_value = float(
        chi2_p
    )

    statistic_name = "Chi-square"

else:

    fisher_odds_ratio, fisher_p = (
        stats.fisher_exact(
            observed,
            alternative="two-sided"
        )
    )

    selected_method = "Fisher exact"

    selected_statistic = float(
        fisher_odds_ratio
    )

    raw_p_value = float(
        fisher_p
    )

    statistic_name = "Fisher odds ratio"

print("\nSELECTED TEST")
print("-" * 70)

print("Method:", selected_method)
print(
    f"{statistic_name}: {selected_statistic:.6f}"
)
print(f"Raw p-value: {raw_p_value:.10g}")

# ------------------------------------------------------------
# 9. Phi coefficient (signed)
# ------------------------------------------------------------

# Positive phi indicates that the Multiple sellers
# group has a higher late-delivery rate.

phi_magnitude = np.sqrt(
    chi2_stat / observed.sum()
)

phi_signed = (
    np.sign(risk_difference)
    * phi_magnitude
)

print("\nEFFECT SIZE")
print("-" * 70)

print(
    f"Phi coefficient (signed): {phi_signed:.6f}"
)

print(
    f"Risk difference: "
    f"{risk_difference * 100:.4f} percentage points"
)

# ------------------------------------------------------------
# 10. Store results
# ------------------------------------------------------------

rq9_result = {
    "rq": "RQ9",
    "variable": "seller_count_group",
    "method": selected_method,
    "initial_n": int(initial_n),
    "missing_n": int(missing_n),
    "n": int(retained_n),
    "n_groups": 2,
    "n_single": int(n0),
    "n_multiple": int(n1),
    "test_statistic": selected_statistic,
    "test_statistic_name": statistic_name,
    "degrees_of_freedom": (
        int(dof) if asymptotic_check else np.nan
    ),
    "p_value_raw": raw_p_value,
    "p_value_holm": np.nan,
    "effect_size_name": "Signed Phi coefficient",
    "effect_size": float(phi_signed),
    "estimate_name": "Risk difference",
    "estimate": float(risk_difference),
    "ci_lower": float(rd_ci_lower),
    "ci_upper": float(rd_ci_upper),
    "ci_method": "Newcombe",
    "expected_min": expected_min,
    "expected_lt5": expected_lt5,
    "assumption_screening_pass": bool(asymptotic_check)
}

# ------------------------------------------------------------
# 11. Preliminary interpretation
# ------------------------------------------------------------

print("\nPRELIMINARY RESULT")
print("-" * 70)

if raw_p_value < ALPHA:
    print(
        "Unadjusted evidence that late delivery "
        "rates differ by seller-count group."
    )
else:
    print(
        "Insufficient unadjusted evidence "
        "to reject equal late delivery rates."
    )

if risk_difference > 0:
    print(
        "Multiple-seller orders have a higher "
        "observed late delivery rate."
    )
elif risk_difference < 0:
    print(
        "Single-seller orders have a higher "
        "observed late delivery rate."
    )
else:
    print(
        "Observed late delivery rates are equal."
    )

print(
    "Final inference requires Holm adjustment."
)

print(
    "Results are observational and assume "
    "independent orders for the reported test and CI."
)

print("\nRQ9 ANALYSIS COMPLETED.")


RQ9 - SELLER COUNT VS LATE DELIVERY

ORIGINAL SELLER COUNT DISTRIBUTION
----------------------------------------------------------------------


,orders
seller_count,
1,95195
2,1216
3,54
4,3
5,2



SAMPLE SUMMARY
----------------------------------------------------------------------
Initial labeled orders: 96470
Missing seller_count: 0
Retained orders: 96470

CONTINGENCY TABLE
----------------------------------------------------------------------


,Not_Late,Late
seller_count_group,,
Single seller,88674,6521
Multiple sellers,1262,13



GROUP SUMMARY WITH WILSON 95% CI
----------------------------------------------------------------------


,Not_Late,Late,total_orders,late_rate,ci_lower,ci_upper,late_rate_pct
seller_count_group,,,,,,,
Single seller,88674,6521,95195,0.0685,0.0669,0.0701,6.8501
Multiple sellers,1262,13,1275,0.0102,0.0060,0.0174,1.0196



RISK DIFFERENCE
----------------------------------------------------------------------
Multiple sellers late rate: 1.0196%
Single seller late rate: 6.8501%
Difference: -5.8305 percentage points
Newcombe 95% CI: [-6.2834, -5.0961] percentage points

CHI-SQUARE DIAGNOSTICS
----------------------------------------------------------------------
Chi-square statistic: 67.736342
Degrees of freedom: 1
Pearson p-value: 1.86886942e-16
Minimum expected count: 86.3569
Cells with expected count < 5: 0
Cells with expected count < 1: 0
Percentage of cells < 5: 0.00%
Expected-count screening: PASS

SELECTED TEST
----------------------------------------------------------------------
Method: Pearson Chi-square
Chi-square: 67.736342
Raw p-value: 1.86886942e-16

EFFECT SIZE
----------------------------------------------------------------------
Phi coefficient (signed): -0.026498
Risk difference: -5.8305 percentage points

PRELIMINARY RESULT
----------------------------------------------------------------

In [19]:

# ============================================================
# Cell 18: Shared 2x2 analysis - RQ10 and RQ12
# ============================================================

from statsmodels.stats.proportion import (
    confint_proportions_2indep
)

def analyze_binary_rq(rq_id, variable, group_function,
                      labels):

    data = df_labeled[
        ["order_id", "is_late", variable]
    ].copy()

    initial_n = len(data)
    missing_n = int(data[variable].isna().sum())
    data = data.dropna(subset=[variable]).copy()

    numeric = pd.to_numeric(data[variable], errors="raise")

    if not np.isfinite(numeric.to_numpy(dtype=float)).all():
        raise ValueError(f"{variable}: non-finite values")

    data[variable] = numeric

    # Validate source values before grouping.
    if rq_id == "RQ10":
        valid = (
            (data[variable] >= 1)
            & (data[variable] == np.floor(data[variable]))
        )
    else:
        valid = data[variable].isin([0, 1])

    if not valid.all():
        raise ValueError(
            f"{rq_id}: invalid source values, "
            f"count = {(~valid).sum()}"
        )

    data["group"] = group_function(data[variable])

    table = pd.crosstab(
        data["group"],
        data["is_late"]
    ).reindex(
        index=[0, 1],
        columns=[0, 1],
        fill_value=0
    )

    if (table.sum(axis=1) == 0).any():
        raise ValueError(f"{rq_id}: empty group")

    if (table.sum(axis=0) == 0).any():
        raise ValueError(f"{rq_id}: empty target class")

    observed = table.to_numpy()
    chi2, chi_p, dof, expected = (
        stats.chi2_contingency(
            observed, correction=False
        )
    )

    if expected.min() >= 5:
        method = "Pearson Chi-square"
        statistic = float(chi2)
        p_raw = float(chi_p)
        statistic_name = "Chi-square"
        df_value = int(dof)
    else:
        odds_ratio, fisher_p = stats.fisher_exact(
            observed, alternative="two-sided"
        )
        method = "Fisher exact"
        statistic = float(odds_ratio)
        p_raw = float(fisher_p)
        statistic_name = "Odds ratio"
        df_value = np.nan

    x0, n0 = int(table.loc[0, 1]), int(table.loc[0].sum())
    x1, n1 = int(table.loc[1, 1]), int(table.loc[1].sum())

    p0, p1 = x0 / n0, x1 / n1
    rd = p1 - p0

    rd_lo, rd_hi = confint_proportions_2indep(
        count1=x1, nobs1=n1,
        count2=x0, nobs2=n0,
        method="newcomb",
        compare="diff",
        alpha=ALPHA
    )

    phi = np.sign(rd) * np.sqrt(chi2 / len(data))

    print(f"\n{'=' * 65}")
    print(f"{rq_id} - {variable}")
    print(f"{'=' * 65}")
    print("Initial:", initial_n)
    print("Missing:", missing_n)
    print("Retained:", len(data))
    print("Expected minimum:", round(float(expected.min()), 4))

    summary = table.copy()
    summary.index = labels
    summary.columns = ["Not_Late", "Late"]
    summary["Total"] = summary.sum(axis=1)
    summary["Late_rate_pct"] = (
        100 * summary["Late"] / summary["Total"]
    )

    display(summary)

    print("Method:", method)
    print(f"{statistic_name}: {statistic:.6f}")
    print(f"Raw p-value: {p_raw:.10g}")
    print(f"Signed Phi: {phi:.6f}")
    print(f"Risk difference: {rd * 100:.4f} pp")
    print(
        f"Newcombe 95% CI: "
        f"[{rd_lo * 100:.4f}, {rd_hi * 100:.4f}] pp"
    )

    result = {
        "rq": rq_id,
        "variable": variable,
        "method": method,
        "initial_n": initial_n,
        "missing_n": missing_n,
        "n": len(data),
        "n_groups": 2,
        "test_statistic": statistic,
        "test_statistic_name": statistic_name,
        "degrees_of_freedom": df_value,
        "p_value_raw": p_raw,
        "p_value_holm": np.nan,
        "effect_size_name": "Signed Phi coefficient",
        "effect_size": float(phi),
        "estimate_name": "Risk difference",
        "estimate": float(rd),
        "ci_lower": float(rd_lo),
        "ci_upper": float(rd_hi),
        "ci_method": "Newcombe",
        "expected_min": float(expected.min()),
        "n_group_0": n0,
        "n_group_1": n1
    }

    return result


# RQ10: 1 category vs >= 2 categories
rq10_result = analyze_binary_rq(
    rq_id="RQ10",
    variable="product_category_count",
    group_function=lambda s: (s >= 2).astype(int),
    labels=["1 category", "2+ categories"]
)

# RQ12: Not national holiday vs national holiday
rq12_result = analyze_binary_rq(
    rq_id="RQ12",
    variable="purchase_is_national_holiday",
    group_function=lambda s: s.astype(int),
    labels=["Non-holiday", "National holiday"]
)

binary_batch_results_df = pd.DataFrame([
    rq10_result,
    rq12_result
])

print("\nCOMBINED RQ10 + RQ12 RESULTS")
display(binary_batch_results_df[
    [
        "rq", "variable", "method", "n", "missing_n",
        "p_value_raw", "effect_size",
        "estimate", "ci_lower", "ci_upper"
    ]
])

print("\nRQ10 AND RQ12 COMPLETED.")



RQ10 - product_category_count
Initial: 96470
Missing: 1392
Retained: 95078
Expected minimum: 48.7154


,Not_Late,Late,Total,Late_rate_pct
1 category,87936,6422,94358,6.8060
2+ categories,709,11,720,1.5278


Method: Pearson Chi-square
Chi-square: 31.557156
Raw p-value: 1.936554526e-08
Signed Phi: -0.018218
Risk difference: -5.2782 pp
Newcombe 95% CI: [-5.9701, -4.0805] pp



RQ12 - purchase_is_national_holiday
Initial: 96470
Missing: 0
Retained: 96470
Expected minimum: 100.0385


,Not_Late,Late,Total,Late_rate_pct
Non-holiday,88531,6462,94993,6.8026
National holiday,1405,72,1477,4.8747


Method: Pearson Chi-square
Chi-square: 8.560573
Raw p-value: 0.003435212469
Signed Phi: -0.009420
Risk difference: -1.9279 pp
Newcombe 95% CI: [-2.9270, -0.6976] pp

COMBINED RQ10 + RQ12 RESULTS


,rq,variable,method,n,missing_n,p_value_raw,effect_size,estimate,ci_lower,ci_upper
0,RQ10,product_category_count,Pearson Chi-square,95078,1392,0.0000,-0.0182,-0.0528,-0.0597,-0.0408
1,RQ12,purchase_is_national_holiday,Pearson Chi-square,96470,0,0.0034,-0.0094,-0.0193,-0.0293,-0.0070



RQ10 AND RQ12 COMPLETED.


In [20]:

# ============================================================
# Cell 19: RQ11 - Estimated Delivery Window
# ============================================================

VARIABLE = "estimated_window_days"

print("RQ11 - ESTIMATED DELIVERY WINDOW")
print("=" * 65)

# 1. Prepare sample
initial_n = len(df_labeled)

sample = df_labeled[
    ["order_id", "is_late", VARIABLE]
].copy()

missing_n = int(sample[VARIABLE].isna().sum())
sample = sample.dropna(subset=[VARIABLE]).copy()

sample[VARIABLE] = pd.to_numeric(
    sample[VARIABLE], errors="raise"
).astype(float)

values = sample[VARIABLE].to_numpy()

if not np.isfinite(values).all():
    raise ValueError("Non-finite delivery windows found.")

if (values < 0).any():
    raise ValueError(
        "Negative delivery windows found. "
        "Review source data before testing."
    )

assert sample["order_id"].is_unique
assert initial_n == missing_n + len(sample)

# 2. Split groups
late = sample.loc[
    sample["is_late"] == 1, VARIABLE
].to_numpy()

not_late = sample.loc[
    sample["is_late"] == 0, VARIABLE
].to_numpy()

n1, n0 = len(late), len(not_late)

if min(n1, n0) < 2:
    raise ValueError("Insufficient observations.")

# 3. Descriptive statistics
rq11_descriptive_df = pd.DataFrame([
    {
        "group": label,
        "n": len(x),
        "mean": np.mean(x),
        "sd": np.std(x, ddof=1),
        "median": np.median(x),
        "q1": np.percentile(x, 25),
        "q3": np.percentile(x, 75),
        "min": np.min(x),
        "max": np.max(x)
    }
    for label, x in [
        ("Late", late),
        ("Not Late", not_late)
    ]
])

print("\nSAMPLE SUMMARY")
print("Initial:", initial_n)
print("Missing:", missing_n)
print("Retained:", len(sample))
print("Late:", n1)
print("Not Late:", n0)

print("\nDESCRIPTIVE STATISTICS (DAYS)")
display(rq11_descriptive_df)

# 4. Mann-Whitney U
mw = stats.mannwhitneyu(
    late,
    not_late,
    alternative="two-sided",
    method="asymptotic",
    use_continuity=True
)

u = float(mw.statistic)
p_raw = float(mw.pvalue)

prob_superiority = u / (n1 * n0)
rank_biserial = 2 * prob_superiority - 1

# 5. Diagnostic information
print("\nDISTRIBUTION DIAGNOSTICS")
print("Unique values:", np.unique(values).size)
print("Zero values:", int((values == 0).sum()))
print(
    "Repeated values beyond unique:",
    len(values) - np.unique(values).size
)

# 6. Bootstrap rank-biserial CI
# Exploratory order-level resampling, not cluster robust.

BOOT_REPS = 200
rng = np.random.default_rng(RANDOM_SEED + 11)

boot_effects = np.empty(BOOT_REPS)

for i in range(BOOT_REPS):
    a = rng.choice(late, size=n1, replace=True)
    b = rng.choice(not_late, size=n0, replace=True)

    ranks = stats.rankdata(
        np.concatenate([a, b]),
        method="average"
    )

    boot_u = (
        ranks[:n1].sum()
        - n1 * (n1 + 1) / 2
    )

    boot_effects[i] = (
        2 * boot_u / (n1 * n0) - 1
    )

ci_lo, ci_hi = np.percentile(
    boot_effects, [2.5, 97.5]
)

# 7. Save for final Holm adjustment
rq11_result = {
    "rq": "RQ11",
    "variable": VARIABLE,
    "method": "Mann-Whitney U (two-sided)",
    "initial_n": initial_n,
    "missing_n": missing_n,
    "n": len(sample),
    "n_late": n1,
    "n_not_late": n0,
    "test_statistic": u,
    "degrees_of_freedom": np.nan,
    "p_value_raw": p_raw,
    "p_value_holm": np.nan,
    "effect_size_name": "Rank-biserial correlation",
    "effect_size": float(rank_biserial),
    "estimate_name": "Probability superiority",
    "estimate": float(prob_superiority),
    "ci_lower": float(ci_lo),
    "ci_upper": float(ci_hi),
    "ci_method": "Order-level bootstrap percentile",
    "bootstrap_reps": BOOT_REPS,
    "unit": "days",
    "cluster_robust_ci": False
}

# 8. Compact output
print("\nRQ11 TEST RESULTS")
print("-" * 65)
print(f"U statistic: {u:.4f}")
print(f"Raw p-value: {p_raw:.10g}")
print(f"Probability superiority: {prob_superiority:.6f}")
print(f"Rank-biserial: {rank_biserial:.6f}")
print(f"Bootstrap 95% CI: [{ci_lo:.6f}, {ci_hi:.6f}]")
print("Bootstrap replications:", BOOT_REPS)

print(
    "\nNOTE: Test and CI assume independent orders."
)
print(
    "Mann-Whitney U is not automatically "
    "a median-difference test."
)
print("Final inference requires Holm adjustment.")

print("\nRQ11 ANALYSIS COMPLETED.")


RQ11 - ESTIMATED DELIVERY WINDOW



SAMPLE SUMMARY
Initial: 96470
Missing: 0
Retained: 96470
Late: 6534
Not Late: 89936

DESCRIPTIVE STATISTICS (DAYS)


,group,n,mean,sd,median,q1,q3,min,max
0,Late,6534,23.2903,7.5650,23.0000,20.0000,28.0000,3.0000,70.0000
1,Not Late,89936,24.4514,8.8337,24.0000,19.0000,29.0000,3.0000,156.0000



DISTRIBUTION DIAGNOSTICS


Unique values: 100
Zero values: 0
Repeated values beyond unique: 96370



RQ11 TEST RESULTS
-----------------------------------------------------------------
U statistic: 278633678.5000
Raw p-value: 2.693616516e-12
Probability superiority: 0.474156
Rank-biserial: -0.051689
Bootstrap 95% CI: [-0.064536, -0.039512]
Bootstrap replications: 200

NOTE: Test and CI assume independent orders.
Mann-Whitney U is not automatically a median-difference test.
Final inference requires Holm adjustment.

RQ11 ANALYSIS COMPLETED.


In [21]:

# ============================================================
# Cell 20: RQ13 + RQ14 - Holiday-Related Factors
# ============================================================

HOLIDAY_GROUPS = {
    "RQ13": {
        "variable": "national_holiday_count_in_estimated_window",
        "labels": ["0 holidays", "1 holiday", "2+ holidays"]
    },
    "RQ14": {
        "variable": "days_to_next_national_holiday",
        "labels": ["0 days", "1-7 days", "8-14 days", "15+ days"]
    }
}

holiday_results = []
holiday_group_summaries = []

def analyze_holiday_rq(rq_id, config):
    variable = config["variable"]
    labels = config["labels"]

    data = df_labeled[
        ["order_id", "is_late", variable]
    ].copy()

    initial_n = len(data)
    missing_n = int(data[variable].isna().sum())

    data = data.dropna(subset=[variable]).copy()
    values = pd.to_numeric(data[variable], errors="raise")
    values_np = values.to_numpy(dtype=float)

    if not np.isfinite(values_np).all():
        raise ValueError(f"{rq_id}: non-finite values")

    if ((values_np < 0) | (values_np != np.floor(values_np))).any():
        raise ValueError(
            f"{rq_id}: negative or non-integer values"
        )

    if rq_id == "RQ13":
        data["group"] = np.select(
            [values_np == 0, values_np == 1, values_np >= 2],
            labels,
            default="INVALID"
        )
    else:
        data["group"] = np.select(
            [
                values_np == 0,
                (values_np >= 1) & (values_np <= 7),
                (values_np >= 8) & (values_np <= 14),
                values_np >= 15
            ],
            labels,
            default="INVALID"
        )

    if (data["group"] == "INVALID").any():
        raise ValueError(f"{rq_id}: grouping failed")

    table = pd.crosstab(
        data["group"], data["is_late"]
    ).reindex(
        index=labels,
        columns=[0, 1],
        fill_value=0
    )

    # Empty groups cannot contribute to Chi-square.
    empty_groups = table.index[
        table.sum(axis=1) == 0
    ].tolist()

    table = table.loc[table.sum(axis=1) > 0]

    if len(table) < 2 or (table.sum(axis=0) == 0).any():
        raise ValueError(f"{rq_id}: insufficient groups/classes")

    observed = table.to_numpy()

    chi2, p_asym, dof, expected = stats.chi2_contingency(
        observed, correction=False
    )

    expected_min = float(expected.min())
    expected_lt5 = int((expected < 5).sum())
    expected_lt1 = int((expected < 1).sum())
    expected_lt5_pct = 100 * expected_lt5 / expected.size

    screening_pass = (
        expected_min >= 1 and expected_lt5_pct <= 20
    )

    if screening_pass:
        method = "Pearson Chi-square"
        p_raw = float(p_asym)
        n_resamples = 0
        extreme = np.nan
    else:
        method = "Conditional Monte Carlo Chi-square"
        n_resamples = 20000

        rng = np.random.default_rng(
            RANDOM_SEED + (13 if rq_id == "RQ13" else 14)
        )

        simulations = stats.random_table.rvs(
            observed.sum(axis=1),
            observed.sum(axis=0),
            size=n_resamples,
            random_state=rng
        )

        sim_chi2 = (
            ((simulations - expected) ** 2 / expected)
            .sum(axis=(1, 2))
        )

        extreme = int(
            np.count_nonzero(sim_chi2 >= chi2 - 1e-12)
        )

        p_raw = (extreme + 1) / (n_resamples + 1)

    n = int(observed.sum())

    cramers_v = float(np.sqrt(
        chi2 / (n * min(observed.shape[0] - 1,
                        observed.shape[1] - 1))
    ))

    summary = table.copy()
    summary.columns = ["Not_Late", "Late"]
    summary["total_orders"] = summary.sum(axis=1)
    summary["late_rate"] = (
        summary["Late"] / summary["total_orders"]
    )

    lo, hi = proportion_confint(
        summary["Late"].to_numpy(),
        summary["total_orders"].to_numpy(),
        alpha=ALPHA,
        method="wilson"
    )

    summary["ci_lower"] = lo
    summary["ci_upper"] = hi
    summary["late_rate_pct"] = 100 * summary["late_rate"]
    summary["rq"] = rq_id
    summary["variable"] = variable

    holiday_group_summaries.append(
        summary.reset_index().rename(columns={"group": "category"})
    )

    result = {
        "rq": rq_id,
        "variable": variable,
        "method": method,
        "initial_n": initial_n,
        "missing_n": missing_n,
        "n": n,
        "n_groups": len(table),
        "test_statistic": float(chi2),
        "degrees_of_freedom": int(dof),
        "p_value_raw": float(p_raw),
        "p_value_holm": np.nan,
        "effect_size_name": "Cramer's V",
        "effect_size": cramers_v,
        "expected_min": expected_min,
        "expected_lt5": expected_lt5,
        "expected_lt1": expected_lt1,
        "assumption_screening_pass": bool(screening_pass),
        "monte_carlo_resamples": n_resamples,
        "monte_carlo_extreme": extreme,
        "empty_groups": ", ".join(empty_groups)
    }

    print(f"\n{'=' * 65}")
    print(f"{rq_id}: {variable}")
    print(f"{'=' * 65}")
    print("Initial:", initial_n)
    print("Missing:", missing_n)
    print("Retained:", n)
    print("Empty groups:", empty_groups)
    display(summary[
        ["Not_Late", "Late", "total_orders",
         "late_rate_pct", "ci_lower", "ci_upper"]
    ])

    print("Method:", method)
    print(f"Chi-square: {chi2:.6f} (df={dof})")
    print(f"Expected minimum: {expected_min:.4f}")
    print(f"Cells expected < 5: {expected_lt5}")
    print(f"Cells expected < 1: {expected_lt1}")
    print(f"Raw p-value: {p_raw:.10g}")
    print(f"Cramer's V: {cramers_v:.6f}")

    if n_resamples:
        print("Monte Carlo resamples:", n_resamples)
        print("Extreme simulations:", extreme)
        print(
            "Note: Monte Carlo p-value has finite resolution."
        )

    return result


rq13_result = analyze_holiday_rq(
    "RQ13", HOLIDAY_GROUPS["RQ13"]
)

rq14_result = analyze_holiday_rq(
    "RQ14", HOLIDAY_GROUPS["RQ14"]
)

holiday_results_df = pd.DataFrame([
    rq13_result, rq14_result
])

holiday_summary_df = pd.concat(
    holiday_group_summaries, ignore_index=True
)

print("\nCOMBINED RQ13 + RQ14 RESULTS")
display(holiday_results_df[
    [
        "rq", "variable", "method", "n",
        "missing_n", "n_groups", "p_value_raw",
        "effect_size", "expected_min"
    ]
])

print("\nRQ13 AND RQ14 COMPLETED.")



RQ13: national_holiday_count_in_estimated_window
Initial: 96470
Missing: 0
Retained: 96470
Empty groups: []


,Not_Late,Late,total_orders,late_rate_pct,ci_lower,ci_upper
group,,,,,,
0 holidays,59467,4266,63733,6.6935,0.0650,0.0689
1 holiday,19809,1551,21360,7.2612,0.0692,0.0762
2+ holidays,10660,717,11377,6.3022,0.0587,0.0676


Method: Pearson Chi-square
Chi-square: 12.694662 (df=2)
Expected minimum: 770.5745
Cells expected < 5: 0
Cells expected < 1: 0
Raw p-value: 0.001751415125
Cramer's V: 0.011471

RQ14: days_to_next_national_holiday
Initial: 96470
Missing: 0
Retained: 96470
Empty groups: []


,Not_Late,Late,total_orders,late_rate_pct,ci_lower,ci_upper
group,,,,,,
0 days,1405,72,1477,4.8747,0.0389,0.0609
1-7 days,11574,665,12239,5.4335,0.0505,0.0585
8-14 days,9256,562,9818,5.7242,0.0528,0.0620
15+ days,67701,5235,72936,7.1775,0.0699,0.0737


Method: Pearson Chi-square
Chi-square: 79.215038 (df=3)
Expected minimum: 100.0385
Cells expected < 5: 0
Cells expected < 1: 0
Raw p-value: 4.522713945e-17
Cramer's V: 0.028655

COMBINED RQ13 + RQ14 RESULTS


,rq,variable,method,n,missing_n,n_groups,p_value_raw,effect_size,expected_min
0,RQ13,national_holiday_count_in_estimated_window,Pearson Chi-square,96470,0,3,0.0018,0.0115,770.5745
1,RQ14,days_to_next_national_holiday,Pearson Chi-square,96470,0,4,0.0000,0.0287,100.0385



RQ13 AND RQ14 COMPLETED.


In [22]:

# ============================================================
# Cell 21: RQ15-RQ17 - Weather Block Bootstrap
# ============================================================

WEATHER_TESTS = {
    "RQ15a": ("origin_rain_state_mean_mm", "mm"),
    "RQ15b": ("destination_rain_state_mean_mm", "mm"),
    "RQ16a": ("origin_humidity_state_mean_pct", "%"),
    "RQ16b": ("destination_humidity_state_mean_pct", "%"),
    "RQ17a": ("origin_air_temp_state_mean_c", "C"),
    "RQ17b": ("destination_air_temp_state_mean_c", "C"),
}

WEATHER_BOOT_REPS = 2000
WEATHER_PRIMARY_BLOCK_DAYS = 7
WEATHER_SENSITIVITY_BLOCK_DAYS = 14

weather_results = []
weather_descriptive_records = []
weather_sensitivity_records = []
weather_sample_records = []

# Fixed Monday anchor for non-overlapping calendar blocks.
BLOCK_ORIGIN = pd.Timestamp("2016-01-04")

def weather_block_bootstrap(data, variable, block_days, seed):
    """
    Resample entire chronological blocks with replacement.

    The estimated difference is mean(Late) - mean(Not Late).
    CI: percentile bootstrap.
    P-value: two-sided, null-centered bootstrap approximation.

    This accounts for some temporal dependence,
    but not all seller/geographic dependence.
    """

    frame = data[["is_late", "purchase_time", variable]].copy()

    block_ids = (
        (
            frame["purchase_time"].dt.normalize()
            - BLOCK_ORIGIN
        ).dt.days // block_days
    )

    block_codes, unique_blocks = pd.factorize(
        block_ids, sort=True
    )

    n_blocks = len(unique_blocks)

    if n_blocks < 3:
        raise ValueError("Too few temporal blocks.")

    labels = frame["is_late"].to_numpy(dtype=int)
    values = frame[variable].to_numpy(dtype=float)

    # Each block has two class-specific sums and counts.
    block_sums = np.zeros((n_blocks, 2), dtype=float)
    block_counts = np.zeros((n_blocks, 2), dtype=int)

    np.add.at(block_sums, (block_codes, labels), values)
    np.add.at(block_counts, (block_codes, labels), 1)

    total_sums = block_sums.sum(axis=0)
    total_counts = block_counts.sum(axis=0)

    if (total_counts == 0).any():
        raise ValueError("Missing target class.")

    observed_difference = (
        total_sums[1] / total_counts[1]
        - total_sums[0] / total_counts[0]
    )

    rng = np.random.default_rng(seed)

    bootstrap_differences = []

    # Draw entire blocks; preserve orders within each block.
    for _ in range(WEATHER_BOOT_REPS):
        sampled_indices = rng.integers(
            0, n_blocks, size=n_blocks
        )

        sampled_sums = block_sums[sampled_indices].sum(axis=0)
        sampled_counts = block_counts[sampled_indices].sum(axis=0)

        if (sampled_counts == 0).any():
            continue

        difference = (
            sampled_sums[1] / sampled_counts[1]
            - sampled_sums[0] / sampled_counts[0]
        )

        bootstrap_differences.append(difference)

    bootstrap_differences = np.asarray(
        bootstrap_differences, dtype=float
    )

    valid_reps = len(bootstrap_differences)

    if valid_reps < WEATHER_BOOT_REPS * 0.95:
        raise ValueError(
            "Too many invalid bootstrap samples."
        )

    ci_lower, ci_upper = np.percentile(
        bootstrap_differences, [2.5, 97.5]
    )

    # Center the bootstrap error around the observed estimate
    # to approximate the sampling distribution under H0: diff=0.
    centered_errors = (
        bootstrap_differences - observed_difference
    )

    extreme = int(np.count_nonzero(
        np.abs(centered_errors)
        >= abs(observed_difference)
    ))

    approximate_p = (
        extreme + 1
    ) / (
        valid_reps + 1
    )

    both_classes_blocks = int(
        np.all(block_counts > 0, axis=1).sum()
    )

    return {
        "estimate": float(observed_difference),
        "ci_lower": float(ci_lower),
        "ci_upper": float(ci_upper),
        "p_value": float(approximate_p),
        "n_blocks": n_blocks,
        "blocks_with_both_classes": both_classes_blocks,
        "valid_bootstrap_reps": valid_reps,
        "extreme_resamples": extreme,
        "block_days": block_days
    }


print("RQ15-RQ17 - WEATHER BLOCK BOOTSTRAP")
print("=" * 70)

for index, (rq, (variable, unit)) in enumerate(
    WEATHER_TESTS.items()
):

    print(f"\n{rq}: {variable}")
    print("-" * 70)

    initial_n = len(df_labeled)

    sample = df_labeled[
        [
            "order_id",
            "is_late",
            "order_purchase_timestamp",
            variable
        ]
    ].copy()

    missing_weather_n = int(sample[variable].isna().sum())

    # Dates interpreted using the project's UTC assumption.
    sample["purchase_time"] = pd.to_datetime(
        sample["order_purchase_timestamp"],
        errors="coerce",
        utc=True
    ).dt.tz_localize(None)

    missing_timestamp_n = int(
        sample["purchase_time"].isna().sum()
    )

    sample = sample.dropna(
        subset=[variable, "purchase_time"]
    ).copy()

    sample[variable] = pd.to_numeric(
        sample[variable], errors="raise"
    ).astype(float)

    if not np.isfinite(sample[variable].to_numpy()).all():
        raise ValueError(
            f"{variable}: non-finite weather values."
        )

    assert sample["order_id"].is_unique

    n_late = int((sample["is_late"] == 1).sum())
    n_not_late = int((sample["is_late"] == 0).sum())

    if min(n_late, n_not_late) < 2:
        raise ValueError(f"{rq}: insufficient observations.")

    for group_name, group_label in [
        ("Late", 1), ("Not Late", 0)
    ]:
        v = sample.loc[
            sample["is_late"] == group_label,
            variable
        ].to_numpy()

        weather_descriptive_records.append({
            "rq": rq,
            "variable": variable,
            "group": group_name,
            "n": len(v),
            "mean": float(np.mean(v)),
            "sd": float(np.std(v, ddof=1)),
            "median": float(np.median(v)),
            "q1": float(np.percentile(v, 25)),
            "q3": float(np.percentile(v, 75)),
            "unit": unit
        })

    primary = weather_block_bootstrap(
        sample,
        variable,
        WEATHER_PRIMARY_BLOCK_DAYS,
        RANDOM_SEED + 100 + index
    )

    sensitivity = weather_block_bootstrap(
        sample,
        variable,
        WEATHER_SENSITIVITY_BLOCK_DAYS,
        RANDOM_SEED + 200 + index
    )

    weather_sample_records.append({
        "rq": rq,
        "variable": variable,
        "initial_n": initial_n,
        "missing_weather_n": missing_weather_n,
        "missing_timestamp_n": missing_timestamp_n,
        "retained_n": len(sample),
        "n_late": n_late,
        "n_not_late": n_not_late
    })

    weather_results.append({
        "rq": rq,
        "variable": variable,
        "method": "Temporal block bootstrap (7 days)",
        "initial_n": initial_n,
        "missing_n": initial_n - len(sample),
        "n": len(sample),
        "n_late": n_late,
        "n_not_late": n_not_late,
        "test_statistic": np.nan,
        "degrees_of_freedom": np.nan,
        "p_value_raw": primary["p_value"],
        "p_value_holm": np.nan,
        "effect_size_name": "Mean difference",
        "effect_size": primary["estimate"],
        "estimate_name": "Mean difference (Late - Not Late)",
        "estimate": primary["estimate"],
        "ci_lower": primary["ci_lower"],
        "ci_upper": primary["ci_upper"],
        "ci_method": "7-day block bootstrap percentile",
        "unit": unit,
        "n_blocks": primary["n_blocks"],
        "blocks_with_both_classes": primary[
            "blocks_with_both_classes"
        ],
        "bootstrap_reps": primary["valid_bootstrap_reps"],
        "p_value_approximate": True
    })

    weather_sensitivity_records.append({
        "rq": rq,
        "variable": variable,
        "block_days": 14,
        "mean_difference": sensitivity["estimate"],
        "ci_lower": sensitivity["ci_lower"],
        "ci_upper": sensitivity["ci_upper"],
        "p_value_raw": sensitivity["p_value"],
        "n_blocks": sensitivity["n_blocks"],
        "bootstrap_reps": sensitivity[
            "valid_bootstrap_reps"
        ]
    })

    print("Retained orders:", len(sample))
    print("Missing weather:", missing_weather_n)
    print("Missing purchase timestamp:", missing_timestamp_n)
    print("Late:", n_late, "| Not Late:", n_not_late)

    print("\nPRIMARY: 7-DAY BLOCKS")
    print("Temporal blocks:", primary["n_blocks"])
    print(
        "Blocks containing both classes:",
        primary["blocks_with_both_classes"]
    )
    print(
        f"Mean difference: {primary['estimate']:.6f} {unit}"
    )
    print(
        f"95% CI: [{primary['ci_lower']:.6f}, "
        f"{primary['ci_upper']:.6f}] {unit}"
    )
    print(f"Approximate raw p: {primary['p_value']:.8f}")

    print("\nSENSITIVITY: 14-DAY BLOCKS")
    print("Temporal blocks:", sensitivity["n_blocks"])
    print(
        f"95% CI: [{sensitivity['ci_lower']:.6f}, "
        f"{sensitivity['ci_upper']:.6f}] {unit}"
    )
    print(
        f"Approximate raw p: {sensitivity['p_value']:.8f}"
    )

weather_results_df = pd.DataFrame(weather_results)
weather_sample_df = pd.DataFrame(weather_sample_records)
weather_descriptive_df = pd.DataFrame(
    weather_descriptive_records
)
weather_sensitivity_df = pd.DataFrame(
    weather_sensitivity_records
)

assert len(weather_results_df) == 6
assert len(weather_sensitivity_df) == 6

print("\nWEATHER - PRIMARY TEST SUMMARY")
display(weather_results_df[
    [
        "rq", "variable", "n", "missing_n",
        "estimate", "ci_lower", "ci_upper",
        "p_value_raw", "n_blocks"
    ]
])

print("\nWEATHER - 14-DAY SENSITIVITY SUMMARY")
display(weather_sensitivity_df)

print(
    "\nLIMITATIONS: Block bootstrap assumes temporal "
    "blocks are sufficiently independent. Shared "
    "state-day measurements and seller/geographic "
    "clustering may remain."
)
print(
    "The UTC scenario is an assumption, not "
    "the confirmed original Olist timezone."
)
print(
    "P-values are finite-simulation approximations. "
    "Final decisions require Holm adjustment."
)

print("\nRQ15-RQ17 WEATHER ANALYSIS COMPLETED.")


RQ15-RQ17 - WEATHER BLOCK BOOTSTRAP

RQ15a: origin_rain_state_mean_mm
----------------------------------------------------------------------


Retained orders: 96311
Missing weather: 159
Missing purchase timestamp: 0
Late: 6527 | Not Late: 89784

PRIMARY: 7-DAY BLOCKS
Temporal blocks: 91
Blocks containing both classes: 87
Mean difference: 1.110220 mm
95% CI: [0.556610, 1.605576] mm
Approximate raw p: 0.00049975

SENSITIVITY: 14-DAY BLOCKS
Temporal blocks: 48
95% CI: [0.428142, 1.608468] mm
Approximate raw p: 0.00049975

RQ15b: destination_rain_state_mean_mm
----------------------------------------------------------------------


Retained orders: 95544
Missing weather: 926
Missing purchase timestamp: 0
Late: 6465 | Not Late: 89079

PRIMARY: 7-DAY BLOCKS
Temporal blocks: 91
Blocks containing both classes: 87
Mean difference: 1.314173 mm
95% CI: [0.793928, 1.767082] mm
Approximate raw p: 0.00049975

SENSITIVITY: 14-DAY BLOCKS
Temporal blocks: 48
95% CI: [0.652222, 1.817749] mm
Approximate raw p: 0.00049975

RQ16a: origin_humidity_state_mean_pct
----------------------------------------------------------------------


Retained orders: 96317
Missing weather: 153
Missing purchase timestamp: 0
Late: 6529 | Not Late: 89788

PRIMARY: 7-DAY BLOCKS
Temporal blocks: 91
Blocks containing both classes: 87
Mean difference: 2.215929 %
95% CI: [1.171461, 3.135379] %
Approximate raw p: 0.00049975

SENSITIVITY: 14-DAY BLOCKS
Temporal blocks: 48
95% CI: [0.841889, 3.369389] %
Approximate raw p: 0.00399800

RQ16b: destination_humidity_state_mean_pct
----------------------------------------------------------------------


Retained orders: 95700
Missing weather: 770
Missing purchase timestamp: 0
Late: 6487 | Not Late: 89213

PRIMARY: 7-DAY BLOCKS
Temporal blocks: 91
Blocks containing both classes: 87
Mean difference: 2.743005 %
95% CI: [1.834306, 3.585656] %
Approximate raw p: 0.00049975

SENSITIVITY: 14-DAY BLOCKS
Temporal blocks: 48
95% CI: [1.535617, 3.799907] %
Approximate raw p: 0.00049975

RQ17a: origin_air_temp_state_mean_c
----------------------------------------------------------------------


Retained orders: 96332
Missing weather: 138
Missing purchase timestamp: 0
Late: 6530 | Not Late: 89802

PRIMARY: 7-DAY BLOCKS
Temporal blocks: 91
Blocks containing both classes: 87
Mean difference: 1.163361 C
95% CI: [0.721801, 1.572471] C
Approximate raw p: 0.00049975

SENSITIVITY: 14-DAY BLOCKS
Temporal blocks: 48
95% CI: [0.534125, 1.665475] C
Approximate raw p: 0.00049975

RQ17b: destination_air_temp_state_mean_c
----------------------------------------------------------------------


Retained orders: 95884
Missing weather: 586
Missing purchase timestamp: 0
Late: 6498 | Not Late: 89386

PRIMARY: 7-DAY BLOCKS
Temporal blocks: 91
Blocks containing both classes: 87
Mean difference: 1.210311 C
95% CI: [0.769762, 1.619792] C
Approximate raw p: 0.00049975

SENSITIVITY: 14-DAY BLOCKS
Temporal blocks: 48
95% CI: [0.632021, 1.714143] C
Approximate raw p: 0.00049975

WEATHER - PRIMARY TEST SUMMARY


,rq,variable,n,missing_n,estimate,ci_lower,ci_upper,p_value_raw,n_blocks
0,RQ15a,origin_rain_state_mean_mm,96311,159,1.1102,0.5566,1.6056,0.0005,91
1,RQ15b,destination_rain_state_mean_mm,95544,926,1.3142,0.7939,1.7671,0.0005,91
2,RQ16a,origin_humidity_state_mean_pct,96317,153,2.2159,1.1715,3.1354,0.0005,91
3,RQ16b,destination_humidity_state_mean_pct,95700,770,2.7430,1.8343,3.5857,0.0005,91
4,RQ17a,origin_air_temp_state_mean_c,96332,138,1.1634,0.7218,1.5725,0.0005,91
5,RQ17b,destination_air_temp_state_mean_c,95884,586,1.2103,0.7698,1.6198,0.0005,91



WEATHER - 14-DAY SENSITIVITY SUMMARY


,rq,variable,block_days,mean_difference,ci_lower,ci_upper,p_value_raw,n_blocks,bootstrap_reps
0,RQ15a,origin_rain_state_mean_mm,14,1.1102,0.4281,1.6085,0.0005,48,2000
1,RQ15b,destination_rain_state_mean_mm,14,1.3142,0.6522,1.8177,0.0005,48,2000
2,RQ16a,origin_humidity_state_mean_pct,14,2.2159,0.8419,3.3694,0.0040,48,2000
3,RQ16b,destination_humidity_state_mean_pct,14,2.7430,1.5356,3.7999,0.0005,48,2000
4,RQ17a,origin_air_temp_state_mean_c,14,1.1634,0.5341,1.6655,0.0005,48,2000
5,RQ17b,destination_air_temp_state_mean_c,14,1.2103,0.6320,1.7141,0.0005,48,2000



LIMITATIONS: Block bootstrap assumes temporal blocks are sufficiently independent. Shared state-day measurements and seller/geographic clustering may remain.
The UTC scenario is an assumption, not the confirmed original Olist timezone.
P-values are finite-simulation approximations. Final decisions require Holm adjustment.

RQ15-RQ17 WEATHER ANALYSIS COMPLETED.


In [23]:

# ============================================================
# Cell 22: RQ19 + RQ20 - Data Availability Audit
# ============================================================

from pathlib import Path

print("RQ19 + RQ20 - DATA AVAILABILITY AUDIT")
print("=" * 65)

# 1. Check existing columns
print("\nMAIN DATASET")
print("Total orders:", len(df))
print("Unique order IDs:", df["order_id"].nunique())

for col in ["review_score", "low_review",
            "order_status", "is_canceled"]:
    print(f"{col}: {'FOUND' if col in df.columns else 'MISSING'}")

# 2. Inspect all order statuses, including unlabeled orders
if "order_status" in df.columns:
    print("\nALL ORDER STATUSES")
    display(
        df["order_status"]
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .rename_axis("order_status")
        .to_frame("orders")
    )

    print("\nORDER STATUS VS IS_LATE")
    display(
        pd.crosstab(
            df["order_status"].fillna("<MISSING>"),
            df["is_late"].fillna(-1),
            dropna=False
        )
    )

# 3. Find available review-related files
print("\nREVIEW FILE CANDIDATES")

candidate_files = sorted(
    p for p in (PROJECT_ROOT / "data").rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".csv", ".parquet"}
    and "review" in p.name.lower()
    and ".git" not in p.parts
)

if not candidate_files:
    print("No review-related CSV/Parquet found in repository.")
else:
    for path in candidate_files:
        print("\nFile:", path.relative_to(PROJECT_ROOT))

        try:
            if path.suffix.lower() == ".csv":
                preview = pd.read_csv(path, nrows=5)
            else:
                preview = pd.read_parquet(path).head(5)

            print("Columns:", list(preview.columns))
        except Exception as exc:
            print("Cannot inspect file:", str(exc))

print("\nRQ19 + RQ20 DATA AUDIT COMPLETED.")


RQ19 + RQ20 - DATA AVAILABILITY AUDIT

MAIN DATASET
Total orders: 99441
Unique order IDs: 99441
review_score: MISSING
low_review: MISSING
order_status: FOUND
is_canceled: MISSING

ALL ORDER STATUSES


,orders
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2



ORDER STATUS VS IS_LATE


is_late,-1.0000,0.0000,1.0000
order_status,,,
approved,2,0,0
canceled,625,0,0
created,5,0,0
delivered,8,89936,6534
invoiced,314,0,0
processing,301,0,0
shipped,1107,0,0
unavailable,609,0,0



REVIEW FILE CANDIDATES

File: data\processed\reviews_clean.csv
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'review_score_valid', 'flag_invalid_score', 'has_comment_title', 'has_comment_message']

File: data\raw\olist_order_reviews_dataset.csv
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

RQ19 + RQ20 DATA AUDIT COMPLETED.


In [24]:

# ============================================================
# Cell 23: RQ19 + RQ20
# Review satisfaction and cancellation analysis
# ============================================================

from statsmodels.stats.proportion import (
    confint_proportions_2indep
)

# Proposed review threshold:
# 1-2 stars = Low review
# 3-5 stars = Not low review
#
# This definition requires team confirmation.
#
# RQ19 uses delivered orders with valid is_late.
# RQ20 uses all eligible orders, including canceled.
#
# All tests are two-sided, alpha = 0.05.

print("RQ19 + RQ20 - FINAL DATA ANALYSES")
print("=" * 70)

# ============================================================
# PART A: RQ19 - LOW REVIEW VS LATE DELIVERY
# ============================================================

print("\nRQ19 - LOW REVIEW VS LATE DELIVERY")
print("=" * 70)

REVIEW_PATH = (
    PROJECT_ROOT / "data" / "processed" / "reviews_clean.csv"
)

if not REVIEW_PATH.exists():
    raise FileNotFoundError(REVIEW_PATH)

reviews = pd.read_csv(REVIEW_PATH, low_memory=False)

required_cols = {"order_id", "review_id", "review_score"}

if not required_cols.issubset(reviews.columns):
    raise ValueError(
        f"Missing review columns: "
        f"{required_cols - set(reviews.columns)}"
    )

print("Raw cleaned-review rows:", len(reviews))

# ------------------------------------------------------------
# A1. Validate review scores
# ------------------------------------------------------------

reviews["review_score_numeric"] = pd.to_numeric(
    reviews["review_score"],
    errors="coerce"
)

# Accept only valid review scores from 1 to 5.
valid_score = reviews["review_score_numeric"].isin(
    [1, 2, 3, 4, 5]
)

# In this dataset, review_score_valid stores a
# cleaned numeric score, not a Boolean flag.
if "review_score_valid" in reviews.columns:
    cleaned_score = pd.to_numeric(
        reviews["review_score_valid"],
        errors="coerce"
    )

    # Check row-by-row agreement.
    disagreement = (
        valid_score
        & cleaned_score.notna()
        & cleaned_score.ne(reviews["review_score_numeric"])
    )

    if disagreement.any():
        raise ValueError(
            f"Found {disagreement.sum()} conflicting "
            "review scores. Check the cleaning rules."
        )

    # Require a valid cleaned score too.
    valid_score &= cleaned_score.isin([1, 2, 3, 4, 5])

reviews_valid = reviews.loc[
    valid_score & reviews["order_id"].notna()
].copy()

print("Valid review rows:", len(reviews_valid))
print("Excluded review rows:", len(reviews) - len(reviews_valid))

# ------------------------------------------------------------
# A2. Audit multiple reviews per order
# ------------------------------------------------------------

review_counts = reviews_valid.groupby("order_id").size()

duplicate_order_count = int(
    (review_counts > 1).sum()
)

conflicting_score_count = int(
    (
        reviews_valid.groupby("order_id")[
            "review_score_numeric"
        ].nunique() > 1
    ).sum()
)

print("\nREVIEW DUPLICATE AUDIT")
print("Orders with multiple valid reviews:", duplicate_order_count)
print("Orders with conflicting valid scores:", conflicting_score_count)

# Deterministic rule:
# Keep the latest valid review_creation_date.
# If tied, use answer timestamp and review_id.
# This preserves exactly one review per order.

for col in [
    "review_creation_date",
    "review_answer_timestamp"
]:
    if col in reviews_valid.columns:
        reviews_valid[col + "_parsed"] = pd.to_datetime(
            reviews_valid[col],
            errors="coerce"
        )
    else:
        reviews_valid[col + "_parsed"] = pd.NaT

reviews_valid["review_id_sort"] = (
    reviews_valid["review_id"]
    .astype("string")
    .fillna("")
)

reviews_one = (
    reviews_valid.sort_values(
        [
            "order_id",
            "review_creation_date_parsed",
            "review_answer_timestamp_parsed",
            "review_id_sort"
        ],
        kind="stable",
        na_position="first"
    )
    .drop_duplicates("order_id", keep="last")
    [
        ["order_id", "review_score_numeric"]
    ]
    .rename(
        columns={"review_score_numeric": "review_score"}
    )
)

assert reviews_one["order_id"].is_unique

# ------------------------------------------------------------
# A3. Merge only with labeled delivery orders
# ------------------------------------------------------------

rq19_all = df_labeled[
    ["order_id", "is_late"]
].merge(
    reviews_one,
    on="order_id",
    how="left",
    validate="one_to_one"
)

assert len(rq19_all) == len(df_labeled)

print("\nREVIEW COVERAGE BY DELIVERY LABEL")

coverage = rq19_all.groupby("is_late").agg(
    labeled_orders=("order_id", "size"),
    reviewed_orders=("review_score", "count")
)

coverage["missing_reviews"] = (
    coverage["labeled_orders"] - coverage["reviewed_orders"]
)

coverage["missing_review_pct"] = (
    100 * coverage["missing_reviews"]
    / coverage["labeled_orders"]
)

display(coverage)

rq19_missing_n = int(
    rq19_all["review_score"].isna().sum()
)

rq19 = rq19_all.dropna(
    subset=["review_score"]
).copy()

# Proposed low-review threshold.
rq19["low_review"] = (
    rq19["review_score"] <= 2
).astype(int)

# ------------------------------------------------------------
# A4. Review-score distribution
# ------------------------------------------------------------

print("\nREVIEW SCORE DISTRIBUTION")

display(
    pd.crosstab(
        rq19["review_score"],
        rq19["is_late"]
    ).reindex(
        columns=[0, 1],
        fill_value=0
    )
)

# ------------------------------------------------------------
# A5. Main 2x2 analysis
# ------------------------------------------------------------

# Rows: Not Late / Late
# Columns: Not low review / Low review

rq19_table = pd.crosstab(
    rq19["is_late"],
    rq19["low_review"]
).reindex(
    index=[0, 1],
    columns=[0, 1],
    fill_value=0
)

if (rq19_table.sum(axis=1) == 0).any():
    raise ValueError("RQ19: missing delivery group")

if (rq19_table.sum(axis=0) == 0).any():
    raise ValueError("RQ19: missing review category")

print("\nRQ19 CONTINGENCY TABLE")
display(rq19_table)

chi2_19, p_chi19, df19, expected19 = (
    stats.chi2_contingency(
        rq19_table.to_numpy(),
        correction=False
    )
)

if expected19.min() >= 5:
    rq19_method = "Pearson Chi-square"
    rq19_stat = float(chi2_19)
    rq19_p = float(p_chi19)
    rq19_df = int(df19)
else:
    odds19, fisher19 = stats.fisher_exact(
        rq19_table.to_numpy(),
        alternative="two-sided"
    )
    rq19_method = "Fisher exact"
    rq19_stat = float(odds19)
    rq19_p = float(fisher19)
    rq19_df = np.nan

# Probability of low review by delivery label.
x0 = int(rq19_table.loc[0, 1])
n0 = int(rq19_table.loc[0].sum())

x1 = int(rq19_table.loc[1, 1])
n1 = int(rq19_table.loc[1].sum())

p0 = x0 / n0
p1 = x1 / n1
rd19 = p1 - p0

rd19_lo, rd19_hi = confint_proportions_2indep(
    x1, n1, x0, n0,
    compare="diff",
    method="newcomb",
    alpha=ALPHA
)

phi19 = (
    np.sign(rd19)
    * np.sqrt(chi2_19 / len(rq19))
)

rq19_result = {
    "rq": "RQ19",
    "variable": "low_review",
    "method": rq19_method,
    "initial_n": len(df_labeled),
    "missing_n": rq19_missing_n,
    "n": len(rq19),
    "n_late": n1,
    "n_not_late": n0,
    "test_statistic": rq19_stat,
    "degrees_of_freedom": rq19_df,
    "p_value_raw": rq19_p,
    "p_value_holm": np.nan,
    "effect_size_name": "Signed Phi coefficient",
    "effect_size": float(phi19),
    "estimate_name": "Low-review risk difference",
    "estimate": float(rd19),
    "ci_lower": float(rd19_lo),
    "ci_upper": float(rd19_hi),
    "ci_method": "Newcombe",
    "expected_min": float(expected19.min()),
    "review_threshold": "<=2 stars"
}

print("\nRQ19 RESULTS")
print("Retained orders:", len(rq19))
print("Excluded missing reviews:", rq19_missing_n)
print("Low-review rate, Late:", f"{p1:.4%}")
print("Low-review rate, Not Late:", f"{p0:.4%}")
print("Risk difference:", f"{rd19 * 100:.4f} pp")
print(
    "Newcombe 95% CI:",
    f"[{rd19_lo * 100:.4f}, {rd19_hi * 100:.4f}] pp"
)
print("Method:", rq19_method)
print("Raw p-value:", f"{rq19_p:.10g}")
print("Signed Phi:", f"{phi19:.6f}")

print(
    "NOTE: Reviews occur after purchase/delivery. "
    "This test is associative, not predictive or causal."
)

# ============================================================
# PART B: RQ20 - CANCELLATION VS CUSTOMER STATE
# ============================================================

print("\nRQ20 - CANCELLATION VS CUSTOMER STATE")
print("=" * 70)

rq20 = df[
    ["order_id", "order_status", "customer_state"]
].copy()

rq20_initial_n = len(rq20)

# Do not silently classify missing statuses.
rq20_missing_n = int(
    rq20[["order_status", "customer_state"]]
    .isna().any(axis=1).sum()
)

rq20 = rq20.dropna(
    subset=["order_status", "customer_state"]
).copy()

assert rq20["order_id"].is_unique

rq20["is_canceled"] = (
    rq20["order_status"] == "canceled"
).astype(int)

rq20_table = pd.crosstab(
    rq20["customer_state"],
    rq20["is_canceled"]
).reindex(
    columns=[0, 1],
    fill_value=0
)

if len(rq20_table) < 2:
    raise ValueError("RQ20: fewer than two states")

if (rq20_table.sum(axis=0) == 0).any():
    raise ValueError("RQ20: missing cancellation class")

rq20_summary = rq20_table.copy()
rq20_summary.columns = ["Not_Canceled", "Canceled"]

rq20_summary["total_orders"] = (
    rq20_summary["Not_Canceled"]
    + rq20_summary["Canceled"]
)

rq20_summary["cancel_rate"] = (
    rq20_summary["Canceled"]
    / rq20_summary["total_orders"]
)

lo20, hi20 = proportion_confint(
    rq20_summary["Canceled"].to_numpy(),
    rq20_summary["total_orders"].to_numpy(),
    alpha=ALPHA,
    method="wilson"
)

rq20_summary["ci_lower"] = lo20
rq20_summary["ci_upper"] = hi20
rq20_summary["cancel_rate_pct"] = (
    100 * rq20_summary["cancel_rate"]
)

print("\nCANCELLATION BY CUSTOMER STATE")
display(
    rq20_summary.sort_values(
        "cancel_rate", ascending=False
    )
)

observed20 = rq20_table.to_numpy()

chi2_20, asym_p20, df20, expected20 = (
    stats.chi2_contingency(
        observed20,
        correction=False
    )
)

expected_min20 = float(expected20.min())
expected_lt5_20 = int((expected20 < 5).sum())
expected_lt1_20 = int((expected20 < 1).sum())
expected_lt5_pct20 = (
    100 * expected_lt5_20 / expected20.size
)

# Sparse multi-state tables use conditional Monte Carlo.
screening_pass20 = (
    expected_min20 >= 1
    and expected_lt5_pct20 <= 20
)

if screening_pass20:
    rq20_method = "Pearson Chi-square"
    rq20_p = float(asym_p20)
    rq20_resamples = 0
    rq20_extreme = np.nan
else:
    rq20_method = "Conditional Monte Carlo Chi-square"
    rq20_resamples = 20000

    rng20 = np.random.default_rng(RANDOM_SEED + 20)

    simulated20 = stats.random_table.rvs(
        observed20.sum(axis=1),
        observed20.sum(axis=0),
        size=rq20_resamples,
        random_state=rng20
    )

    simulated_chi20 = (
        ((simulated20 - expected20) ** 2 / expected20)
        .sum(axis=(1, 2))
    )

    rq20_extreme = int(
        np.count_nonzero(
            simulated_chi20 >= chi2_20 - 1e-12
        )
    )

    rq20_p = (
        rq20_extreme + 1
    ) / (rq20_resamples + 1)

n20 = int(observed20.sum())

cramers_v20 = float(np.sqrt(
    chi2_20
    / (
        n20
        * min(observed20.shape[0] - 1,
              observed20.shape[1] - 1)
    )
))

rq20_result = {
    "rq": "RQ20",
    "variable": "customer_state",
    "method": rq20_method,
    "initial_n": rq20_initial_n,
    "missing_n": rq20_missing_n,
    "n": n20,
    "n_groups": len(rq20_table),
    "test_statistic": float(chi2_20),
    "degrees_of_freedom": int(df20),
    "p_value_raw": float(rq20_p),
    "p_value_holm": np.nan,
    "effect_size_name": "Cramer's V",
    "effect_size": cramers_v20,
    "expected_min": expected_min20,
    "expected_lt5": expected_lt5_20,
    "expected_lt1": expected_lt1_20,
    "monte_carlo_resamples": rq20_resamples,
    "monte_carlo_extreme": rq20_extreme
}

print("\nRQ20 RESULTS")
print("Initial orders:", rq20_initial_n)
print("Excluded missing fields:", rq20_missing_n)
print("Retained orders:", n20)
print("Canceled orders:", int(rq20["is_canceled"].sum()))
print("Customer states:", len(rq20_table))
print("Chi-square:", f"{chi2_20:.6f}")
print("Degrees of freedom:", df20)
print("Minimum expected count:", f"{expected_min20:.4f}")
print("Cells expected < 5:", expected_lt5_20)
print("Cells expected < 1:", expected_lt1_20)
print("Method:", rq20_method)
print("Raw p-value:", f"{rq20_p:.10g}")
print("Cramer's V:", f"{cramers_v20:.6f}")

if rq20_resamples:
    print("Monte Carlo resamples:", rq20_resamples)
    print("Extreme simulations:", rq20_extreme)

print(
    "NOTE: Non-canceled includes delivered and "
    "other statuses, including orders still in progress."
)

# ============================================================
# PART C: COMBINED RESULTS
# ============================================================

final_two_results_df = pd.DataFrame([
    rq19_result,
    rq20_result
])

print("\nCOMBINED RQ19 + RQ20 RESULTS")
display(
    final_two_results_df[
        [
            "rq",
            "variable",
            "method",
            "initial_n",
            "missing_n",
            "n",
            "p_value_raw",
            "effect_size",
            "estimate",
            "ci_lower",
            "ci_upper"
        ]
    ]
)

print("\nRQ19 + RQ20 ANALYSIS COMPLETED.")


RQ19 + RQ20 - FINAL DATA ANALYSES

RQ19 - LOW REVIEW VS LATE DELIVERY


Raw cleaned-review rows: 99224
Valid review rows: 99224
Excluded review rows: 0



REVIEW DUPLICATE AUDIT
Orders with multiple valid reviews: 547
Orders with conflicting valid scores: 202



REVIEW COVERAGE BY DELIVERY LABEL


,labeled_orders,reviewed_orders,missing_reviews,missing_review_pct
is_late,,,,
0,89936,89443,493,0.5482
1,6534,6381,153,2.3416



REVIEW SCORE DISTRIBUTION


... (4 outputs left out)

,Not_Canceled,Canceled,total_orders,cancel_rate,ci_lower,ci_upper,cancel_rate_pct
customer_state,,,,,,,
RR,45,1,46,0.0217,0.0038,0.1134,2.1739
RO,250,3,253,0.0119,0.0040,0.0343,1.1858
PI,491,4,495,0.0081,0.0031,0.0206,0.8081
SP,41419,327,41746,0.0078,0.0070,0.0087,0.7833
RJ,12766,86,12852,0.0067,0.0054,0.0083,0.6692
GO,2007,13,2020,0.0064,0.0038,0.0110,0.6436
MG,11571,64,11635,0.0055,0.0043,0.0070,0.5501
MA,743,4,747,0.0054,0.0021,0.0137,0.5355
CE,1329,7,1336,0.0052,0.0025,0.0108,0.5240



RQ20 RESULTS
Initial orders: 99441
Excluded missing fields: 0
Retained orders: 99441
Canceled orders: 625
Customer states: 27
Chi-square: 47.713059
Degrees of freedom: 26
Minimum expected count: 0.2891
Cells expected < 5: 13
Cells expected < 1: 4
Method: Conditional Monte Carlo Chi-square
Raw p-value: 0.01479926004
Cramer's V: 0.021905
Monte Carlo resamples: 20000
Extreme simulations: 295
NOTE: Non-canceled includes delivered and other statuses, including orders still in progress.

COMBINED RQ19 + RQ20 RESULTS


,rq,variable,method,initial_n,missing_n,n,p_value_raw,effect_size,estimate,ci_lower,ci_upper
0,RQ19,low_review,Pearson Chi-square,96470,646,95824,0.0000,0.3964,0.5314,0.5192,0.5433
1,RQ20,customer_state,Conditional Monte Carlo Chi-square,99441,0,99441,0.0148,0.0219,NaN,NaN,NaN



RQ19 + RQ20 ANALYSIS COMPLETED.


In [25]:

# ============================================================
# Cell 24: Consolidate results and apply Holm correction
# ============================================================

from collections import Counter

# Collect all primary tests. Sensitivity tests are excluded.
result_rows = [
    rq1_result,
    rq2_result,
    rq3_result,
    rq4_result,
    rq5_result,
    *rq6_results_df.to_dict("records"),
    *rq7_results_df.to_dict("records"),
    *rq8_results_df.to_dict("records"),
    rq9_result,
    rq10_result,
    rq11_result,
    rq12_result,
    rq13_result,
    rq14_result,
    *weather_results_df.to_dict("records"),
    rq19_result,
    rq20_result,
]

all_results_df = pd.DataFrame(result_rows).copy()

# The testing plan uses group names for these three RQs.
# The corresponding result objects use source column names.

canonical_names = {
    "RQ10": "product_category_count_group",
    "RQ13": "national_holiday_count_group",
    "RQ14": "days_to_next_holiday_group",
    "RQ20": "is_canceled",
}


all_results_df["plan_variable"] = all_results_df.apply(
    lambda row: canonical_names.get(
        row["rq"], row["variable"]
    ),
    axis=1
)

# Validate planned tests against actual results.
expected_keys = list(zip(
    test_plan_df["rq"],
    test_plan_df["variable"]
))

actual_keys = list(zip(
    all_results_df["rq"],
    all_results_df["plan_variable"]
))

assert len(all_results_df) == 26, (
    f"Expected 26 tests, got {len(all_results_df)}"
)

assert Counter(expected_keys) == Counter(actual_keys), (
    "Actual tests do not match the testing plan."
)

all_results_df["p_value_raw"] = pd.to_numeric(
    all_results_df["p_value_raw"],
    errors="raise"
)

p_values = all_results_df["p_value_raw"].to_numpy()

assert np.isfinite(p_values).all()
assert ((p_values >= 0) & (p_values <= 1)).all()

# Holm family-wise error rate correction.
reject, adjusted_p, _, _ = multipletests(
    p_values,
    alpha=ALPHA,
    method="holm"
)

all_results_df["p_value_holm"] = adjusted_p
all_results_df["reject_h0_holm"] = reject
all_results_df["holm_family"] = "stage3_all"

# Preserve the original testing-plan order.
all_results_df = (
    test_plan_df[["rq", "variable"]]
    .rename(columns={"variable": "plan_variable"})
    .merge(
        all_results_df,
        on=["rq", "plan_variable"],
        how="left",
        validate="one_to_one",
        sort=False
    )
)

assert all_results_df["p_value_raw"].notna().all()

print("STAGE 3 - HOLM CORRECTION")
print("=" * 70)
print("Planned tests:", len(test_plan_df))
print("Actual tests:", len(all_results_df))
print("Adjustment method: Holm")
print("Alpha:", ALPHA)
print("Significant after Holm:", int(reject.sum()))
print("Not significant after Holm:", int((~reject).sum()))

print("\nFINAL STATISTICAL TEST SUMMARY")

with pd.option_context(
    "display.float_format",
    lambda value: f"{value:.6g}"
):
    display(
        all_results_df[
            [
                "rq",
                "variable",
                "n",
                "p_value_raw",
                "p_value_holm",
                "reject_h0_holm",
                "effect_size_name",
                "effect_size"
            ]
        ]
    )

print(
    "\nNOTE: Any displayed p-value of 0 reflects "
    "numerical underflow, not an exact zero probability."
)
print(
    "Holm correction addresses multiple comparisons; "
    "it does not resolve dependence or selection bias."
)
print(
    "Final claims require team confirmation of the "
    "testing plan and review of methodological limits."
)

print("\nHOLM CORRECTION COMPLETED.")


STAGE 3 - HOLM CORRECTION
Planned tests: 26
Actual tests: 26
Adjustment method: Holm
Alpha: 0.05
Significant after Holm: 26
Not significant after Holm: 0

FINAL STATISTICAL TEST SUMMARY


,rq,variable,n,p_value_raw,p_value_holm,reject_h0_holm,effect_size_name,effect_size
0,RQ1,customer_state,96470,4.99975e-05,0.000699965,True,Cramer's V,0.136057
1,RQ2,purchase_weekday,96470,0.000199452,0.00239342,True,Cramer's V,0.0164976
2,RQ3,purchase_month,96470,4.99975e-05,0.000699965,True,Cramer's V,0.193282
3,RQ4,has_interstate_seller,96470,3.60318e-96,8.64764e-95,True,Phi magnitude,0.0669963
4,RQ5,distance_km_mean,95990,1.98063e-102,4.95158e-101,True,Rank-biserial correlation,0.159501
5,RQ6,order_total_value,96470,1.20244e-08,2.04415e-07,True,Mean difference,17.4984
6,RQ6,freight_value,96470,2.71148e-18,5.96526e-17,True,Mean difference,2.64065
7,RQ7,freight_to_items_ratio,96470,0.00843886,0.0168777,True,Mean difference,0.0108135
8,RQ7,mean_item_value,96470,1.97575e-10,3.55634e-09,True,Mean difference,18.0071
9,RQ7,freight_per_item,96470,1.78893e-37,4.11454e-36,True,Mean difference,3.02392



NOTE: Any displayed p-value of 0 reflects numerical underflow, not an exact zero probability.
Holm correction addresses multiple comparisons; it does not resolve dependence or selection bias.
Final claims require team confirmation of the testing plan and review of methodological limits.

HOLM CORRECTION COMPLETED.


In [26]:

# ============================================================
# Cell 25: Export Stage 3 Statistical Tables
# ============================================================

# All results are exported from previously computed objects.
# No statistical tests are rerun.
# The source dataset is not modified.

TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "inferential_statistics"
)

TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("STAGE 3 - EXPORT STATISTICAL TABLES")
print("=" * 65)

# ------------------------------------------------------------
# 1. Primary statistical test results
# ------------------------------------------------------------

statistical_test_results_df = all_results_df.copy()

assert len(statistical_test_results_df) == 26
assert statistical_test_results_df[
    ["rq", "plan_variable"]
].duplicated().sum() == 0

assert statistical_test_results_df[
    "p_value_holm"
].notna().all()

statistical_test_results_df[
    "p_value_numerical_underflow"
] = (
    statistical_test_results_df["p_value_raw"] == 0
)

test_path = TABLE_DIR / "statistical_test_results.csv"

statistical_test_results_df.to_csv(
    test_path,
    index=False,
    encoding="utf-8-sig",
    float_format="%.12g"
)

# ------------------------------------------------------------
# 2. Sample summary - one row per test
# ------------------------------------------------------------

sample_columns = [
    "rq",
    "variable",
    "initial_n",
    "missing_n",
    "n",
    "n_late",
    "n_not_late"
]

sample_summary_df = (
    statistical_test_results_df
    .reindex(columns=sample_columns)
    .copy()
)

# For RQ20, the target is cancellation, not is_late.
# Do not label its canceled/non-canceled groups as Late.
sample_summary_df.loc[
    sample_summary_df["rq"] == "RQ20",
    ["n_late", "n_not_late"]
] = np.nan

sample_summary_df["excluded_n"] = (
    sample_summary_df["initial_n"]
    - sample_summary_df["n"]
)

sample_summary_df["excluded_pct"] = (
    100
    * sample_summary_df["excluded_n"]
    / sample_summary_df["initial_n"]
)

assert (sample_summary_df["excluded_n"] >= 0).all()
assert (
    sample_summary_df["excluded_n"]
    == sample_summary_df["missing_n"]
).all()

sample_path = TABLE_DIR / "sample_summary.csv"

sample_summary_df.to_csv(
    sample_path,
    index=False,
    encoding="utf-8-sig",
    float_format="%.12g"
)

# ------------------------------------------------------------
# 3. Continuous-variable descriptive statistics
# ------------------------------------------------------------

def normalize_descriptive(source, rq, variable=None,
                          unit=None):

    result = source.copy()

    if "rq" not in result.columns:
        result["rq"] = rq

    if variable is not None:
        result["variable"] = variable

    if unit is not None:
        result["unit"] = unit

    # Use one standard set of descriptive columns.
    required = [
        "rq", "variable", "group", "unit",
        "n", "mean", "sd", "median",
        "q1", "q3", "min", "max"
    ]

    return result.reindex(columns=required)


continuous_parts = [
    normalize_descriptive(
        rq5_descriptive,
        "RQ5",
        "distance_km_mean",
        "km"
    ),
    normalize_descriptive(
        rq6_descriptive_df,
        "RQ6"
    ),
    normalize_descriptive(
        rq7_descriptive_df,
        "RQ7"
    ),
    normalize_descriptive(
        rq8_descriptive_df,
        "RQ8"
    ),
    normalize_descriptive(
        rq11_descriptive_df,
        "RQ11",
        "estimated_window_days",
        "days"
    ),
    normalize_descriptive(
        weather_descriptive_df,
        "WEATHER"
    )
]

descriptive_summary_df = pd.concat(
    continuous_parts,
    ignore_index=True
)

# Weather rows already retain their specific RQ IDs.
assert descriptive_summary_df[
    ["rq", "variable", "group"]
].duplicated().sum() == 0

descriptive_path = (
    TABLE_DIR / "descriptive_summary.csv"
)

descriptive_summary_df.to_csv(
    descriptive_path,
    index=False,
    encoding="utf-8-sig",
    float_format="%.12g"
)

# ------------------------------------------------------------
# 4. Weather-specific test results
# ------------------------------------------------------------

weather_test_results_df = (
    weather_results_df.copy()
)


# Keep only the six weather tests when merging
# Holm-adjusted results.

weather_holm_lookup = (
    statistical_test_results_df.loc[
        statistical_test_results_df["rq"].isin(
            WEATHER_TESTS.keys()
        ),
        ["rq", "p_value_holm", "reject_h0_holm"]
    ]
    .copy()
)

assert len(weather_holm_lookup) == 6
assert weather_holm_lookup["rq"].is_unique

weather_test_results_df = weather_test_results_df.merge(
    weather_holm_lookup,
    on="rq",
    how="left",
    suffixes=("", "_final"),
    validate="one_to_one"
)


# If primary weather table already contains placeholder
# p_value_holm, replace it with the finalized values.

if "p_value_holm_final" in weather_test_results_df.columns:
    weather_test_results_df["p_value_holm"] = (
        weather_test_results_df["p_value_holm_final"]
    )
    weather_test_results_df = (
        weather_test_results_df.drop(
            columns=["p_value_holm_final"]
        )
    )

# Add 14-day block sensitivity results.
weather_test_results_df = (
    weather_test_results_df.merge(
        weather_sensitivity_df[
            [
                "rq",
                "block_days",
                "ci_lower",
                "ci_upper",
                "p_value_raw",
                "n_blocks"
            ]
        ].rename(columns={
            "block_days": "sensitivity_block_days",
            "ci_lower": "sensitivity_ci_lower",
            "ci_upper": "sensitivity_ci_upper",
            "p_value_raw": "sensitivity_p_value_raw",
            "n_blocks": "sensitivity_n_blocks"
        }),
        on="rq",
        how="left",
        validate="one_to_one"
    )
)

assert len(weather_test_results_df) == 6
assert weather_test_results_df[
    "p_value_holm"
].notna().all()

weather_path = TABLE_DIR / "weather_test_results.csv"

weather_test_results_df.to_csv(
    weather_path,
    index=False,
    encoding="utf-8-sig",
    float_format="%.12g"
)

# ------------------------------------------------------------
# 5. Final export audit
# ------------------------------------------------------------

export_audit = pd.DataFrame([
    {
        "file": test_path.name,
        "rows": len(statistical_test_results_df)
    },
    {
        "file": sample_path.name,
        "rows": len(sample_summary_df)
    },
    {
        "file": descriptive_path.name,
        "rows": len(descriptive_summary_df)
    },
    {
        "file": weather_path.name,
        "rows": len(weather_test_results_df)
    }
])

print("\nEXPORT AUDIT")
display(export_audit)

print("\nOUTPUT FOLDER")
print(TABLE_DIR)

print("\nEXPORT VALIDATION")
print("Primary tests:", len(statistical_test_results_df))
print("Sample rows:", len(sample_summary_df))
print("Weather tests:", len(weather_test_results_df))

for path in [
    test_path,
    sample_path,
    descriptive_path,
    weather_path
]:
    assert path.exists()
    assert path.stat().st_size > 0
    print("CREATED:", path.name)

print("\nSTAGE 3 TABLE EXPORT COMPLETED.")


STAGE 3 - EXPORT STATISTICAL TABLES

EXPORT AUDIT


,file,rows
0,statistical_test_results.csv,26
1,sample_summary.csv,26
2,descriptive_summary.csv,30
3,weather_test_results.csv,6



OUTPUT FOLDER
C:\Users\LENOVO\OneDrive\Desktop\PTTK\outputs\tables\inferential_statistics

EXPORT VALIDATION
Primary tests: 26
Sample rows: 26
Weather tests: 6
CREATED: statistical_test_results.csv
CREATED: sample_summary.csv
CREATED: descriptive_summary.csv
CREATED: weather_test_results.csv

STAGE 3 TABLE EXPORT COMPLETED.


In [27]:

# ============================================================
# Cell 26: Generate Stage 3 Figures
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGURE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
    / "inferential_statistics"
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})

figure_audit = []

def save_stage3_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(
        path,
        dpi=180,
        bbox_inches="tight",
        facecolor="white"
    )
    figure_audit.append({
        "file": filename,
        "bytes": path.stat().st_size
    })
    plt.show()
    plt.close(fig)


# ------------------------------------------------------------
# FIGURE 1: Late delivery rates by customer state
# ------------------------------------------------------------

state_table = pd.crosstab(
    df_labeled["customer_state"],
    df_labeled["is_late"]
).reindex(columns=[0, 1], fill_value=0)

state_table["n"] = state_table[0] + state_table[1]
state_table["late_rate_pct"] = (
    100 * state_table[1] / state_table["n"]
)

# Select states with at least 100 orders for readability.
# Smaller states remain in the RQ1 statistical test.
state_plot = (
    state_table.loc[state_table["n"] >= 100]
    .sort_values("late_rate_pct", ascending=False)
)

fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(
    state_plot.index,
    state_plot["late_rate_pct"],
    color="#4277A7"
)
ax.invert_yaxis()

ax.set_title(
    "Late Delivery Rate by Customer State",
    fontweight="bold"
)
ax.set_xlabel("Late delivery rate (%)")
ax.set_ylabel("Customer state")

ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

fig.text(
    0.5, -0.01,
    "Descriptive plot: states with n >= 100. "
    "The RQ1 test retains all observed states.",
    ha="center",
    fontsize=9
)

fig.tight_layout()

save_stage3_figure(
    fig,
    "rq1_late_rate_by_customer_state.png"
)


# ------------------------------------------------------------
# FIGURE 2: Monthly late delivery rate
# ------------------------------------------------------------

monthly = rq3_summary.copy()

monthly_plot = monthly.loc[
    monthly["total_orders"] >= 100
].copy()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    monthly_plot.index,
    monthly_plot["late_rate_pct"],
    marker="o",
    linewidth=2,
    color="#A85273"
)

ax.fill_between(
    np.arange(len(monthly_plot)),
    monthly_plot["ci_lower_pct"].to_numpy(),
    monthly_plot["ci_upper_pct"].to_numpy(),
    alpha=0.15,
    color="#A85273"
)

ax.set_xticks(np.arange(len(monthly_plot)))
ax.set_xticklabels(
    monthly_plot.index,
    rotation=60,
    ha="right"
)

ax.set_title(
    "Late Delivery Rates Across Purchase Months",
    fontweight="bold"
)
ax.set_ylabel("Late delivery rate (%)")
ax.set_xlabel("Purchase month")

ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

fig.text(
    0.5, -0.03,
    "Wilson 95% intervals. Months with fewer than "
    "100 labeled orders are omitted from this plot only. "
    "Not evidence of seasonality.",
    ha="center",
    fontsize=8
)

fig.tight_layout()

save_stage3_figure(
    fig,
    "rq3_monthly_late_delivery_rates.png"
)


# ------------------------------------------------------------
# FIGURE 3: Low review rate by delivery label
# ------------------------------------------------------------

review_plot = pd.DataFrame({
    "group": ["Not Late", "Late"],
    "rate": [
        100 * rq19_table.loc[0, 1]
        / rq19_table.loc[0].sum(),
        100 * rq19_table.loc[1, 1]
        / rq19_table.loc[1].sum()
    ]
})

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    review_plot["group"],
    review_plot["rate"],
    color=["#4277A7", "#A85273"],
    width=0.55
)

ax.bar_label(
    bars,
    fmt="%.2f%%",
    padding=4
)

ax.set_ylim(
    0,
    max(review_plot["rate"]) * 1.2
)

ax.set_ylabel("Low review rate (%)")
ax.set_title(
    "Low Review Rate by Delivery Status",
    fontweight="bold"
)

ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

fig.text(
    0.5, -0.02,
    "Low review = 1-2 stars; one selected valid "
    "review per order. Observational association.",
    ha="center",
    fontsize=9
)

fig.tight_layout()

save_stage3_figure(
    fig,
    "rq19_low_review_by_delivery_status.png"
)


# ------------------------------------------------------------
# FIGURE 4: Weather mean differences with block CI
# ------------------------------------------------------------

weather_plot = weather_results_df[
    ["rq", "variable", "estimate", "ci_lower", "ci_upper"]
].copy()

weather_labels = {
    "RQ15a": "Origin rain (mm)",
    "RQ15b": "Destination rain (mm)",
    "RQ16a": "Origin humidity (percentage points)",
    "RQ16b": "Destination humidity (percentage points)",
    "RQ17a": "Origin temperature (C)",
    "RQ17b": "Destination temperature (C)"
}

# Separate units in individual panels.
weather_plot["domain"] = weather_plot["rq"].str[:4]

fig, axes = plt.subplots(
    1, 3,
    figsize=(14, 4.8)
)

domain_config = [
    ("RQ15", "Rainfall", "mm"),
    ("RQ16", "Humidity", "percentage points"),
    ("RQ17", "Air temperature", "C")
]

for ax, (domain, title, unit) in zip(
    axes, domain_config
):

    subset = weather_plot.loc[
        weather_plot["domain"] == domain
    ].copy()

    y = np.arange(len(subset))

    estimates = subset["estimate"].to_numpy()
    lower = subset["ci_lower"].to_numpy()
    upper = subset["ci_upper"].to_numpy()

    asymmetric_errors = np.vstack([
        estimates - lower,
        upper - estimates
    ])

    ax.errorbar(
        estimates,
        y,
        xerr=asymmetric_errors,
        fmt="o",
        capsize=5,
        color="#4277A7",
        ecolor="#4277A7"
    )

    ax.axvline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1
    )

    ax.set_yticks(y)
    ax.set_yticklabels(
        [
            "Origin" if "origin_" in v else "Destination"
            for v in subset["variable"]
        ]
    )

    ax.invert_yaxis()
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(f"Mean difference ({unit})")
    ax.grid(axis="x", alpha=0.2)

fig.suptitle(
    "Weather Differences: Late - Not Late",
    fontweight="bold",
    fontsize=14
)

fig.text(
    0.5, -0.02,
    "Error bars: exploratory 95% temporal block "
    "bootstrap intervals (7-day blocks). "
    "Different panels have different units.",
    ha="center",
    fontsize=9
)

fig.tight_layout()

save_stage3_figure(
    fig,
    "rq15_rq17_weather_mean_differences.png"
)


# ------------------------------------------------------------
# FINAL FIGURE AUDIT
# ------------------------------------------------------------

figure_audit_df = pd.DataFrame(figure_audit)

print("\nSTAGE 3 - FIGURE EXPORT AUDIT")
print("=" * 65)

display(figure_audit_df)

assert len(figure_audit_df) == 4
assert (figure_audit_df["bytes"] > 0).all()

print("\nFigure folder:", FIGURE_DIR)
print("Figures created:", len(figure_audit_df))
print("\nSTAGE 3 FIGURES COMPLETED.")


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16436\3782257629.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16436\3782257629.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



STAGE 3 - FIGURE EXPORT AUDIT


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16436\3782257629.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,file,bytes
0,rq1_late_rate_by_customer_state.png,66755
1,rq3_monthly_late_delivery_rates.png,165186
2,rq19_low_review_by_delivery_status.png,47278
3,rq15_rq17_weather_mean_differences.png,65629



Figure folder: C:\Users\LENOVO\OneDrive\Desktop\PTTK\outputs\figures\inferential_statistics
Figures created: 4

STAGE 3 FIGURES COMPLETED.


In [28]:
# ============================================================
# Cell 27: Write docs/inferential_statistics/delivery_risk_association_results.md
# ============================================================

from pathlib import Path
import math
import pandas as pd
import numpy as np

DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = DOCS_DIR / "inferential_statistics/delivery_risk_association_results.md"

print("STAGE 3 - WRITE MARKDOWN REPORT")
print("=" * 70)

def fmt_num(x, digits=4):
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}"

def fmt_pct_from_prop(x, digits=2):
    if pd.isna(x):
        return "NA"
    return f"{x * 100:.{digits}f}%"

def fmt_p(x):
    if pd.isna(x):
        return "NA"
    if x == 0:
        return "Numerically underflowed (reported as 0)"
    if x < 0.0001:
        return f"{x:.2e}"
    return f"{x:.6f}"

def fmt_bool(x):
    return "Reject H0" if bool(x) else "Do not reject H0"

# ------------------------------------------------------------
# 1. Core overview values
# ------------------------------------------------------------

n_tests = len(all_results_df)
n_sig_holm = int(all_results_df["reject_h0_holm"].sum())
n_not_sig_holm = n_tests - n_sig_holm


# For short highlights.
rq19_row = all_results_df.loc[
    all_results_df["rq"] == "RQ19"
].iloc[0]

rq20_row = all_results_df.loc[
    all_results_df["rq"] == "RQ20"
].iloc[0]

rq7_ratio_row = all_results_df.loc[
    (all_results_df["rq"] == "RQ7")
    & (all_results_df["variable"] == "freight_to_items_ratio")
].iloc[0]

# Weather rows
weather_rows = all_results_df.loc[
    all_results_df["rq"].isin(list(WEATHER_TESTS.keys()))
].copy()

# ------------------------------------------------------------
# 2. Build per-RQ summary text
# ------------------------------------------------------------

rq_sections = []


# Build lookup using the canonical test names
# defined in the Holm testing plan.

results_lookup = {
    (row["rq"], row["plan_variable"]): row
    for _, row in all_results_df.iterrows()
}

# Validate that all 26 planned tests are accessible.
assert len(results_lookup) == 26

assert set(results_lookup.keys()) == set(
    zip(
        test_plan_df["rq"],
        test_plan_df["variable"]
    )
)

print("Report lookup validation: PASS (26/26 tests)")


# RQ1
rq_sections.append(
    f"""## RQ1 — Customer state and late delivery

- Variable(s): `customer_state`, `is_late`
- Sample size: {int(rq1_result["n"])}
- Method: {rq1_result["method"]}
- Effect size: Cramér's V = {fmt_num(rq1_result["effect_size"], 6)}
- Raw p-value: {fmt_p(rq1_result["p_value_raw"])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ1", "customer_state")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

# RQ2
rq_sections.append(
    f"""## RQ2 — Purchase weekday and late delivery

- Variable(s): `purchase_weekday`, `is_late`
- Sample size: {int(rq2_result["n"])}
- Method: {rq2_result["method"]}
- Effect size: Cramér's V = {fmt_num(rq2_result["effect_size"], 6)}
- Raw p-value: {fmt_p(rq2_result["p_value_raw"])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ2", "purchase_weekday")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

# RQ3
rq_sections.append(
    f"""## RQ3 — Purchase month and late delivery

- Variable(s): `purchase_month`, `is_late`
- Sample size: {int(rq3_result["n"])}
- Method: {rq3_result["method"]}
- Effect size: Cramér's V = {fmt_num(rq3_result["effect_size"], 6)}
- Raw p-value: {fmt_p(rq3_result["p_value_raw"])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ3", "purchase_month")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

# RQ4
rq_sections.append(
    f"""## RQ4 — Interstate seller and late delivery

- Variable(s): `has_interstate_seller`, `is_late`
- Sample size: {int(rq4_result["n"])}
- Method: {rq4_result["method"]}
- Risk difference: {fmt_num(rq4_result["estimate"] * 100, 4)} percentage points
- 95% CI: [{fmt_num(rq4_result["ci_lower"] * 100, 4)}, {fmt_num(rq4_result["ci_upper"] * 100, 4)}] percentage points
- Effect size: Phi = {fmt_num(rq4_result["effect_size"], 6)}
- Raw p-value: {fmt_p(rq4_result["p_value_raw"])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ4", "has_interstate_seller")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

# RQ5
rq_sections.append(
    f"""## RQ5 — Distance and late delivery

- Variable(s): `distance_km_mean`, `is_late`
- Sample size: {int(rq5_result["n"])}
- Method: {rq5_result["method"]}
- Effect size: Rank-biserial correlation = {fmt_num(rq5_result["effect_size"], 6)}
- 95% CI: [{fmt_num(rq5_result["ci_lower"], 6)}, {fmt_num(rq5_result["ci_upper"], 6)}]
- Raw p-value: {fmt_p(rq5_result["p_value_raw"])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ5", "distance_km_mean")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

# RQ6
rq_sections.append(
    f"""## RQ6 — Order value and freight value

Two separate Welch t-tests were used.

### RQ6a — `order_total_value`
- Mean difference (Late - Not Late): {fmt_num(rq6_results_df.loc[rq6_results_df["variable"] == "order_total_value", "estimate"].iloc[0], 4)} BRL
- 95% CI: [{fmt_num(rq6_results_df.loc[rq6_results_df["variable"] == "order_total_value", "ci_lower"].iloc[0], 4)}, {fmt_num(rq6_results_df.loc[rq6_results_df["variable"] == "order_total_value", "ci_upper"].iloc[0], 4)}]
- Raw p-value: {fmt_p(rq6_results_df.loc[rq6_results_df["variable"] == "order_total_value", "p_value_raw"].iloc[0])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ6", "order_total_value")]["p_value_holm"])}

### RQ6b — `freight_value`
- Mean difference (Late - Not Late): {fmt_num(rq6_results_df.loc[rq6_results_df["variable"] == "freight_value", "estimate"].iloc[0], 4)} BRL
- 95% CI: [{fmt_num(rq6_results_df.loc[rq6_results_df["variable"] == "freight_value", "ci_lower"].iloc[0], 4)}, {fmt_num(rq6_results_df.loc[rq6_results_df["variable"] == "freight_value", "ci_upper"].iloc[0], 4)}]
- Raw p-value: {fmt_p(rq6_results_df.loc[rq6_results_df["variable"] == "freight_value", "p_value_raw"].iloc[0])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ6", "freight_value")]["p_value_holm"])}

Sensitivity analysis using 10% trimmed means showed the same direction of differences, although the estimands differ.
"""
)

# RQ7
rq_sections.append(
    f"""## RQ7 — Relative freight cost and item value

Three separate Welch t-tests were used.

### RQ7a — `freight_to_items_ratio`
- Mean difference (Late - Not Late): {fmt_num(rq7_results_df.loc[rq7_results_df["variable"] == "freight_to_items_ratio", "estimate"].iloc[0], 6)}
- 95% CI: [{fmt_num(rq7_results_df.loc[rq7_results_df["variable"] == "freight_to_items_ratio", "ci_lower"].iloc[0], 6)}, {fmt_num(rq7_results_df.loc[rq7_results_df["variable"] == "freight_to_items_ratio", "ci_upper"].iloc[0], 6)}]
- Raw p-value: {fmt_p(rq7_results_df.loc[rq7_results_df["variable"] == "freight_to_items_ratio", "p_value_raw"].iloc[0])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ7", "freight_to_items_ratio")]["p_value_holm"])}

### RQ7b — `mean_item_value`
- Mean difference (Late - Not Late): {fmt_num(rq7_results_df.loc[rq7_results_df["variable"] == "mean_item_value", "estimate"].iloc[0], 4)} BRL/item
- 95% CI: [{fmt_num(rq7_results_df.loc[rq7_results_df["variable"] == "mean_item_value", "ci_lower"].iloc[0], 4)}, {fmt_num(rq7_results_df.loc[rq7_results_df["variable"] == "mean_item_value", "ci_upper"].iloc[0], 4)}]
- Raw p-value: {fmt_p(rq7_results_df.loc[rq7_results_df["variable"] == "mean_item_value", "p_value_raw"].iloc[0])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ7", "mean_item_value")]["p_value_holm"])}

### RQ7c — `freight_per_item`
- Mean difference (Late - Not Late): {fmt_num(rq7_results_df.loc[rq7_results_df["variable"] == "freight_per_item", "estimate"].iloc[0], 4)} BRL/item
- 95% CI: [{fmt_num(rq7_results_df.loc[rq7_results_df["variable"] == "freight_per_item", "ci_lower"].iloc[0], 4)}, {fmt_num(rq7_results_df.loc[rq7_results_df["variable"] == "freight_per_item", "ci_upper"].iloc[0], 4)}]
- Raw p-value: {fmt_p(rq7_results_df.loc[rq7_results_df["variable"] == "freight_per_item", "p_value_raw"].iloc[0])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ7", "freight_per_item")]["p_value_holm"])}

Sensitivity analysis showed that `freight_to_items_ratio` was the most fragile among the three variables: the trimmed-mean test was weaker than the primary Welch result. This should be discussed carefully.
"""
)

# RQ8
rq_sections.append(
    f"""## RQ8 — Product weight and volume

Two separate Mann–Whitney U tests were used.

### RQ8a — `total_product_weight_g`
- Rank-biserial correlation: {fmt_num(rq8_results_df.loc[rq8_results_df["variable"] == "total_product_weight_g", "effect_size"].iloc[0], 6)}
- 95% CI: [{fmt_num(rq8_results_df.loc[rq8_results_df["variable"] == "total_product_weight_g", "ci_lower"].iloc[0], 6)}, {fmt_num(rq8_results_df.loc[rq8_results_df["variable"] == "total_product_weight_g", "ci_upper"].iloc[0], 6)}]
- Raw p-value: {fmt_p(rq8_results_df.loc[rq8_results_df["variable"] == "total_product_weight_g", "p_value_raw"].iloc[0])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ8", "total_product_weight_g")]["p_value_holm"])}

### RQ8b — `total_product_volume_cm3`
- Rank-biserial correlation: {fmt_num(rq8_results_df.loc[rq8_results_df["variable"] == "total_product_volume_cm3", "effect_size"].iloc[0], 6)}
- 95% CI: [{fmt_num(rq8_results_df.loc[rq8_results_df["variable"] == "total_product_volume_cm3", "ci_lower"].iloc[0], 6)}, {fmt_num(rq8_results_df.loc[rq8_results_df["variable"] == "total_product_volume_cm3", "ci_upper"].iloc[0], 6)}]
- Raw p-value: {fmt_p(rq8_results_df.loc[rq8_results_df["variable"] == "total_product_volume_cm3", "p_value_raw"].iloc[0])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ8", "total_product_volume_cm3")]["p_value_holm"])}

Both effects were statistically significant but small.
"""
)

# RQ9-14
rq_sections.append(
    f"""## RQ9 — Seller count group and late delivery

- Variable(s): `seller_count_group`, `is_late`
- Method: {rq9_result["method"]}
- Risk difference: {fmt_num(rq9_result["estimate"] * 100, 4)} percentage points
- 95% CI: [{fmt_num(rq9_result["ci_lower"] * 100, 4)}, {fmt_num(rq9_result["ci_upper"] * 100, 4)}]
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ9", "seller_count_group")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

rq_sections.append(
    f"""## RQ10 — Product-category count group and late delivery

- Variable(s): `product_category_count_group`, `is_late`
- Method: {rq10_result["method"]}
- Risk difference: {fmt_num(rq10_result["estimate"] * 100, 4)} percentage points
- 95% CI: [{fmt_num(rq10_result["ci_lower"] * 100, 4)}, {fmt_num(rq10_result["ci_upper"] * 100, 4)}]
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ10", "product_category_count_group")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

rq_sections.append(
    f"""## RQ11 — Estimated delivery window and late delivery

- Variable(s): `estimated_window_days`, `is_late`
- Method: {rq11_result["method"]}
- Rank-biserial correlation: {fmt_num(rq11_result["effect_size"], 6)}
- 95% CI: [{fmt_num(rq11_result["ci_lower"], 6)}, {fmt_num(rq11_result["ci_upper"], 6)}]
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ11", "estimated_window_days")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

rq_sections.append(
    f"""## RQ12 — National holiday purchase day and late delivery

- Variable(s): `purchase_is_national_holiday`, `is_late`
- Method: {rq12_result["method"]}
- Risk difference: {fmt_num(rq12_result["estimate"] * 100, 4)} percentage points
- 95% CI: [{fmt_num(rq12_result["ci_lower"] * 100, 4)}, {fmt_num(rq12_result["ci_upper"] * 100, 4)}]
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ12", "purchase_is_national_holiday")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

rq_sections.append(
    f"""## RQ13 — Number of holidays in the estimated delivery window

- Variable(s): `national_holiday_count_group`, `is_late`
- Method: {rq13_result["method"]}
- Effect size: Cramér's V = {fmt_num(rq13_result["effect_size"], 6)}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ13", "national_holiday_count_group")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

rq_sections.append(
    f"""## RQ14 — Days to the next national holiday

- Variable(s): `days_to_next_holiday_group`, `is_late`
- Method: {rq14_result["method"]}
- Effect size: Cramér's V = {fmt_num(rq14_result["effect_size"], 6)}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ14", "days_to_next_holiday_group")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

# Weather section
weather_lines = []
for _, row in weather_rows.iterrows():
    weather_lines.append(
        f"- {row['rq']} (`{row['variable']}`): mean difference = "
        f"{fmt_num(row['estimate'], 4)}; 95% CI "
        f"[{fmt_num(row['ci_lower'], 4)}, {fmt_num(row['ci_upper'], 4)}]; "
        f"Holm-adjusted p-value = {fmt_p(row['p_value_holm'])}."
    )

rq_sections.append(
    "## RQ15–RQ17 — Weather before purchase and late delivery\n\n"
    "Six separate temporal block bootstrap analyses were used "
    "(7-day blocks as primary analysis; 14-day blocks as sensitivity analysis).\n\n"
    + "\n".join(weather_lines)
    + "\n\nInterpretation: all six weather comparisons showed positive mean differences "
      "(Late - Not Late), but conclusions remain sensitive to assumptions about temporal "
      "dependence, shared state-day weather measurements, and the adopted UTC scenario."
)

# RQ19
rq_sections.append(
    f"""## RQ19 — Low review and late delivery

- Variable(s): `low_review`, `is_late`
- Review rule: low review = 1–2 stars
- Sample size after excluding missing reviews: {int(rq19_result["n"])}
- Missing reviews excluded: {int(rq19_result["missing_n"])}
- Method: {rq19_result["method"]}
- Low-review rate in Late group: {fmt_num((rq19_table.loc[1, 1] / rq19_table.loc[1].sum()) * 100, 4)}%
- Low-review rate in Not Late group: {fmt_num((rq19_table.loc[0, 1] / rq19_table.loc[0].sum()) * 100, 4)}%
- Risk difference: {fmt_num(rq19_result["estimate"] * 100, 4)} percentage points
- 95% CI: [{fmt_num(rq19_result["ci_lower"] * 100, 4)}, {fmt_num(rq19_result["ci_upper"] * 100, 4)}]
- Effect size: Phi = {fmt_num(rq19_result["effect_size"], 6)}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ19", "low_review")]["p_value_holm"])}

The observed review association has signed Phi = {fmt_num(rq19_result["effect_size"], 6)}. Reviews occur after delivery, so this result is not a purchase-time predictor and must not be interpreted causally. Duplicate valid reviews existed for some orders, and the analysis kept one selected valid review per order using a deterministic rule.
"""
)

# RQ20
rq_sections.append(
    f"""## RQ20 — Cancellation and customer state

- Variable(s): `is_canceled`, `customer_state`
- Sample size: {int(rq20_result["n"])}
- Method: {rq20_result["method"]}
- Effect size: Cramér's V = {fmt_num(rq20_result["effect_size"], 6)}
- Raw p-value: {fmt_p(rq20_result["p_value_raw"])}
- Holm-adjusted p-value: {fmt_p(results_lookup[("RQ20", "is_canceled")]["p_value_holm"])}
- Interpretation: assess the reported estimate, confidence interval and Holm-adjusted p-value together; this observational comparison does not establish causation.
"""
)

# ------------------------------------------------------------
# 3. Final statistical summary table in markdown
# ------------------------------------------------------------

summary_lines = []
summary_lines.append("| RQ | Variable | n | Raw p | Holm p | Reject H0 after Holm | Effect size |")
summary_lines.append("|---|---|---:|---:|---:|---|---:|")

for _, row in all_results_df.iterrows():
    summary_lines.append(
        f"| {row['rq']} | {row['variable']} | {int(row['n'])} | "
        f"{fmt_p(row['p_value_raw'])} | {fmt_p(row['p_value_holm'])} | "
        f"{'Yes' if row['reject_h0_holm'] else 'No'} | "
        f"{fmt_num(row['effect_size'], 6)} |"
    )

summary_table_md = "\n".join(summary_lines)

# ------------------------------------------------------------
# 4. Compose full markdown
# ------------------------------------------------------------

report_text = f"""# Inferential Statistics — Stage 3

## 1. Objective

Stage 3 evaluated whether the differences observed in exploratory data analysis (EDA) were supported by inferential evidence. The goal was to estimate the magnitude of differences, quantify uncertainty, and test whether observed patterns were statistically distinguishable under the planned methods. This stage does **not** train machine-learning models and does **not** establish causation.

The main research question was:

> Which characteristics are associated with late delivery, how large are the observed differences, and how uncertain are these estimates?

## 2. Data used

- Primary analytic dataset: `data/processed/research_data_weather_price_holiday_assume_utc.csv`
- Each row represents one `order_id`
- The main late-delivery analyses used labeled orders with `is_late` equal to 0 (Not Late) or 1 (Late)
- Missing values were removed only for the specific variable used in each analysis
- RQ19 used a separate review-linked sample
- RQ20 used the full order-level dataset and was not restricted to delivered orders

## 3. Statistical design

- Significance level: 0.05
- Two-sided alternatives were used for relevant continuous and binary comparisons. Chi-square association tests use the right tail of the chi-square reference distribution.
- Multiple-testing adjustment: Holm family-wise correction across all 26 pre-specified primary tests
- Continuous variables were analysed with Welch t-tests, Mann–Whitney U tests, or temporal block bootstrap procedures, depending on the research question
- Categorical associations were analysed using Pearson chi-square, Fisher's exact test, or conditional Monte Carlo chi-square when sparse cells required it

## 4. Overall results

- Number of primary tests: {n_tests}
- Significant after Holm correction: {n_sig_holm}
- Not significant after Holm correction: {n_not_sig_holm}
- Review association: RQ19 (low review vs late delivery), signed Phi = {fmt_num(rq19_row["effect_size"], 6)}. Effect sizes with different units or definitions are not ranked against one another.

Of {n_tests} primary tests, {n_sig_holm} met the Holm-adjusted significance criterion and {n_not_sig_holm} did not. Statistical significance alone does not establish practical importance.

## 5. Final summary table

{summary_table_md}

## 6. Research-question summaries

{chr(10).join(rq_sections)}

## 7. Key interpretation points

1. **{n_sig_holm} of {n_tests} planned primary tests met the Holm-adjusted significance criterion**. Assess effect sizes and uncertainty separately.
2. **RQ19 (`low_review`) has signed Phi = {fmt_num(rq19_row["effect_size"], 6)}**. Reviews occur after delivery, so this result concerns outcome interpretation, not predictive feature selection at purchase time.
3. **Review practical magnitude and uncertainty separately from significance**, including:
   - RQ7 `freight_to_items_ratio`
   - RQ20 cancellation by state
   - several holiday-related variables with very small Cramér's V / Phi values
4. **Weather results were based on temporal block bootstrap**, not ordinary independent-order tests, because many orders shared the same state-day weather context.
5. **No post hoc pairwise group-comparison file was created** because the Stage 3 workflow did not include a finalized family of additional pairwise comparisons beyond the pre-specified primary tests.

## 8. Limitations

- This stage estimates association, **not causation**
- Many tests assume order-level independence or only partially address dependence
- Weather analyses still rely on assumptions about block independence and the UTC scenario
- `estimated_window_days` is conceptually related to the construction of the late-delivery outcome
- Review analysis required a deterministic rule to select one valid review per order when multiple reviews existed
- Cancellation analysis used a broad non-canceled comparison group that included delivered and in-progress orders
- Some statistically significant effects were practically small

## 9. Output files

### Tables
- `outputs/tables/inferential_statistics/sample_summary.csv`
- `outputs/tables/inferential_statistics/descriptive_summary.csv`
- `outputs/tables/inferential_statistics/statistical_test_results.csv`
- `outputs/tables/inferential_statistics/weather_test_results.csv`

### Figures
- `outputs/figures/inferential_statistics/rq1_late_rate_by_customer_state.png`
- `outputs/figures/inferential_statistics/rq3_monthly_late_delivery_rates.png`
- `outputs/figures/inferential_statistics/rq19_low_review_by_delivery_status.png`
- `outputs/figures/inferential_statistics/rq15_rq17_weather_mean_differences.png`

## 10. Completion note

Stage 3 inferential-statistics analysis is complete at the level of the planned primary tests, exported tables, and key figures. Any future Stage 4 modeling should not rely only on p-values; feature choice must also consider leakage, timing, interpretability, and dependence structure.
"""

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(report_text, encoding="utf-8")

print("CREATED REPORT:")
print(REPORT_PATH)
print("\nREPORT SIZE (bytes):", REPORT_PATH.stat().st_size)
print("\nSTAGE 3 MARKDOWN REPORT COMPLETED.")

STAGE 3 - WRITE MARKDOWN REPORT
Report lookup validation: PASS (26/26 tests)
CREATED REPORT:
C:\Users\LENOVO\OneDrive\Desktop\PTTK\docs\inferential_statistics\delivery_risk_association_results.md

REPORT SIZE (bytes): 16258

STAGE 3 MARKDOWN REPORT COMPLETED.


In [29]:

# ============================================================
# Cell 28: Stage 3 Final Delivery Audit
# Read-only: no statistical tests, no file modifications.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("STAGE 3 - FINAL DELIVERY AUDIT")
print("=" * 70)

ROOT = Path(PROJECT_ROOT)

required_files = {
    "Notebook": ROOT / "notebooks" / "03_inferential_statistics" / "test_delivery_risk_associations.ipynb",
    "Report": ROOT / "docs" / "inferential_statistics/delivery_risk_association_results.md",
    "Statistical results": TABLE_DIR / "statistical_test_results.csv",
    "Sample summary": TABLE_DIR / "sample_summary.csv",
    "Descriptive summary": TABLE_DIR / "descriptive_summary.csv",
    "Weather results": TABLE_DIR / "weather_test_results.csv",
    "Figure RQ1": FIGURE_DIR / "rq1_late_rate_by_customer_state.png",
    "Figure RQ3": FIGURE_DIR / "rq3_monthly_late_delivery_rates.png",
    "Figure RQ19": FIGURE_DIR / "rq19_low_review_by_delivery_status.png",
    "Figure Weather": FIGURE_DIR / "rq15_rq17_weather_mean_differences.png",
}

# ------------------------------------------------------------
# 1. File existence and size
# ------------------------------------------------------------

file_audit = pd.DataFrame([
    {
        "artifact": name,
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else 0,
        "status": (
            "PASS"
            if path.is_file() and path.stat().st_size > 0
            else "FAIL"
        ),
    }
    for name, path in required_files.items()
])

print("\n1. FILE AUDIT")
display(file_audit)

# ------------------------------------------------------------
# 2. Read exported CSV files
# ------------------------------------------------------------

test_csv = pd.read_csv(
    TABLE_DIR / "statistical_test_results.csv"
)
sample_csv = pd.read_csv(
    TABLE_DIR / "sample_summary.csv"
)
descriptive_csv = pd.read_csv(
    TABLE_DIR / "descriptive_summary.csv"
)
weather_csv = pd.read_csv(
    TABLE_DIR / "weather_test_results.csv"
)

checks = {}

checks["All 10 required artifacts exist"] = (
    file_audit["status"].eq("PASS").all()
)

checks["26 primary test rows"] = len(test_csv) == 26
checks["26 sample summary rows"] = len(sample_csv) == 26
checks["30 descriptive rows"] = len(descriptive_csv) == 30
checks["6 weather result rows"] = len(weather_csv) == 6

checks["No duplicate primary tests"] = (
    not test_csv[["rq", "plan_variable"]]
    .duplicated().any()
)

checks["All raw p-values valid"] = (
    test_csv["p_value_raw"].notna().all()
    and test_csv["p_value_raw"].between(0, 1).all()
)

checks["All Holm p-values valid"] = (
    test_csv["p_value_holm"].notna().all()
    and test_csv["p_value_holm"].between(0, 1).all()
)

checks["Holm p >= raw p"] = (
    (
        test_csv["p_value_holm"]
        >= test_csv["p_value_raw"] - 1e-12
    ).all()
)

# Recompute the correction; a non-significant result is valid.
from statsmodels.stats.multitest import multipletests
recomputed_reject, recomputed_holm, _, _ = multipletests(
    test_csv["p_value_raw"].to_numpy(dtype=float), alpha=ALPHA, method="holm"
)
checks["Holm correction matches recomputation"] = np.allclose(
    test_csv["p_value_holm"], recomputed_holm, rtol=1e-9, atol=1e-12
)
saved_reject = test_csv["reject_h0_holm"].astype(str).str.lower()
checks["Rejection flags valid"] = saved_reject.isin(["true", "false"]).all()
checks["Rejection flags match Holm"] = np.array_equal(
    saved_reject.eq("true").to_numpy(), recomputed_reject
)
checks["Sample accounting"] = (
    sample_csv["initial_n"].eq(sample_csv["n"] + sample_csv["excluded_n"]).all()
    and sample_csv["missing_n"].eq(sample_csv["excluded_n"]).all()
)
for name, table in [("Primary", test_csv), ("Weather", weather_csv)]:
    intervals = table[["ci_lower", "ci_upper"]].dropna()
    checks[f"{name} confidence interval ordering"] = intervals["ci_lower"].le(intervals["ci_upper"]).all()
weather_join = weather_csv.merge(
    test_csv[["rq", "variable", "n", "p_value_raw", "p_value_holm"]],
    on=["rq", "variable"], suffixes=("_weather", "_primary"),
    validate="one_to_one", how="left", indicator=True
)
checks["Weather tests included in primary family"] = weather_join["_merge"].eq("both").all()
for column in ["n", "p_value_raw", "p_value_holm"]:
    checks[f"Weather {column} matches primary table"] = np.allclose(
        weather_join[f"{column}_weather"], weather_join[f"{column}_primary"],
        rtol=1e-9, atol=1e-12
    )

checks["Sample sizes nonnegative"] = (
    sample_csv["n"].ge(0).all()
    and sample_csv["excluded_n"].ge(0).all()
)

# ------------------------------------------------------------
# 3. Check report structure
# ------------------------------------------------------------

report_text = (
    ROOT / "docs" / "inferential_statistics/delivery_risk_association_results.md"
).read_text(encoding="utf-8")

required_headings = [
    "## 1. Objective",
    "## 2. Data used",
    "## 3. Statistical design",
    "## 4. Overall results",
    "## 5. Final summary table",
    "## 6. Research-question summaries",
    "## 7. Key interpretation points",
    "## 8. Limitations",
    "## 9. Output files",
]

checks["Report required sections"] = all(
    heading in report_text
    for heading in required_headings
)

rq_ids = [
    "RQ1", "RQ2", "RQ3", "RQ4", "RQ5",
    "RQ6", "RQ7", "RQ8", "RQ9", "RQ10",
    "RQ11", "RQ12", "RQ13", "RQ14",
    "RQ15a", "RQ15b",
    "RQ16a", "RQ16b",
    "RQ17a", "RQ17b",
    "RQ19", "RQ20"
]

checks["All RQ IDs documented"] = all(
    rq in report_text for rq in rq_ids
)

# ------------------------------------------------------------
# 4. Methodological review flags
# ------------------------------------------------------------

review_flags = []

if "Strongest observed effect size:" in report_text:
    review_flags.append(
        "Report ranks effect sizes across incompatible "
        "units/scales. Replace with a specific association "
        "measure, e.g. RQ19 Phi = 0.396430."
    )

if "< 1e-300 (numerical underflow)" in report_text:
    review_flags.append(
        "Report uses an unsupported numerical bound "
        "for p-value underflow. Prefer 'numerically "
        "underflowed; software returned 0'."
    )

if "All primary tests were two-sided" in report_text:
    review_flags.append(
        "Clarify that two-sided alternatives apply "
        "where appropriate; chi-square tests are "
        "right-tail association tests."
    )

print("\n2. VALIDATION CHECKS")

check_df = pd.DataFrame([
    {
        "check": name,
        "status": "PASS" if bool(passed) else "FAIL"
    }
    for name, passed in checks.items()
])

display(check_df)

print("\n3. REPORT REVIEW FLAGS")
if review_flags:
    for i, flag in enumerate(review_flags, 1):
        print(f"{i}. {flag}")
else:
    print("No predefined report issues detected.")

# ------------------------------------------------------------
# 5. Overall results
# ------------------------------------------------------------

n_failed = int((check_df["status"] == "FAIL").sum())

print("\n" + "=" * 70)
print("FINAL AUDIT SUMMARY")
print("=" * 70)
print("Required artifacts:", len(required_files))
print("Validation checks:", len(check_df))
print("Failed checks:", n_failed)
print("Report review flags:", len(review_flags))

if n_failed == 0:
    print("RESULT: TECHNICAL AUDIT PASSED")
else:
    print("RESULT: TECHNICAL AUDIT NEEDS FIXES")

if review_flags:
    print("REPORT REQUIRES METHODOLOGICAL EDITS")

assert n_failed == 0, "Stage 3 audit failed; inspect check_df."
assert not review_flags, "Report requires methodological review."
print("\nFINAL DELIVERY AUDIT COMPLETED.")


STAGE 3 - FINAL DELIVERY AUDIT

1. FILE AUDIT


,artifact,exists,size_bytes,status
0,Notebook,True,868231,PASS
1,Report,True,16258,PASS
2,Statistical results,True,8821,PASS
3,Sample summary,True,1679,PASS
4,Descriptive summary,True,3229,PASS
5,Weather results,True,2507,PASS
6,Figure RQ1,True,66755,PASS
7,Figure RQ3,True,165186,PASS
8,Figure RQ19,True,47278,PASS
9,Figure Weather,True,65629,PASS



2. VALIDATION CHECKS


,check,status
0,All 10 required artifacts exist,PASS
1,26 primary test rows,PASS
2,26 sample summary rows,PASS
3,30 descriptive rows,PASS
4,6 weather result rows,PASS
5,No duplicate primary tests,PASS
6,All raw p-values valid,PASS
7,All Holm p-values valid,PASS
8,Holm p >= raw p,PASS
9,Holm correction matches recomputation,PASS



3. REPORT REVIEW FLAGS
No predefined report issues detected.

FINAL AUDIT SUMMARY
Required artifacts: 10
Validation checks: 22
Failed checks: 0
Report review flags: 0
RESULT: TECHNICAL AUDIT PASSED

FINAL DELIVERY AUDIT COMPLETED.
